# Eksperyment: ile dobrych sztuk przed włączeniem odrzutu

Otwórz **ten jeden plik** w Colabie: *File → Upload notebook*. Komórki są już podzielone. Każda zaczyna się od komentarza `# KOMÓRKA n/10`. Nie łącz ich z powrotem w jedną.

Przed uruchomieniem:

1. *Runtime → Change runtime type → GPU*.
2. Ustaw zmienną `DATA_ROOT` na katalog MVTec AD (w środku foldery `bottle`, `cable`, … z `train/good` i `test`).
3. Edytuj tylko komórkę 4. Resztę uruchamiaj po kolei.

Wyniki trafiają do `COMMISSIONING_WORK` (domyślnie `commissioning_runs`). Po przerwaniu sesji uruchom komórki 1–4 ponownie, a potem tę, na której stanęło: gotowe pliki są pomijane.

`PILOT = True` liczy tylko klasę transistor i oznacza przebieg jako exploratory. Właściwy eksperyment do artykułu to `PILOT = False`.

In [ ]:
# KOMÓRKA 1/10 — montowanie Dysku Google
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# KOMÓRKA 2/10 — odtworzenie plików projektu na maszynie Colaba
# Ta komórka zapisuje pakiet commissioning, config.yaml i preregistration.md.
import base64
from pathlib import Path

PROJECT = Path("/content/commissioning_project")
FILES = {"config.yaml": "IyBTY2llbnRpZmljIHBhcmFtZXRlcnMuIE1hY2hpbmUtc3BlY2lmaWMgcGF0aHMgYXJlIG5vdCBwYXJ0IG9mIHRoZSBjb25maWcgaGFzaC4NCiMgQ09NTUlTU0lPTklOR19EQVRBLCBDT01NSVNTSU9OSU5HX1dPUkssIC0tZGF0YS1yb290IGFuZCAtLXdvcmstcm9vdCBvdmVycmlkZSBwYXRocy4NCg0KcHJvdG9jb2xfdmVyc2lvbjogMQ0Kc2VlZDogNDINCm5fY2FsaWJyYXRpb246IDIwDQphbHBoYTogMC4wNQ0Kbl9yZXBlYXRzOiAyMA0Kbl9ncmlkOiBbMSwgMiwgNCwgOCwgMTYsIDMyLCA2NF0NCnN0cmF0ZWdpZXM6IFtyYW5kb20sIGtjZW50ZXJdDQphdWxjX25fbWF4OiAzMg0KcHJhY3RpY2FsX3RwcjogMC45NQ0KcHJhY3RpY2FsX2ZwcjogMC4wNQ0Kbl9zdGFyX21pbl9yZXBlYXRzOiAxNg0KZnByX2F1bGNfbWFyZ2luOiAwLjAxDQpzdG9wcGluZ19kZWx0YXM6IFswLjAyNSwgMC4wNSwgMC4xMF0NCnN0b3BwaW5nX2RlbHRhX21haW46IDAuMDUNCmJvb3RzdHJhcF9yZXBsaWNhdGVzOiAxMDAwMA0KYm9vdHN0cmFwX3NlZWQ6IDQyDQoNCmJhY2tib25lOiB3aWRlX3Jlc25ldDUwXzINCmxheWVyczogW2xheWVyMiwgbGF5ZXIzXQ0KcmVzaXplOiAyNTYNCmNlbnRlcl9jcm9wOiAyMjQNCnVwc2FtcGxlX21vZGU6IGJpbGluZWFyDQppbWFnZW5ldF9tZWFuOiBbMC40ODUsIDAuNDU2LCAwLjQwNl0NCmltYWdlbmV0X3N0ZDogWzAuMjI5LCAwLjIyNCwgMC4yMjVdDQpmZWF0dXJlX2JhdGNoX3NpemU6IDgNCmRpc3RhbmNlX3F1ZXJ5X2JhdGNoOiA4DQpkaXN0YW5jZV90cmFpbl9iYXRjaDogMTYNCg0KZTA6DQogIGF1cm9jX21lYW5fdG9sZXJhbmNlOiAwLjAwNQ0KICBzcGVhcm1hbl9taW46IDAuOTkNCiAgbnVtX25laWdoYm9yc19yZWY6IDkNCiAgbnVtX25laWdoYm9yc19ycTogMQ0KICBjb3Jlc2V0X3JhdGlvOiAwLjENCiAgIyBPcGVyYXRpb25hbCBzYW5pdHkgZmxvb3IgZm9yIHRoZSByZWZlcmVuY2UgQW5vbWFsaWIgcnVuLiBSb3RoIGV0IGFsLiByZXBvcnQgfjAuOTkxLg0KICBtZWFuX2F1cm9jX2Zsb29yOiAwLjk3DQogIHJlZmVyZW5jZV9tZWFuX2F1cm9jOiAwLjk5MQ0KDQpwYXRoczoNCiAgZGF0YV9yb290OiBkYXRhL212dGVjDQogIHdvcmtfcm9vdDogLg0KDQpvYmplY3RzOg0KICAtIGJvdHRsZQ0KICAtIGNhYmxlDQogIC0gY2Fwc3VsZQ0KICAtIGhhemVsbnV0DQogIC0gbWV0YWxfbnV0DQogIC0gcGlsbA0KICAtIHNjcmV3DQogIC0gdG9vdGhicnVzaA0KICAtIHRyYW5zaXN0b3INCiAgLSB6aXBwZXINCnRleHR1cmVzOg0KICAtIGNhcnBldA0KICAtIGdyaWQNCiAgLSBsZWF0aGVyDQogIC0gdGlsZQ0KICAtIHdvb2QNCg==", "requirements.txt": "bnVtcHk+PTEuMjYNCnNjaXB5Pj0xLjExDQpwYW5kYXM+PTIuMQ0KcHlhcnJvdz49MTQNCnNjaWtpdC1sZWFybj49MS4zDQptYXRwbG90bGliPj0zLjgNClB5WUFNTD49Ng0KUGlsbG93Pj0xMA0KdHFkbT49NC42Ng0KdG9yY2g+PTIuMQ0KdG9yY2h2aXNpb24+PTAuMTYNCnRpbW0+PTAuOQ0KYW5vbWFsaWI+PTIuMA0KcHl0ZXN0Pj03LjQNCg==", "preregistration.md": "...", "pyproject.toml": "W2J1aWxkLXN5c3RlbV0NCnJlcXVpcmVzID0gWyJzZXR1cHRvb2xzPj02OCJdDQpidWlsZC1iYWNrZW5kID0gInNldHVwdG9vbHMuYnVpbGRfbWV0YSINCg0KW3Byb2plY3RdDQpuYW1lID0gImNvbW1pc3Npb25pbmciDQp2ZXJzaW9uID0gIjAuMS4wIg0KZGVzY3JpcHRpb24gPSAiRGVmZWN0LWZyZWUgY29tbWlzc2lvbmluZyBvZiBtZW1vcnktYmFuayBhbm9tYWx5IGRldGVjdGlvbiBvbiBNVlRlYyBBRCINCnJlcXVpcmVzLXB5dGhvbiA9ICI+PTMuMTAiDQpkZXBlbmRlbmNpZXMgPSBbDQogICAgIm51bXB5Pj0xLjI2IiwNCiAgICAic2NpcHk+PTEuMTEiLA0KICAgICJwYW5kYXM+PTIuMSIsDQogICAgInB5YXJyb3c+PTE0IiwNCiAgICAic2Npa2l0LWxlYXJuPj0xLjMiLA0KICAgICJtYXRwbG90bGliPj0zLjgiLA0KICAgICJQeVlBTUw+PTYiLA0KICAgICJQaWxsb3c+PTEwIiwNCiAgICAidHFkbT49NC42NiIsDQpdDQoNCltwcm9qZWN0Lm9wdGlvbmFsLWRlcGVuZGVuY2llc10NCmdwdSA9IFsidG9yY2g+PTIuMSIsICJ0b3JjaHZpc2lvbj49MC4xNiIsICJ0aW1tPj0wLjkiXQ0KYW5vbWFsaWIgPSBbImFub21hbGliPj0yLjAiXQ0KZGV2ID0gWyJweXRlc3Q+PTcuNCJdDQoNClt0b29sLnNldHVwdG9vbHMucGFja2FnZXMuZmluZF0NCmluY2x1ZGUgPSBbImNvbW1pc3Npb25pbmcqIl0NCg0KW3Rvb2wucHl0ZXN0LmluaV9vcHRpb25zXQ0KcHl0aG9ucGF0aCA9IFsiLiJdDQp0ZXN0cGF0aHMgPSBbInRlc3RzIl0NCg==", "commissioning/__init__.py": "IiIiRGVmZWN0LWZyZWUgY29tbWlzc2lvbmluZyBleHBlcmltZW50cyBmb3IgUGF0Y2hDb3JlIG9uIE1WVGVjIEFELiIiIg0KDQpfX3ZlcnNpb25fXyA9ICIwLjEuMCINCg==", "commissioning/__main__.py": "IiIiQ29tbWFuZCBsaW5lOiBweXRob24gLW0gY29tbWlzc2lvbmluZyB7cHJlcGFyZXxmZWF0dXJlc3xkaXN0YW5jZXN8ZTB8cnVufGFuYWx5emV8cmVwb3J0fGFsbH0iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgYXJncGFyc2UNCmltcG9ydCBsb2dnaW5nDQppbXBvcnQgcGxhdGZvcm0NCmltcG9ydCBzeXMNCg0KZnJvbSBjb21taXNzaW9uaW5nIGltcG9ydCBfX3ZlcnNpb25fXw0KZnJvbSBjb21taXNzaW9uaW5nLmFuYWx5c2lzIGltcG9ydCBhbmFseXplDQpmcm9tIGNvbW1pc3Npb25pbmcuY29uZmlnIGltcG9ydCBsb2FkX2NvbmZpZw0KZnJvbSBjb21taXNzaW9uaW5nLmRhdGEgaW1wb3J0IHByZXBhcmUNCmZyb20gY29tbWlzc2lvbmluZy5kaXN0YW5jZXMgaW1wb3J0IGNvbXB1dGVfZGlzdGFuY2VzDQpmcm9tIGNvbW1pc3Npb25pbmcuZTAgaW1wb3J0IGdhdGVfcGF0aCwgcnVuX2UwDQpmcm9tIGNvbW1pc3Npb25pbmcuZmVhdHVyZXMgaW1wb3J0IGV4dHJhY3QNCmZyb20gY29tbWlzc2lvbmluZy5pb191dGlscyBpbXBvcnQgcmVhZF9qc29uLCB3cml0ZV9qc29uDQpmcm9tIGNvbW1pc3Npb25pbmcucmVwb3J0IGltcG9ydCBidWlsZF9yZXBvcnQNCmZyb20gY29tbWlzc2lvbmluZy5ydW4gaW1wb3J0IHJ1bg0KDQoNCmRlZiB3cml0ZV9lbnZpcm9ubWVudChjZmcpIC0+IE5vbmU6DQogICAgaW5mbyA9IHsNCiAgICAgICAgImNvbW1pc3Npb25pbmciOiBfX3ZlcnNpb25fXywNCiAgICAgICAgInB5dGhvbiI6IHN5cy52ZXJzaW9uLA0KICAgICAgICAicGxhdGZvcm0iOiBwbGF0Zm9ybS5wbGF0Zm9ybSgpLA0KICAgICAgICAiY29uZmlnX2hhc2giOiBjZmcuaGFzaCwNCiAgICAgICAgImRhdGFfcm9vdCI6IHN0cihjZmcuZGF0YV9yb290KSwNCiAgICAgICAgIndvcmtfcm9vdCI6IHN0cihjZmcud29ya19yb290KSwNCiAgICB9DQogICAgZm9yIG5hbWUgaW4gKCJudW1weSIsICJzY2lweSIsICJwYW5kYXMiLCAic2tsZWFybiIsICJtYXRwbG90bGliIiwgIlBJTCIpOg0KICAgICAgICB0cnk6DQogICAgICAgICAgICBtb2R1bGUgPSBfX2ltcG9ydF9fKG5hbWUgaWYgbmFtZSAhPSAic2tsZWFybiIgZWxzZSAic2tsZWFybiIpDQogICAgICAgICAgICBpZiBuYW1lID09ICJQSUwiOg0KICAgICAgICAgICAgICAgIG1vZHVsZSA9IF9faW1wb3J0X18oIlBJTCIpDQogICAgICAgICAgICBpbmZvW25hbWVdID0gZ2V0YXR0cihtb2R1bGUsICJfX3ZlcnNpb25fXyIsICJ1bmtub3duIikNCiAgICAgICAgZXhjZXB0IEltcG9ydEVycm9yOg0KICAgICAgICAgICAgaW5mb1tuYW1lXSA9IE5vbmUNCiAgICB0cnk6DQogICAgICAgIGltcG9ydCB0b3JjaA0KDQogICAgICAgIGluZm9bInRvcmNoIl0gPSB0b3JjaC5fX3ZlcnNpb25fXw0KICAgICAgICBpbmZvWyJjdWRhIl0gPSBib29sKHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkpDQogICAgICAgIGluZm9bImdwdSJdID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCkgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlIE5vbmUNCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6DQogICAgICAgIGluZm9bInRvcmNoIl0gPSBOb25lDQogICAgdHJ5Og0KICAgICAgICBpbXBvcnQgYW5vbWFsaWINCg0KICAgICAgICBpbmZvWyJhbm9tYWxpYiJdID0gZ2V0YXR0cihhbm9tYWxpYiwgIl9fdmVyc2lvbl9fIiwgInVua25vd24iKQ0KICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoNCiAgICAgICAgaW5mb1siYW5vbWFsaWIiXSA9IE5vbmUNCiAgICB3cml0ZV9qc29uKGNmZy53b3JrX3Jvb3QgLyAiZW52aXJvbm1lbnQuanNvbiIsIGluZm8pDQoNCg0KZGVmIF9wYXJzZXIoKSAtPiBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcjoNCiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihwcm9nPSJjb21taXNzaW9uaW5nIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCJzdGFnZSIsIGNob2ljZXM9WyJwcmVwYXJlIiwgImZlYXR1cmVzIiwgImRpc3RhbmNlcyIsICJlMCIsICJydW4iLCAiYW5hbHl6ZSIsICJyZXBvcnQiLCAiYWxsIl0pDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jb25maWciLCBkZWZhdWx0PU5vbmUpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhLXJvb3QiLCBkZWZhdWx0PU5vbmUpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13b3JrLXJvb3QiLCBkZWZhdWx0PU5vbmUpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jbGFzc2VzIiwgbmFyZ3M9IioiLCBkZWZhdWx0PU5vbmUpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kZXZpY2UiLCBkZWZhdWx0PU5vbmUpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1leHBsb3JhdG9yeSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1lMC1wYXJ0IiwgY2hvaWNlcz1bImEiLCAiYiIsICJhbGwiXSwgZGVmYXVsdD0iYWxsIikNCiAgICByZXR1cm4gcGFyc2VyDQoNCg0KZGVmIG1haW4oYXJndjogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUpIC0+IE5vbmU6DQogICAgbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8sIGZvcm1hdD0iJShhc2N0aW1lKXMgJShsZXZlbG5hbWUpcyAlKG1lc3NhZ2UpcyIpDQogICAgYXJncyA9IF9wYXJzZXIoKS5wYXJzZV9hcmdzKGFyZ3YpDQogICAgY2ZnID0gbG9hZF9jb25maWcoYXJncy5jb25maWcsIGRhdGFfcm9vdD1hcmdzLmRhdGFfcm9vdCwgd29ya19yb290PWFyZ3Mud29ya19yb290KQ0KICAgIGNsYXNzZXMgPSBjZmcuc2VsZWN0X2NsYXNzZXMoYXJncy5jbGFzc2VzKQ0KICAgIHdyaXRlX2Vudmlyb25tZW50KGNmZykNCiAgICBsb2dnaW5nLmluZm8oIkNvbmZpZyBoYXNoICVzIiwgY2ZnLmhhc2gpDQogICAgaWYgYXJncy5zdGFnZSBpbiB7InByZXBhcmUiLCAiYWxsIn06DQogICAgICAgIHByZXBhcmUoY2ZnLCBjbGFzc2VzKQ0KICAgIGlmIGFyZ3Muc3RhZ2UgaW4geyJmZWF0dXJlcyIsICJhbGwifToNCiAgICAgICAgZXh0cmFjdChjZmcsIGNsYXNzZXMsIGRldmljZT1hcmdzLmRldmljZSkNCiAgICBpZiBhcmdzLnN0YWdlIGluIHsiZGlzdGFuY2VzIiwgImFsbCJ9Og0KICAgICAgICBjb21wdXRlX2Rpc3RhbmNlcyhjZmcsIGNsYXNzZXMsIGRldmljZT1hcmdzLmRldmljZSkNCiAgICBpZiBhcmdzLnN0YWdlIGluIHsiZTAiLCAiYWxsIn06DQogICAgICAgIGdhdGUgPSBydW5fZTAoY2ZnLCBjbGFzc2VzLCBwYXJ0PWFyZ3MuZTBfcGFydCBpZiBhcmdzLnN0YWdlID09ICJlMCIgZWxzZSAiYWxsIikNCiAgICAgICAgaWYgYXJncy5zdGFnZSA9PSAiYWxsIiBhbmQgZ2F0ZVsic3RhdHVzIl0gIT0gIlBBU1MiIGFuZCBub3QgYXJncy5leHBsb3JhdG9yeToNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRTAgZGlkIG5vdCBwYXNzLiBUaGUgcmVtYWluaW5nIHN0YWdlcyB3ZXJlIG5vdCBzdGFydGVkLiIpDQogICAgaWYgYXJncy5zdGFnZSBpbiB7InJ1biIsICJhbGwifToNCiAgICAgICAgcnVuKGNmZywgY2xhc3NlcywgZXhwbG9yYXRvcnk9YXJncy5leHBsb3JhdG9yeSkNCiAgICBpZiBhcmdzLnN0YWdlIGluIHsiYW5hbHl6ZSIsICJhbGwifToNCiAgICAgICAgYW5hbHl6ZShjZmcpDQogICAgaWYgYXJncy5zdGFnZSBpbiB7InJlcG9ydCIsICJhbGwifToNCiAgICAgICAgZnJvbSBjb21taXNzaW9uaW5nLmFuYWx5c2lzIGltcG9ydCBsb2FkX2FuYWx5c2lzDQoNCiAgICAgICAgZnJhbWVzID0gbG9hZF9hbmFseXNpcyhjZmcud29ya19yb290IC8gImFuYWx5c2lzIikNCiAgICAgICAgZ2F0ZV9maWxlID0gZ2F0ZV9wYXRoKGNmZykNCiAgICAgICAgZ2F0ZSA9IHJlYWRfanNvbihnYXRlX2ZpbGUpIGlmIGdhdGVfZmlsZS5pc19maWxlKCkgZWxzZSBOb25lDQogICAgICAgIHBhdGggPSBidWlsZF9yZXBvcnQoDQogICAgICAgICAgICBmcmFtZXMsDQogICAgICAgICAgICBjZmcud29ya19yb290IC8gInJlcG9ydCIsDQogICAgICAgICAgICBnYXRlPWdhdGUsDQogICAgICAgICAgICBkYXRhX3Jvb3Q9Y2ZnLmRhdGFfcm9vdCwNCiAgICAgICAgICAgIHdvcmtfcm9vdD1jZmcud29ya19yb290LA0KICAgICAgICAgICAgcmVzaXplPWNmZy5yZXNpemUsDQogICAgICAgICAgICBjcm9wPWNmZy5jZW50ZXJfY3JvcCwNCiAgICAgICAgKQ0KICAgICAgICBsb2dnaW5nLmluZm8oIlJlcG9ydCB3cml0dGVuIHRvICVzIiwgcGF0aCkNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIG1haW4oKQ0K", "commissioning/analysis.py": "IiIiQWdncmVnYXRlIHRoZSB0cmFqZWN0b3J5IHRhYmxlIGludG8gQVVMQywgTiosIEZQUiBzdW1tYXJpZXMsIGFuZCBzdG9wcGluZyBvdXRjb21lcy4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmZyb20gc2NpcHkuc3RhdHMgaW1wb3J0IGJldGFiaW5vbQ0KDQpmcm9tIGNvbW1pc3Npb25pbmcuY2FsaWJyYXRpb24gaW1wb3J0IGNsb3BwZXJfcGVhcnNvbl91cHBlciwgZmFsc2VfcmVqZWN0X2J1ZGdldA0KZnJvbSBjb21taXNzaW9uaW5nLmNvbmZpZyBpbXBvcnQgRVhQRUNURURfQ09VTlRTLCBDb25maWcNCmZyb20gY29tbWlzc2lvbmluZy5pb191dGlscyBpbXBvcnQgYXRvbWljX3BhcnF1ZXQsIHdyaXRlX2pzb24NCmZyb20gY29tbWlzc2lvbmluZy5zdGF0cyBpbXBvcnQgY29tcGFyZV9zdHJhdGVnaWVzDQpmcm9tIGNvbW1pc3Npb25pbmcuc3RvcHBpbmcgaW1wb3J0IG5fc3RvcA0KDQpNQUlOX0dSSUQgPSAoMSwgMiwgNCwgOCwgMTYsIDMyKQ0KDQoNCmRlZiBhdWxjKG5zLCB2YWx1ZXMsIG5fbWF4OiBpbnQgPSAzMikgLT4gZmxvYXQ6DQogICAgIiIiVHJhcGV6b2lkIG9uIGxvZzIoTiksIGRpdmlkZWQgYnkgbG9nMihuX21heCkgLSBsb2cyKDEpLiBQb2ludHMgYWJvdmUgbl9tYXggYXJlIGlnbm9yZWQuIiIiDQogICAgaWYgaW50KG5fbWF4KSAhPSAzMjoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUaGUgbWFpbiBBVUxDIG5vcm1hbGl6YXRpb24gaXMgZml4ZWQgYXQgbG9nMigzMikgLSBsb2cyKDEpLiIpDQogICAgcG9pbnRzID0ge2ludChuKTogZmxvYXQoeSkgZm9yIG4sIHkgaW4gemlwKG5zLCB2YWx1ZXMpfQ0KICAgIG1pc3NpbmcgPSBbbiBmb3IgbiBpbiBNQUlOX0dSSUQgaWYgbiBub3QgaW4gcG9pbnRzXQ0KICAgIGlmIG1pc3Npbmc6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkFVTEMgaXMgbWlzc2luZyBtYWluLWdyaWQgcG9pbnRzIHttaXNzaW5nfS4iKQ0KICAgIHhzID0gbnAubG9nMihucC5hc2FycmF5KE1BSU5fR1JJRCwgZHR5cGU9ZmxvYXQpKQ0KICAgIHlzID0gbnAuYXNhcnJheShbcG9pbnRzW25dIGZvciBuIGluIE1BSU5fR1JJRF0sIGR0eXBlPWZsb2F0KQ0KICAgIGFyZWEgPSBmbG9hdChucC5zdW0oKHlzWzotMV0gKyB5c1sxOl0pICogMC41ICogbnAuZGlmZih4cykpKQ0KICAgIHJldHVybiBhcmVhIC8gZmxvYXQobnAubG9nMihuX21heCkgLSBucC5sb2cyKDEuMCkpDQoNCg0KZGVmIG5fc3RhcihncmlkOiBsaXN0W2ludF0sIG5fcGFzc2VkOiBsaXN0W2ludF0sIG1pbl9wYXNzZXM6IGludCkgLT4gZGljdDoNCiAgICAiIiJTbWFsbGVzdCBOIGF0IHdoaWNoIGV2ZXJ5IGdyaWQgcG9pbnQgZnJvbSB0aGVyZSBvbiBoYXMgYXQgbGVhc3QgbWluX3Bhc3NlcyBzdWNjZXNzZXMuIiIiDQogICAgaWYgbGVuKGdyaWQpICE9IGxlbihuX3Bhc3NlZCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiR3JpZCBhbmQgcGFzcyBjb3VudHMgbXVzdCBiZSBhbGlnbmVkLiIpDQogICAgb2sgPSBbaW50KGNvdW50KSA+PSBpbnQobWluX3Bhc3NlcykgZm9yIGNvdW50IGluIG5fcGFzc2VkXQ0KICAgIGZvciBpbmRleCwgbiBpbiBlbnVtZXJhdGUoZ3JpZCk6DQogICAgICAgIGlmIGFsbChva1tpbmRleDpdKToNCiAgICAgICAgICAgIHJldHVybiB7Im5fc3RhciI6IGludChuKSwgImNlbnNvcmVkIjogRmFsc2V9DQogICAgcmV0dXJuIHsibl9zdGFyIjogTm9uZSwgImNlbnNvcmVkIjogVHJ1ZX0NCg0KDQpkZWYgc3RvcF9jYXRlZ29yeShuX3N0b3BfdmFsdWUsIG5fc3Rhcl92YWx1ZSwgY2Vuc29yZWQ6IGJvb2wpIC0+IHN0cjoNCiAgICBpZiBjZW5zb3JlZDoNCiAgICAgICAgcmV0dXJuICJuX3N0YXJfbm90X3JlYWNoZWQiDQogICAgaWYgbl9zdG9wX3ZhbHVlIGlzIE5vbmUgb3IgKGlzaW5zdGFuY2Uobl9zdG9wX3ZhbHVlLCBmbG9hdCkgYW5kIG5wLmlzbmFuKG5fc3RvcF92YWx1ZSkpOg0KICAgICAgICByZXR1cm4gInJ1bGVfbm90X2ZpcmVkIg0KICAgIGlmIGludChuX3N0b3BfdmFsdWUpIDwgaW50KG5fc3Rhcl92YWx1ZSk6DQogICAgICAgIHJldHVybiAicHJlbWF0dXJlIg0KICAgIGlmIGludChuX3N0b3BfdmFsdWUpID09IGludChuX3N0YXJfdmFsdWUpOg0KICAgICAgICByZXR1cm4gImNvbmNvcmRhbnQiDQogICAgcmV0dXJuICJleGNlc3NpdmUiDQoNCg0KZGVmIGxvYWRfbWV0cmljcyhjZmc6IENvbmZpZykgLT4gcGQuRGF0YUZyYW1lOg0KICAgIHBhcnRzID0gc29ydGVkKChjZmcud29ya19yb290IC8gInJ1bnMiIC8gInBhcnRzIikuZ2xvYigiKi5wYXJxdWV0IikpDQogICAgaWYgbm90IHBhcnRzOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigiTm8gdHJhamVjdG9yeSB0YWJsZXMgaW4gcnVucy9wYXJ0cy4gUnVuIHRoZSB0cmFqZWN0b3JpZXMgZmlyc3QuIikNCiAgICBmcmFtZXMgPSBbcGQucmVhZF9wYXJxdWV0KHBhdGgpIGZvciBwYXRoIGluIHBhcnRzXQ0KICAgIG1ldHJpY3MgPSBwZC5jb25jYXQoZnJhbWVzLCBpZ25vcmVfaW5kZXg9VHJ1ZSkNCiAgICBoYXNoZXMgPSBzZXQobWV0cmljc1siY29uZmlnX2hhc2giXS5hc3R5cGUoc3RyKSkNCiAgICBpZiBoYXNoZXMgIT0ge2NmZy5oYXNofToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiVHJhamVjdG9yeSB0YWJsZXMgbWl4IGNvbmZpZyBoYXNoZXM6IHtoYXNoZXN9IikNCiAgICByZXR1cm4gbWV0cmljcw0KDQoNCmRlZiBzdW1tYXJpemVfY2xhc3NlcyhtZXRyaWNzOiBwZC5EYXRhRnJhbWUsIG5fbWF4OiBpbnQsIG1pbl9wYXNzZXM6IGludCkgLT4gcGQuRGF0YUZyYW1lOg0KICAgIHJvd3MgPSBbXQ0KICAgIGdyb3VwZWQgPSBtZXRyaWNzLmdyb3VwYnkoWyJjbGFzc19uYW1lIiwgImdyb3VwIiwgInN0cmF0ZWd5Il0sIHNvcnQ9VHJ1ZSkNCiAgICBmb3IgKGNsYXNzX25hbWUsIGdyb3VwLCBzdHJhdGVneSksIHN1YiBpbiBncm91cGVkOg0KICAgICAgICB0cHJfdmFsdWVzID0gW10NCiAgICAgICAgZnByX3ZhbHVlcyA9IFtdDQogICAgICAgIGZvciBfcmVwLCByZXBfZGYgaW4gc3ViLmdyb3VwYnkoInJlcCIpOg0KICAgICAgICAgICAgb3JkZXJlZCA9IHJlcF9kZi5zb3J0X3ZhbHVlcygiTiIpDQogICAgICAgICAgICB0cHJfdmFsdWVzLmFwcGVuZChhdWxjKG9yZGVyZWRbIk4iXSwgb3JkZXJlZFsidHByIl0sIG5fbWF4KSkNCiAgICAgICAgICAgIGZwcl92YWx1ZXMuYXBwZW5kKGF1bGMob3JkZXJlZFsiTiJdLCBvcmRlcmVkWyJmcHIiXSwgbl9tYXgpKQ0KICAgICAgICBncmlkID0gW2ludChuKSBmb3IgbiBpbiBzb3J0ZWQoc3ViWyJOIl0udW5pcXVlKCkpXQ0KICAgICAgICBwYXNzZWQgPSBbaW50KHN1Yi5sb2Nbc3ViWyJOIl0gPT0gbiwgImNyaXRlcmlvbiJdLnN1bSgpKSBmb3IgbiBpbiBncmlkXQ0KICAgICAgICBzdGFyID0gbl9zdGFyKGdyaWQsIHBhc3NlZCwgbWluX3Bhc3NlcykNCiAgICAgICAgcG9vbF9zaXplID0gaW50KHN1YlsicG9vbF9zaXplIl0uaWxvY1swXSkNCiAgICAgICAgcm93cy5hcHBlbmQoDQogICAgICAgICAgICB7DQogICAgICAgICAgICAgICAgImNsYXNzX25hbWUiOiBjbGFzc19uYW1lLA0KICAgICAgICAgICAgICAgICJncm91cCI6IGdyb3VwLA0KICAgICAgICAgICAgICAgICJzdHJhdGVneSI6IHN0cmF0ZWd5LA0KICAgICAgICAgICAgICAgICJhdWxjX3RwciI6IGZsb2F0KG5wLm1lYW4odHByX3ZhbHVlcykpLA0KICAgICAgICAgICAgICAgICJhdWxjX2ZwciI6IGZsb2F0KG5wLm1lYW4oZnByX3ZhbHVlcykpLA0KICAgICAgICAgICAgICAgICJhdWxjX3Rwcl9pcXIiOiBmbG9hdChucC5zdWJ0cmFjdCgqbnAucXVhbnRpbGUodHByX3ZhbHVlcywgWzAuNzUsIDAuMjVdKSkpLA0KICAgICAgICAgICAgICAgICJhdWxjX2Zwcl9pcXIiOiBmbG9hdChucC5zdWJ0cmFjdCgqbnAucXVhbnRpbGUoZnByX3ZhbHVlcywgWzAuNzUsIDAuMjVdKSkpLA0KICAgICAgICAgICAgICAgICJhdWxjX3Rwcl9yYW5nZSI6IGZsb2F0KG5wLm1heCh0cHJfdmFsdWVzKSAtIG5wLm1pbih0cHJfdmFsdWVzKSksDQogICAgICAgICAgICAgICAgImF1bGNfZnByX3JhbmdlIjogZmxvYXQobnAubWF4KGZwcl92YWx1ZXMpIC0gbnAubWluKGZwcl92YWx1ZXMpKSwNCiAgICAgICAgICAgICAgICAibl9zdGFyIjogKGZsb2F0KHN0YXJbIm5fc3RhciJdKSBpZiBub3Qgc3RhclsiY2Vuc29yZWQiXSBlbHNlIGZsb2F0KCJuYW4iKSksDQogICAgICAgICAgICAgICAgImNlbnNvcmVkIjogYm9vbChzdGFyWyJjZW5zb3JlZCJdKSwNCiAgICAgICAgICAgICAgICAicG9vbF9zaXplIjogcG9vbF9zaXplLA0KICAgICAgICAgICAgICAgICJuX3JlcHMiOiBpbnQoc3ViWyJyZXAiXS5udW5pcXVlKCkpLA0KICAgICAgICAgICAgfQ0KICAgICAgICApDQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzKQ0KDQoNCmRlZiBmcHJfYWdncmVnYXRlKG1ldHJpY3M6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOg0KICAgICIiIlNwcmVhZCBhY3Jvc3MgcmVwZXRpdGlvbnMuIFJlcGV0aXRpb25zIHNoYXJlIG9uZSB0ZXN0IHNldCwgc28gdGhlaXIgY291bnRzIGFyZSBub3QgcG9vbGVkLiIiIg0KICAgIHJvd3MgPSBbXQ0KICAgIGZvciAoY2xhc3NfbmFtZSwgc3RyYXRlZ3ksIG4pLCBzdWIgaW4gbWV0cmljcy5ncm91cGJ5KFsiY2xhc3NfbmFtZSIsICJzdHJhdGVneSIsICJOIl0sIHNvcnQ9VHJ1ZSk6DQogICAgICAgIGZwciA9IHN1YlsiZnByIl0udG9fbnVtcHkoZHR5cGU9ZmxvYXQpDQogICAgICAgIHExLCBxMyA9IG5wLnF1YW50aWxlKGZwciwgWzAuMjUsIDAuNzVdKQ0KICAgICAgICByb3dzLmFwcGVuZCgNCiAgICAgICAgICAgIHsNCiAgICAgICAgICAgICAgICAiY2xhc3NfbmFtZSI6IGNsYXNzX25hbWUsDQogICAgICAgICAgICAgICAgImdyb3VwIjogc3ViWyJncm91cCJdLmlsb2NbMF0sDQogICAgICAgICAgICAgICAgInN0cmF0ZWd5Ijogc3RyYXRlZ3ksDQogICAgICAgICAgICAgICAgIk4iOiBpbnQobiksDQogICAgICAgICAgICAgICAgIm1lYW5fZnByIjogZmxvYXQobnAubWVhbihmcHIpKSwNCiAgICAgICAgICAgICAgICAibWVkaWFuX2ZwciI6IGZsb2F0KG5wLm1lZGlhbihmcHIpKSwNCiAgICAgICAgICAgICAgICAiaXFyX2ZwciI6IGZsb2F0KHEzIC0gcTEpLA0KICAgICAgICAgICAgICAgICJtaW5fZnByIjogZmxvYXQobnAubWluKGZwcikpLA0KICAgICAgICAgICAgICAgICJtYXhfZnByIjogZmxvYXQobnAubWF4KGZwcikpLA0KICAgICAgICAgICAgICAgICJmcmFjdGlvbl9mcHJfYWJvdmVfMDUiOiBmbG9hdChucC5tZWFuKGZwciA+IDAuMDUpKSwNCiAgICAgICAgICAgICAgICAiZnJhY3Rpb25fbWVldGluZ19jcml0ZXJpb24iOiBmbG9hdChucC5tZWFuKHN1YlsiY3JpdGVyaW9uIl0uYXN0eXBlKGJvb2wpKSksDQogICAgICAgICAgICAgICAgIm5fcmVwcyI6IGludChsZW4oc3ViKSksDQogICAgICAgICAgICB9DQogICAgICAgICkNCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpDQoNCg0KZGVmIHN0b3BwaW5nX3RhYmxlKG1ldHJpY3M6IHBkLkRhdGFGcmFtZSwgZGVsdGFzOiBsaXN0W2Zsb2F0XSkgLT4gcGQuRGF0YUZyYW1lOg0KICAgIHJvd3MgPSBbXQ0KICAgIGZvciAoY2xhc3NfbmFtZSwgc3RyYXRlZ3ksIHJlcCksIHN1YiBpbiBtZXRyaWNzLmdyb3VwYnkoWyJjbGFzc19uYW1lIiwgInN0cmF0ZWd5IiwgInJlcCJdLCBzb3J0PVRydWUpOg0KICAgICAgICBvcmRlcmVkID0gc3ViLnNvcnRfdmFsdWVzKCJOIikNCiAgICAgICAgZ3JpZCA9IFtpbnQobikgZm9yIG4gaW4gb3JkZXJlZFsiTiJdLnRvbGlzdCgpXQ0KICAgICAgICBtZWRpYW5zID0gb3JkZXJlZFsibWVkaWFuX2NhbGlicmF0aW9uIl0udG9fbnVtcHkoZHR5cGU9ZmxvYXQpDQogICAgICAgIGZvciBkZWx0YSBpbiBkZWx0YXM6DQogICAgICAgICAgICByb3dzLmFwcGVuZCgNCiAgICAgICAgICAgICAgICB7DQogICAgICAgICAgICAgICAgICAgICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwNCiAgICAgICAgICAgICAgICAgICAgImdyb3VwIjogb3JkZXJlZFsiZ3JvdXAiXS5pbG9jWzBdLA0KICAgICAgICAgICAgICAgICAgICAic3RyYXRlZ3kiOiBzdHJhdGVneSwNCiAgICAgICAgICAgICAgICAgICAgInJlcCI6IGludChyZXApLA0KICAgICAgICAgICAgICAgICAgICAiZGVsdGEiOiBmbG9hdChkZWx0YSksDQogICAgICAgICAgICAgICAgICAgICJuX3N0b3AiOiBuX3N0b3AoZ3JpZCwgbWVkaWFucywgZmxvYXQoZGVsdGEpKSwNCiAgICAgICAgICAgICAgICAgICAgInBvb2xfc2l6ZSI6IGludChvcmRlcmVkWyJwb29sX3NpemUiXS5pbG9jWzBdKSwNCiAgICAgICAgICAgICAgICB9DQogICAgICAgICAgICApDQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzKQ0KDQoNCmRlZiBidWRnZXRfdGFibGUoKSAtPiBwZC5EYXRhRnJhbWU6DQogICAgcm93cyA9IFtdDQogICAgZm9yIGNsYXNzX25hbWUsIGNvdW50cyBpbiBFWFBFQ1RFRF9DT1VOVFMuaXRlbXMoKToNCiAgICAgICAgbl9nb29kID0gaW50KGNvdW50c1sidGVzdF9nb29kIl0pDQogICAgICAgIGJ1ZGdldCA9IGZhbHNlX3JlamVjdF9idWRnZXQobl9nb29kKQ0KICAgICAgICBrX21pbiA9IGJ1ZGdldCArIDENCiAgICAgICAgZXhwbG9yYXRvcnlfcHJvYmFiaWxpdHkgPSBmbG9hdCgxLjAgLSBiZXRhYmlub20uY2RmKGtfbWluIC0gMSwgbl9nb29kLCAxLCAyMCkpDQogICAgICAgIHJvd3MuYXBwZW5kKA0KICAgICAgICAgICAgew0KICAgICAgICAgICAgICAgICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwNCiAgICAgICAgICAgICAgICAibl90ZXN0X2dvb2QiOiBuX2dvb2QsDQogICAgICAgICAgICAgICAgIm5fdHJhaW4iOiBpbnQoY291bnRzWyJ0cmFpbiJdKSwNCiAgICAgICAgICAgICAgICAiZmFsc2VfcmVqZWN0X2J1ZGdldCI6IGJ1ZGdldCwNCiAgICAgICAgICAgICAgICAiY2xvcHBlcl91cHBlcl96ZXJvX2Vycm9ycyI6IGNsb3BwZXJfcGVhcnNvbl91cHBlcigwLCBuX2dvb2QpLA0KICAgICAgICAgICAgICAgICJleHBsb3JhdG9yeV9wX3NhbXBsZV9mcHJfYWJvdmVfMDUiOiBleHBsb3JhdG9yeV9wcm9iYWJpbGl0eSwNCiAgICAgICAgICAgIH0NCiAgICAgICAgKQ0KICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykNCg0KDQpkZWYgX21lZGlhbl9vcl9ub25lKHZhbHVlczogbGlzdFtpbnRdKSAtPiBmbG9hdCB8IE5vbmU6DQogICAgaWYgbm90IHZhbHVlczoNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICByZXR1cm4gZmxvYXQobnAubWVkaWFuKG5wLmFzYXJyYXkodmFsdWVzLCBkdHlwZT1mbG9hdCkpKQ0KDQoNCmRlZiBhdHRhY2hfc3RvcHBpbmcoc3VtbWFyeTogcGQuRGF0YUZyYW1lLCBzdG9wcGluZzogcGQuRGF0YUZyYW1lLCBtYWluX2RlbHRhOiBmbG9hdCkgLT4gcGQuRGF0YUZyYW1lOg0KICAgIG1haW4gPSBzdG9wcGluZ1tucC5pc2Nsb3NlKHN0b3BwaW5nWyJkZWx0YSJdLCBtYWluX2RlbHRhKV0uY29weSgpDQogICAgcm93cyA9IFtdDQogICAgZm9yIHJvdyBpbiBzdW1tYXJ5Lml0ZXJ0dXBsZXMoaW5kZXg9RmFsc2UpOg0KICAgICAgICBzdWIgPSBtYWluWyhtYWluWyJjbGFzc19uYW1lIl0gPT0gcm93LmNsYXNzX25hbWUpICYgKG1haW5bInN0cmF0ZWd5Il0gPT0gcm93LnN0cmF0ZWd5KV0NCiAgICAgICAgZmlyZWQgPSBbaW50KHYpIGZvciB2IGluIHN1Ylsibl9zdG9wIl0uZHJvcG5hKCkudG9saXN0KCldDQogICAgICAgIGNhdGVnb3JpZXMgPSBbDQogICAgICAgICAgICBzdG9wX2NhdGVnb3J5KE5vbmUgaWYgcGQuaXNuYSh2KSBlbHNlIGludCh2KSwgTm9uZSBpZiByb3cuY2Vuc29yZWQgZWxzZSBpbnQocm93Lm5fc3RhciksIGJvb2wocm93LmNlbnNvcmVkKSkNCiAgICAgICAgICAgIGZvciB2IGluIHN1Ylsibl9zdG9wIl0udG9saXN0KCkNCiAgICAgICAgXQ0KICAgICAgICByYXRpbyA9IFtdDQogICAgICAgIGV4Y2VzcyA9IFtdDQogICAgICAgIGlmIG5vdCByb3cuY2Vuc29yZWQ6DQogICAgICAgICAgICByYXRpbyA9IFt2IC8gaW50KHJvdy5uX3N0YXIpIGZvciB2IGluIGZpcmVkXQ0KICAgICAgICAgICAgZXhjZXNzID0gW21heCgwLCB2IC0gaW50KHJvdy5uX3N0YXIpKSBmb3IgdiBpbiBmaXJlZF0NCiAgICAgICAgcmVjb3JkID0gcm93Ll9hc2RpY3QoKQ0KICAgICAgICByZWNvcmRbIm5fc3RvcF9tZWRpYW4iXSA9IF9tZWRpYW5fb3Jfbm9uZShmaXJlZCkNCiAgICAgICAgcmVjb3JkWyJuX3N0b3BfcmF0aW9fbWVkaWFuIl0gPSBmbG9hdChucC5tZWRpYW4ocmF0aW8pKSBpZiByYXRpbyBlbHNlIE5vbmUNCiAgICAgICAgcmVjb3JkWyJleGNlc3NfaW1hZ2VzX21lZGlhbiJdID0gZmxvYXQobnAubWVkaWFuKGV4Y2VzcykpIGlmIGV4Y2VzcyBlbHNlIE5vbmUNCiAgICAgICAgZm9yIG5hbWUgaW4gKCJwcmVtYXR1cmUiLCAiY29uY29yZGFudCIsICJleGNlc3NpdmUiLCAibl9zdGFyX25vdF9yZWFjaGVkIiwgInJ1bGVfbm90X2ZpcmVkIik6DQogICAgICAgICAgICByZWNvcmRbZiJmcmFjdGlvbl97bmFtZX0iXSA9IGZsb2F0KG5wLm1lYW4oW2MgPT0gbmFtZSBmb3IgYyBpbiBjYXRlZ29yaWVzXSkpIGlmIGNhdGVnb3JpZXMgZWxzZSBOb25lDQogICAgICAgIHJvd3MuYXBwZW5kKHJlY29yZCkNCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpDQoNCg0KZGVmIGFuYWx5emUoY2ZnOiBDb25maWcpIC0+IGRpY3Q6DQogICAgbWV0cmljcyA9IGxvYWRfbWV0cmljcyhjZmcpDQogICAgc3VtbWFyeSA9IHN1bW1hcml6ZV9jbGFzc2VzKG1ldHJpY3MsIGNmZy5hdWxjX25fbWF4LCBjZmcubl9zdGFyX21pbl9yZXBlYXRzKQ0KICAgIHN0b3BwaW5nID0gc3RvcHBpbmdfdGFibGUobWV0cmljcywgY2ZnLnN0b3BwaW5nX2RlbHRhcykNCiAgICBzdW1tYXJ5ID0gYXR0YWNoX3N0b3BwaW5nKHN1bW1hcnksIHN0b3BwaW5nLCBjZmcuc3RvcHBpbmdfZGVsdGFfbWFpbikNCiAgICBhZ2dyZWdhdGUgPSBmcHJfYWdncmVnYXRlKG1ldHJpY3MpDQogICAgc3RhdGlzdGljcyA9IGNvbXBhcmVfc3RyYXRlZ2llcygNCiAgICAgICAgc3VtbWFyeSwNCiAgICAgICAgbWFyZ2luPWNmZy5mcHJfYXVsY19tYXJnaW4sDQogICAgICAgIG5fYm9vdHN0cmFwPWNmZy5ib290c3RyYXBfcmVwbGljYXRlcywNCiAgICAgICAgc2VlZD1jZmcuYm9vdHN0cmFwX3NlZWQsDQogICAgKQ0KICAgIG91dCA9IGNmZy53b3JrX3Jvb3QgLyAiYW5hbHlzaXMiDQogICAgb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBhdG9taWNfcGFycXVldChvdXQgLyAibWV0cmljcy5wYXJxdWV0IiwgbWV0cmljcykNCiAgICBzdW1tYXJ5LnRvX2NzdihvdXQgLyAic3VtbWFyeS5jc3YiLCBpbmRleD1GYWxzZSkNCiAgICBhZ2dyZWdhdGUudG9fY3N2KG91dCAvICJmcHJfYWdncmVnYXRlLmNzdiIsIGluZGV4PUZhbHNlKQ0KICAgIHN0b3BwaW5nLnRvX2NzdihvdXQgLyAic3RvcHBpbmcuY3N2IiwgaW5kZXg9RmFsc2UpDQogICAgYnVkZ2V0X3RhYmxlKCkudG9fY3N2KG91dCAvICJmYWxzZV9yZWplY3RfYnVkZ2V0LmNzdiIsIGluZGV4PUZhbHNlKQ0KICAgIHdyaXRlX2pzb24ob3V0IC8gInN0YXRpc3RpY3MuanNvbiIsIHN0YXRpc3RpY3MpDQogICAgcmV0dXJuIHsNCiAgICAgICAgIm1ldHJpY3MiOiBtZXRyaWNzLA0KICAgICAgICAic3VtbWFyeSI6IHN1bW1hcnksDQogICAgICAgICJmcHJfYWdncmVnYXRlIjogYWdncmVnYXRlLA0KICAgICAgICAic3RvcHBpbmciOiBzdG9wcGluZywNCiAgICAgICAgInN0YXRpc3RpY3MiOiBzdGF0aXN0aWNzLA0KICAgICAgICAiYnVkZ2V0IjogYnVkZ2V0X3RhYmxlKCksDQogICAgICAgICJhbmFseXNpc19kaXIiOiBvdXQsDQogICAgfQ0KDQoNCmRlZiBsb2FkX2FuYWx5c2lzKGFuYWx5c2lzX2RpcjogUGF0aCkgLT4gZGljdDoNCiAgICByZXR1cm4gew0KICAgICAgICAibWV0cmljcyI6IHBkLnJlYWRfcGFycXVldChhbmFseXNpc19kaXIgLyAibWV0cmljcy5wYXJxdWV0IiksDQogICAgICAgICJzdW1tYXJ5IjogcGQucmVhZF9jc3YoYW5hbHlzaXNfZGlyIC8gInN1bW1hcnkuY3N2IiksDQogICAgICAgICJmcHJfYWdncmVnYXRlIjogcGQucmVhZF9jc3YoYW5hbHlzaXNfZGlyIC8gImZwcl9hZ2dyZWdhdGUuY3N2IiksDQogICAgICAgICJzdG9wcGluZyI6IHBkLnJlYWRfY3N2KGFuYWx5c2lzX2RpciAvICJzdG9wcGluZy5jc3YiKSwNCiAgICAgICAgImJ1ZGdldCI6IHBkLnJlYWRfY3N2KGFuYWx5c2lzX2RpciAvICJmYWxzZV9yZWplY3RfYnVkZ2V0LmNzdiIpLA0KICAgICAgICAic3RhdGlzdGljcyI6IF9faW1wb3J0X18oImpzb24iKS5sb2FkcygoYW5hbHlzaXNfZGlyIC8gInN0YXRpc3RpY3MuanNvbiIpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSksDQogICAgfQ0K", "commissioning/calibration.py": "IiIiVGhyZXNob2xkIGZyb20gZ29vZCBpbWFnZXMgb25seSwgYW5kIHRoZSBtZXRyaWNzIGNvbXB1dGVkIG9uIHRoZSBvZmZpY2lhbCB0ZXN0IHNldC4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbWF0aA0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmZyb20gc2NpcHkuc3RhdHMgaW1wb3J0IGJldGENCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCByb2NfYXVjX3Njb3JlDQoNCg0KZGVmIG9yZGVyX3N0YXRpc3RpY19yYW5rKG46IGludCwgYWxwaGE6IGZsb2F0KSAtPiBpbnQ6DQogICAgIiIiMS1iYXNlZCByYW5rIGsgPSBjZWlsKChuICsgMSkgKiAoMSAtIGFscGhhKSksIGNsaXBwZWQgdG8gMS4ubi4iIiINCiAgICBpZiBuIDwgMToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJDYWxpYnJhdGlvbiBzZXQgaXMgZW1wdHkuIikNCiAgICBpZiBub3QgMC4wIDwgYWxwaGEgPCAxLjA6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmImFscGhhIG11c3QgYmUgaW4gKDAsIDEpLCBnb3Qge2FscGhhfSIpDQogICAgayA9IGludChtYXRoLmNlaWwoKG4gKyAxKSAqICgxLjAgLSBhbHBoYSkpKQ0KICAgIHJldHVybiBtaW4obWF4KGssIDEpLCBuKQ0KDQoNCmRlZiB0aHJlc2hvbGRfZnJvbV9zY29yZXMoc2NvcmVzOiBucC5uZGFycmF5LCBhbHBoYTogZmxvYXQpIC0+IHR1cGxlW2Zsb2F0LCBpbnRdOg0KICAgIHZhbHVlcyA9IG5wLmFzYXJyYXkoc2NvcmVzLCBkdHlwZT1ucC5mbG9hdDY0KQ0KICAgIGlmIHZhbHVlcy5uZGltICE9IDEgb3Igbm90IG5wLmlzZmluaXRlKHZhbHVlcykuYWxsKCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQ2FsaWJyYXRpb24gc2NvcmVzIG11c3QgYmUgYSBmaW5pdGUgMS1EIHZlY3Rvci4iKQ0KICAgIGsgPSBvcmRlcl9zdGF0aXN0aWNfcmFuayhsZW4odmFsdWVzKSwgYWxwaGEpDQogICAgb3JkZXJlZCA9IG5wLnNvcnQodmFsdWVzKQ0KICAgIHJldHVybiBmbG9hdChvcmRlcmVkW2sgLSAxXSksIGsNCg0KDQpkZWYgY2xvcHBlcl9wZWFyc29uX2xvd2VyKHN1Y2Nlc3NlczogaW50LCBuOiBpbnQsIGFscGhhOiBmbG9hdCA9IDAuMDUpIC0+IGZsb2F0Og0KICAgICIiIk9uZS1zaWRlZCBsb3dlciAxMDAqKDEtYWxwaGEpJSBDbG9wcGVyLVBlYXJzb24gYm91bmQuIiIiDQogICAgaWYgbiA8IDAgb3Igc3VjY2Vzc2VzIDwgMCBvciBzdWNjZXNzZXMgPiBuOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJJbnZhbGlkIGJpbm9taWFsIGNvdW50cyBrPXtzdWNjZXNzZXN9LCBuPXtufSIpDQogICAgaWYgbiA9PSAwOg0KICAgICAgICByZXR1cm4gZmxvYXQoIm5hbiIpDQogICAgaWYgc3VjY2Vzc2VzID09IDA6DQogICAgICAgIHJldHVybiAwLjANCiAgICByZXR1cm4gZmxvYXQoYmV0YS5wcGYoYWxwaGEsIHN1Y2Nlc3NlcywgbiAtIHN1Y2Nlc3NlcyArIDEpKQ0KDQoNCmRlZiBjbG9wcGVyX3BlYXJzb25fdXBwZXIoc3VjY2Vzc2VzOiBpbnQsIG46IGludCwgYWxwaGE6IGZsb2F0ID0gMC4wNSkgLT4gZmxvYXQ6DQogICAgIiIiT25lLXNpZGVkIHVwcGVyIDEwMCooMS1hbHBoYSklIENsb3BwZXItUGVhcnNvbiBib3VuZC4iIiINCiAgICBpZiBuIDwgMCBvciBzdWNjZXNzZXMgPCAwIG9yIHN1Y2Nlc3NlcyA+IG46DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkludmFsaWQgYmlub21pYWwgY291bnRzIGs9e3N1Y2Nlc3Nlc30sIG49e259IikNCiAgICBpZiBuID09IDA6DQogICAgICAgIHJldHVybiBmbG9hdCgibmFuIikNCiAgICBpZiBzdWNjZXNzZXMgPT0gbjoNCiAgICAgICAgcmV0dXJuIDEuMA0KICAgIHJldHVybiBmbG9hdChiZXRhLnBwZigxLjAgLSBhbHBoYSwgc3VjY2Vzc2VzICsgMSwgbiAtIHN1Y2Nlc3NlcykpDQoNCg0KZGVmIGZhbHNlX3JlamVjdF9idWRnZXQobl9nb29kOiBpbnQpIC0+IGludDoNCiAgICAiIiJMYXJnZXN0IGsgd2l0aCBrIC8gbl9nb29kIDw9IDAuMDUuIEludGVnZXIgYXJpdGhtZXRpYyBhdm9pZHMgYSBiaW5hcnkgcm91bmRpbmcgZWRnZS4iIiINCiAgICBpZiBuX2dvb2QgPCAwOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJuX2dvb2QgbXVzdCBiZSBub24tbmVnYXRpdmUsIGdvdCB7bl9nb29kfSIpDQogICAgcmV0dXJuIChpbnQobl9nb29kKSAqIDUpIC8vIDEwMA0KDQoNCmRlZiBvcmFjbGVfZnByX2F0X3RwcihzY29yZXM6IG5wLm5kYXJyYXksIGxhYmVsczogbnAubmRhcnJheSwgdGFyZ2V0X3RwcjogZmxvYXQgPSAwLjk1KSAtPiBmbG9hdDoNCiAgICAiIiJMb3dlc3QgdGVzdC1zZXQgRlBSIGFtb25nIHRocmVzaG9sZHMgd2l0aCBUUFIgPj0gdGFyZ2V0LiBVc2VzIHRoZSBsYWJlbHM7IG5vdCBvcGVyYXRpb25hbC4iIiINCiAgICBzY29yZXMgPSBucC5hc2FycmF5KHNjb3JlcywgZHR5cGU9bnAuZmxvYXQ2NCkNCiAgICBsYWJlbHMgPSBucC5hc2FycmF5KGxhYmVscywgZHR5cGU9aW50KQ0KICAgIGlmIHNjb3Jlcy5zaGFwZSAhPSBsYWJlbHMuc2hhcGUgb3Igc2NvcmVzLm5kaW0gIT0gMToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJTY29yZXMgYW5kIGxhYmVscyBtdXN0IGJlIDEtRCBhbmQgYWxpZ25lZC4iKQ0KICAgIGlmIG5vdCBucC5pc2Zpbml0ZShzY29yZXMpLmFsbCgpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vbi1maW5pdGUgdGVzdCBzY29yZXMuIikNCiAgICBkZWZlY3QgPSBzY29yZXNbbGFiZWxzID09IDFdDQogICAgZ29vZCA9IHNjb3Jlc1tsYWJlbHMgPT0gMF0NCiAgICBpZiBkZWZlY3Quc2l6ZSA9PSAwIG9yIGdvb2Quc2l6ZSA9PSAwOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk9yYWNsZSBGUFIgbmVlZHMgYm90aCBnb29kIGFuZCBkZWZlY3QgdGVzdCBpbWFnZXMuIikNCiAgICBjYW5kaWRhdGVzID0gbnAudW5pcXVlKG5wLmNvbmNhdGVuYXRlKFtzY29yZXMsIFtzY29yZXMubWluKCkgLSAxLjBdXSkpDQogICAgYmVzdCA9IDEuMA0KICAgIGZvdW5kID0gRmFsc2UNCiAgICBmb3IgdGhyZXNob2xkIGluIGNhbmRpZGF0ZXM6DQogICAgICAgIHRwciA9IGZsb2F0KG5wLm1lYW4oZGVmZWN0ID4gdGhyZXNob2xkKSkNCiAgICAgICAgaWYgdHByICsgMWUtMTIgPj0gdGFyZ2V0X3RwcjoNCiAgICAgICAgICAgIGZwciA9IGZsb2F0KG5wLm1lYW4oZ29vZCA+IHRocmVzaG9sZCkpDQogICAgICAgICAgICBpZiBub3QgZm91bmQgb3IgZnByIDwgYmVzdDoNCiAgICAgICAgICAgICAgICBiZXN0ID0gZnByDQogICAgICAgICAgICAgICAgZm91bmQgPSBUcnVlDQogICAgaWYgbm90IGZvdW5kOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vIHRocmVzaG9sZCByZWFjaGVzIHRoZSB0YXJnZXQgVFBSLiIpDQogICAgcmV0dXJuIGJlc3QNCg0KDQpkZWYgX21lZXRzKGNvdW50OiBpbnQsIHRvdGFsOiBpbnQsIGxldmVsOiBmbG9hdCwgYXRfbGVhc3Q6IGJvb2wpIC0+IGJvb2w6DQogICAgIiIiQ29tcGFyZSBjb3VudCAvIHRvdGFsIHdpdGggYSBsZXZlbCBpbiBpbnRlZ2VyIGFyaXRobWV0aWMsIHNvIDE5LzIwID49IDAuOTUgYW5kIDMvNjAgPD0gMC4wNSBob2xkIGV4YWN0bHkuIiIiDQogICAgZnJvbSBmcmFjdGlvbnMgaW1wb3J0IEZyYWN0aW9uDQoNCiAgICByYXRlID0gRnJhY3Rpb24oaW50KGNvdW50KSwgaW50KHRvdGFsKSkNCiAgICBib3VuZCA9IEZyYWN0aW9uKHN0cihsZXZlbCkpDQogICAgcmV0dXJuIGJvb2wocmF0ZSA+PSBib3VuZCkgaWYgYXRfbGVhc3QgZWxzZSBib29sKHJhdGUgPD0gYm91bmQpDQoNCg0KZGVmIGV2YWx1YXRlX3Njb3JlcygNCiAgICB0ZXN0X3Njb3JlczogbnAubmRhcnJheSwNCiAgICBsYWJlbHM6IG5wLm5kYXJyYXksDQogICAgY2FsaWJyYXRpb25fc2NvcmVzOiBucC5uZGFycmF5LA0KICAgIGFscGhhOiBmbG9hdCwNCiAgICBwcmFjdGljYWxfdHByOiBmbG9hdCwNCiAgICBwcmFjdGljYWxfZnByOiBmbG9hdCwNCikgLT4gZGljdDoNCiAgICAiIiJEZWNpc2lvbiBydWxlOiBhIHRlc3QgaW1hZ2UgaXMgcmVqZWN0ZWQgd2hlbiBpdHMgc2NvcmUgaXMgc3RyaWN0bHkgYWJvdmUgbWF4KEMpIChmb3IgYWxwaGE9MC4wNSwgbj0yMCkuIiIiDQogICAgdGVzdF9zY29yZXMgPSBucC5hc2FycmF5KHRlc3Rfc2NvcmVzLCBkdHlwZT1ucC5mbG9hdDY0KQ0KICAgIGxhYmVscyA9IG5wLmFzYXJyYXkobGFiZWxzLCBkdHlwZT1pbnQpDQogICAgY2FsaWJyYXRpb25fc2NvcmVzID0gbnAuYXNhcnJheShjYWxpYnJhdGlvbl9zY29yZXMsIGR0eXBlPW5wLmZsb2F0NjQpDQogICAgaWYgbm90IG5wLmlzZmluaXRlKHRlc3Rfc2NvcmVzKS5hbGwoKSBvciBub3QgbnAuaXNmaW5pdGUoY2FsaWJyYXRpb25fc2NvcmVzKS5hbGwoKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJOb24tZmluaXRlIHNjb3JlczsgcmVmdXNpbmcgdG8gcmVwbGFjZSB0aGVtIHdpdGggYSBzZW50aW5lbCB2YWx1ZS4iKQ0KICAgIHRocmVzaG9sZCwgcmFuayA9IHRocmVzaG9sZF9mcm9tX3Njb3JlcyhjYWxpYnJhdGlvbl9zY29yZXMsIGFscGhhKQ0KICAgIGRlZmVjdCA9IHRlc3Rfc2NvcmVzW2xhYmVscyA9PSAxXQ0KICAgIGdvb2QgPSB0ZXN0X3Njb3Jlc1tsYWJlbHMgPT0gMF0NCiAgICBpZiBkZWZlY3Quc2l6ZSA9PSAwIG9yIGdvb2Quc2l6ZSA9PSAwOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoZSBvZmZpY2lhbCB0ZXN0IHNwbGl0IG11c3QgY29udGFpbiBnb29kIGFuZCBkZWZlY3QgaW1hZ2VzLiIpDQogICAgbl90cCA9IGludChucC5zdW0oZGVmZWN0ID4gdGhyZXNob2xkKSkNCiAgICBuX2ZwID0gaW50KG5wLnN1bShnb29kID4gdGhyZXNob2xkKSkNCiAgICBuX2RlZmVjdCA9IGludChkZWZlY3Quc2l6ZSkNCiAgICBuX2dvb2QgPSBpbnQoZ29vZC5zaXplKQ0KICAgIHRwciA9IG5fdHAgLyBuX2RlZmVjdA0KICAgIGZwciA9IG5fZnAgLyBuX2dvb2QNCiAgICByZXR1cm4gew0KICAgICAgICAidGhyZXNob2xkIjogdGhyZXNob2xkLA0KICAgICAgICAidGhyZXNob2xkX3JhbmsiOiByYW5rLA0KICAgICAgICAibWVkaWFuX2NhbGlicmF0aW9uIjogZmxvYXQobnAubWVkaWFuKGNhbGlicmF0aW9uX3Njb3JlcykpLA0KICAgICAgICAidHByIjogZmxvYXQodHByKSwNCiAgICAgICAgInRwcl9sb3dlciI6IGNsb3BwZXJfcGVhcnNvbl9sb3dlcihuX3RwLCBuX2RlZmVjdCksDQogICAgICAgICJmcHIiOiBmbG9hdChmcHIpLA0KICAgICAgICAiZnByX3VwcGVyIjogY2xvcHBlcl9wZWFyc29uX3VwcGVyKG5fZnAsIG5fZ29vZCksDQogICAgICAgICJuX3RwIjogbl90cCwNCiAgICAgICAgIm5fZnAiOiBuX2ZwLA0KICAgICAgICAibl90ZXN0X2RlZmVjdCI6IG5fZGVmZWN0LA0KICAgICAgICAibl90ZXN0X2dvb2QiOiBuX2dvb2QsDQogICAgICAgICJhdXJvYyI6IGZsb2F0KHJvY19hdWNfc2NvcmUobGFiZWxzLCB0ZXN0X3Njb3JlcykpLA0KICAgICAgICAib3JhY2xlX2ZwciI6IG9yYWNsZV9mcHJfYXRfdHByKHRlc3Rfc2NvcmVzLCBsYWJlbHMsIHByYWN0aWNhbF90cHIpLA0KICAgICAgICAiY3JpdGVyaW9uIjogX21lZXRzKG5fdHAsIG5fZGVmZWN0LCBwcmFjdGljYWxfdHByLCBhdF9sZWFzdD1UcnVlKQ0KICAgICAgICBhbmQgX21lZXRzKG5fZnAsIG5fZ29vZCwgcHJhY3RpY2FsX2ZwciwgYXRfbGVhc3Q9RmFsc2UpLA0KICAgIH0NCg==", "commissioning/config.py": "IiIiTG9hZCB0aGUgZnJvemVuIGV4cGVyaW1lbnQgY29uZmlndXJhdGlvbiBhbmQgcmVzb2x2ZSBwYXRocy4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgaGFzaGxpYg0KaW1wb3J0IGpzb24NCmltcG9ydCBvcw0KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCmZyb20gdHlwaW5nIGltcG9ydCBBbnkNCg0KaW1wb3J0IHlhbWwNCg0KIyBTZWVkIGluZGljZXMgZm9sbG93IHRoaXMgb3JkZXIuIEl0IGlzIHBhcnQgb2YgdGhlIGhhc2hlZCBwYXlsb2FkIHZpYSBjb25maWcueWFtbC4NCk9CSkVDVFMgPSBbDQogICAgImJvdHRsZSIsDQogICAgImNhYmxlIiwNCiAgICAiY2Fwc3VsZSIsDQogICAgImhhemVsbnV0IiwNCiAgICAibWV0YWxfbnV0IiwNCiAgICAicGlsbCIsDQogICAgInNjcmV3IiwNCiAgICAidG9vdGhicnVzaCIsDQogICAgInRyYW5zaXN0b3IiLA0KICAgICJ6aXBwZXIiLA0KXQ0KVEVYVFVSRVMgPSBbImNhcnBldCIsICJncmlkIiwgImxlYXRoZXIiLCAidGlsZSIsICJ3b29kIl0NCg0KIyBPZmZpY2lhbCBNVlRlYyBBRCBjb3VudHMuIEEgbWlzbWF0Y2ggc3RvcHMgdGhlIHJ1biBpbnN0ZWFkIG9mIHNpbGVudGx5IHVzaW5nIHRoZSB3cm9uZyBzcGxpdC4NCkVYUEVDVEVEX0NPVU5UUzogZGljdFtzdHIsIGRpY3Rbc3RyLCBpbnRdXSA9IHsNCiAgICAiYm90dGxlIjogeyJ0cmFpbiI6IDIwOSwgInRlc3RfZ29vZCI6IDIwfSwNCiAgICAiY2FibGUiOiB7InRyYWluIjogMjI0LCAidGVzdF9nb29kIjogNTh9LA0KICAgICJjYXBzdWxlIjogeyJ0cmFpbiI6IDIxOSwgInRlc3RfZ29vZCI6IDIzfSwNCiAgICAiY2FycGV0IjogeyJ0cmFpbiI6IDI4MCwgInRlc3RfZ29vZCI6IDI4fSwNCiAgICAiZ3JpZCI6IHsidHJhaW4iOiAyNjQsICJ0ZXN0X2dvb2QiOiAyMX0sDQogICAgImhhemVsbnV0IjogeyJ0cmFpbiI6IDM5MSwgInRlc3RfZ29vZCI6IDQwfSwNCiAgICAibGVhdGhlciI6IHsidHJhaW4iOiAyNDUsICJ0ZXN0X2dvb2QiOiAzMn0sDQogICAgIm1ldGFsX251dCI6IHsidHJhaW4iOiAyMjAsICJ0ZXN0X2dvb2QiOiAyMn0sDQogICAgInBpbGwiOiB7InRyYWluIjogMjY3LCAidGVzdF9nb29kIjogMjZ9LA0KICAgICJzY3JldyI6IHsidHJhaW4iOiAzMjAsICJ0ZXN0X2dvb2QiOiA0MX0sDQogICAgInRpbGUiOiB7InRyYWluIjogMjMwLCAidGVzdF9nb29kIjogMzN9LA0KICAgICJ0b290aGJydXNoIjogeyJ0cmFpbiI6IDYwLCAidGVzdF9nb29kIjogMTJ9LA0KICAgICJ0cmFuc2lzdG9yIjogeyJ0cmFpbiI6IDIxMywgInRlc3RfZ29vZCI6IDYwfSwNCiAgICAid29vZCI6IHsidHJhaW4iOiAyNDcsICJ0ZXN0X2dvb2QiOiAxOX0sDQogICAgInppcHBlciI6IHsidHJhaW4iOiAyNDAsICJ0ZXN0X2dvb2QiOiAzMn0sDQp9DQoNCiMgTGFyZ2VzdCBrIHdpdGggayAvIG5fZ29vZCA8PSAwLjA1IG9uIHRoZSBvZmZpY2lhbCBnb29kIHRlc3QgaW1hZ2VzLg0KRkFMU0VfUkVKRUNUX0JVREdFVDogZGljdFtzdHIsIGludF0gPSB7DQogICAgInRvb3RoYnJ1c2giOiAwLA0KICAgICJ3b29kIjogMCwNCiAgICAiYm90dGxlIjogMSwNCiAgICAiZ3JpZCI6IDEsDQogICAgIm1ldGFsX251dCI6IDEsDQogICAgImNhcHN1bGUiOiAxLA0KICAgICJwaWxsIjogMSwNCiAgICAiY2FycGV0IjogMSwNCiAgICAibGVhdGhlciI6IDEsDQogICAgInppcHBlciI6IDEsDQogICAgInRpbGUiOiAxLA0KICAgICJoYXplbG51dCI6IDIsDQogICAgInNjcmV3IjogMiwNCiAgICAiY2FibGUiOiAyLA0KICAgICJ0cmFuc2lzdG9yIjogMywNCn0NCg0KDQpkZWYgcHJvamVjdF9yb290KCkgLT4gUGF0aDoNCiAgICByZXR1cm4gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudC5wYXJlbnQNCg0KDQpkZWYgX3Jlc29sdmVfcGF0aCh2YWx1ZTogc3RyIHwgUGF0aCwgYmFzZTogUGF0aCkgLT4gUGF0aDoNCiAgICBwYXRoID0gUGF0aCh2YWx1ZSkNCiAgICBpZiBwYXRoLmlzX2Fic29sdXRlKCk6DQogICAgICAgIHJldHVybiBwYXRoDQogICAgcmV0dXJuIChiYXNlIC8gcGF0aCkucmVzb2x2ZSgpDQoNCg0KQGRhdGFjbGFzcw0KY2xhc3MgQ29uZmlnOg0KICAgIHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldDQogICAgY29uZmlnX3BhdGg6IFBhdGgNCiAgICBkYXRhX3Jvb3Q6IFBhdGgNCiAgICB3b3JrX3Jvb3Q6IFBhdGgNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBwcm90b2NvbF92ZXJzaW9uKHNlbGYpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIGludChzZWxmLnBheWxvYWRbInByb3RvY29sX3ZlcnNpb24iXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBzZWVkKHNlbGYpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIGludChzZWxmLnBheWxvYWRbInNlZWQiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBuX2NhbGlicmF0aW9uKHNlbGYpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIGludChzZWxmLnBheWxvYWRbIm5fY2FsaWJyYXRpb24iXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBhbHBoYShzZWxmKSAtPiBmbG9hdDoNCiAgICAgICAgcmV0dXJuIGZsb2F0KHNlbGYucGF5bG9hZFsiYWxwaGEiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBuX3JlcGVhdHMoc2VsZikgLT4gaW50Og0KICAgICAgICByZXR1cm4gaW50KHNlbGYucGF5bG9hZFsibl9yZXBlYXRzIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgbl9ncmlkKHNlbGYpIC0+IGxpc3RbaW50XToNCiAgICAgICAgcmV0dXJuIFtpbnQobikgZm9yIG4gaW4gc2VsZi5wYXlsb2FkWyJuX2dyaWQiXV0NCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBzdHJhdGVnaWVzKHNlbGYpIC0+IGxpc3Rbc3RyXToNCiAgICAgICAgcmV0dXJuIFtzdHIocykgZm9yIHMgaW4gc2VsZi5wYXlsb2FkWyJzdHJhdGVnaWVzIl1dDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgYXVsY19uX21heChzZWxmKSAtPiBpbnQ6DQogICAgICAgIHJldHVybiBpbnQoc2VsZi5wYXlsb2FkWyJhdWxjX25fbWF4Il0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgcHJhY3RpY2FsX3RwcihzZWxmKSAtPiBmbG9hdDoNCiAgICAgICAgcmV0dXJuIGZsb2F0KHNlbGYucGF5bG9hZFsicHJhY3RpY2FsX3RwciJdKQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIHByYWN0aWNhbF9mcHIoc2VsZikgLT4gZmxvYXQ6DQogICAgICAgIHJldHVybiBmbG9hdChzZWxmLnBheWxvYWRbInByYWN0aWNhbF9mcHIiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBuX3N0YXJfbWluX3JlcGVhdHMoc2VsZikgLT4gaW50Og0KICAgICAgICByZXR1cm4gaW50KHNlbGYucGF5bG9hZFsibl9zdGFyX21pbl9yZXBlYXRzIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgZnByX2F1bGNfbWFyZ2luKHNlbGYpIC0+IGZsb2F0Og0KICAgICAgICByZXR1cm4gZmxvYXQoc2VsZi5wYXlsb2FkWyJmcHJfYXVsY19tYXJnaW4iXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBzdG9wcGluZ19kZWx0YXMoc2VsZikgLT4gbGlzdFtmbG9hdF06DQogICAgICAgIHJldHVybiBbZmxvYXQodikgZm9yIHYgaW4gc2VsZi5wYXlsb2FkWyJzdG9wcGluZ19kZWx0YXMiXV0NCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBzdG9wcGluZ19kZWx0YV9tYWluKHNlbGYpIC0+IGZsb2F0Og0KICAgICAgICByZXR1cm4gZmxvYXQoc2VsZi5wYXlsb2FkWyJzdG9wcGluZ19kZWx0YV9tYWluIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgYm9vdHN0cmFwX3JlcGxpY2F0ZXMoc2VsZikgLT4gaW50Og0KICAgICAgICByZXR1cm4gaW50KHNlbGYucGF5bG9hZFsiYm9vdHN0cmFwX3JlcGxpY2F0ZXMiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBib290c3RyYXBfc2VlZChzZWxmKSAtPiBpbnQ6DQogICAgICAgIHJldHVybiBpbnQoc2VsZi5wYXlsb2FkWyJib290c3RyYXBfc2VlZCJdKQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIGJhY2tib25lKHNlbGYpIC0+IHN0cjoNCiAgICAgICAgcmV0dXJuIHN0cihzZWxmLnBheWxvYWRbImJhY2tib25lIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgbGF5ZXJzKHNlbGYpIC0+IGxpc3Rbc3RyXToNCiAgICAgICAgcmV0dXJuIFtzdHIodikgZm9yIHYgaW4gc2VsZi5wYXlsb2FkWyJsYXllcnMiXV0NCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiByZXNpemUoc2VsZikgLT4gaW50Og0KICAgICAgICByZXR1cm4gaW50KHNlbGYucGF5bG9hZFsicmVzaXplIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgY2VudGVyX2Nyb3Aoc2VsZikgLT4gaW50Og0KICAgICAgICByZXR1cm4gaW50KHNlbGYucGF5bG9hZFsiY2VudGVyX2Nyb3AiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiB1cHNhbXBsZV9tb2RlKHNlbGYpIC0+IHN0cjoNCiAgICAgICAgcmV0dXJuIHN0cihzZWxmLnBheWxvYWRbInVwc2FtcGxlX21vZGUiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBpbWFnZW5ldF9tZWFuKHNlbGYpIC0+IGxpc3RbZmxvYXRdOg0KICAgICAgICByZXR1cm4gW2Zsb2F0KHYpIGZvciB2IGluIHNlbGYucGF5bG9hZFsiaW1hZ2VuZXRfbWVhbiJdXQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIGltYWdlbmV0X3N0ZChzZWxmKSAtPiBsaXN0W2Zsb2F0XToNCiAgICAgICAgcmV0dXJuIFtmbG9hdCh2KSBmb3IgdiBpbiBzZWxmLnBheWxvYWRbImltYWdlbmV0X3N0ZCJdXQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIGZlYXR1cmVfYmF0Y2hfc2l6ZShzZWxmKSAtPiBpbnQ6DQogICAgICAgIHJldHVybiBpbnQoc2VsZi5wYXlsb2FkWyJmZWF0dXJlX2JhdGNoX3NpemUiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBkaXN0YW5jZV9xdWVyeV9iYXRjaChzZWxmKSAtPiBpbnQ6DQogICAgICAgIHJldHVybiBpbnQoc2VsZi5wYXlsb2FkWyJkaXN0YW5jZV9xdWVyeV9iYXRjaCJdKQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIGRpc3RhbmNlX3RyYWluX2JhdGNoKHNlbGYpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIGludChzZWxmLnBheWxvYWRbImRpc3RhbmNlX3RyYWluX2JhdGNoIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgZTAoc2VsZikgLT4gZGljdFtzdHIsIEFueV06DQogICAgICAgIHJldHVybiBkaWN0KHNlbGYucGF5bG9hZFsiZTAiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBvYmplY3RzKHNlbGYpIC0+IGxpc3Rbc3RyXToNCiAgICAgICAgcmV0dXJuIFtzdHIodikgZm9yIHYgaW4gc2VsZi5wYXlsb2FkWyJvYmplY3RzIl1dDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgdGV4dHVyZXMoc2VsZikgLT4gbGlzdFtzdHJdOg0KICAgICAgICByZXR1cm4gW3N0cih2KSBmb3IgdiBpbiBzZWxmLnBheWxvYWRbInRleHR1cmVzIl1dDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgY2xhc3NlcyhzZWxmKSAtPiBsaXN0W3N0cl06DQogICAgICAgIHJldHVybiBzZWxmLm9iamVjdHMgKyBzZWxmLnRleHR1cmVzDQoNCiAgICBkZWYgY2xhc3NfaW5kZXgoc2VsZiwgY2xhc3NfbmFtZTogc3RyKSAtPiBpbnQ6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHJldHVybiBzZWxmLmNsYXNzZXMuaW5kZXgoY2xhc3NfbmFtZSkNCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3IgYXMgZXhjOg0KICAgICAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJVbmtub3duIGNsYXNzOiB7Y2xhc3NfbmFtZX0iKSBmcm9tIGV4Yw0KDQogICAgZGVmIGdyb3VwX29mKHNlbGYsIGNsYXNzX25hbWU6IHN0cikgLT4gc3RyOg0KICAgICAgICBpZiBjbGFzc19uYW1lIGluIHNlbGYub2JqZWN0czoNCiAgICAgICAgICAgIHJldHVybiAib2JqZWN0Ig0KICAgICAgICBpZiBjbGFzc19uYW1lIGluIHNlbGYudGV4dHVyZXM6DQogICAgICAgICAgICByZXR1cm4gInRleHR1cmUiDQogICAgICAgIHJhaXNlIEtleUVycm9yKGYiVW5rbm93biBjbGFzczoge2NsYXNzX25hbWV9IikNCg0KICAgIGRlZiBzZWxlY3RfY2xhc3NlcyhzZWxmLCByZXF1ZXN0ZWQ6IGxpc3Rbc3RyXSB8IE5vbmUpIC0+IGxpc3Rbc3RyXToNCiAgICAgICAgaWYgbm90IHJlcXVlc3RlZDoNCiAgICAgICAgICAgIHJldHVybiBsaXN0KHNlbGYuY2xhc3NlcykNCiAgICAgICAgdW5rbm93biA9IFtuYW1lIGZvciBuYW1lIGluIHJlcXVlc3RlZCBpZiBuYW1lIG5vdCBpbiBzZWxmLmNsYXNzZXNdDQogICAgICAgIGlmIHVua25vd246DQogICAgICAgICAgICByYWlzZSBLZXlFcnJvcihmIlVua25vd24gY2xhc3Nlczoge3Vua25vd259IikNCiAgICAgICAgcmV0dXJuIGxpc3QocmVxdWVzdGVkKQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIGhhc2goc2VsZikgLT4gc3RyOg0KICAgICAgICBzY2llbnRpZmljID0ge2tleTogdmFsdWUgZm9yIGtleSwgdmFsdWUgaW4gc2VsZi5wYXlsb2FkLml0ZW1zKCkgaWYga2V5ICE9ICJwYXRocyJ9DQogICAgICAgIGJsb2IgPSB7DQogICAgICAgICAgICAicGF5bG9hZCI6IHNjaWVudGlmaWMsDQogICAgICAgICAgICAiZXhwZWN0ZWRfY291bnRzIjogRVhQRUNURURfQ09VTlRTLA0KICAgICAgICAgICAgImZhbHNlX3JlamVjdF9idWRnZXQiOiBGQUxTRV9SRUpFQ1RfQlVER0VULA0KICAgICAgICB9DQogICAgICAgIHJhdyA9IGpzb24uZHVtcHMoYmxvYiwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSkNCiAgICAgICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHJhdy5lbmNvZGUoInV0Zi04IikpLmhleGRpZ2VzdCgpWzoxNl0NCg0KICAgIGRlZiB3b3JrKHNlbGYsICpwYXJ0czogc3RyKSAtPiBQYXRoOg0KICAgICAgICBwYXRoID0gc2VsZi53b3JrX3Jvb3Quam9pbnBhdGgoKnBhcnRzKQ0KICAgICAgICBwYXRoLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICAgICAgcmV0dXJuIHBhdGgNCg0KDQpkZWYgbG9hZF9jb25maWcoDQogICAgY29uZmlnX3BhdGg6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwNCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwNCiAgICB3b3JrX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwNCikgLT4gQ29uZmlnOg0KICAgIHJvb3QgPSBwcm9qZWN0X3Jvb3QoKQ0KICAgIHBhdGggPSBQYXRoKGNvbmZpZ19wYXRoKSBpZiBjb25maWdfcGF0aCBlbHNlIHJvb3QgLyAiY29uZmlnLnlhbWwiDQogICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkNvbmZpZyBub3QgZm91bmQ6IHtwYXRofSIpDQogICAgcGF5bG9hZCA9IHlhbWwuc2FmZV9sb2FkKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQ0KICAgIGlmIG5vdCBpc2luc3RhbmNlKHBheWxvYWQsIGRpY3QpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJDb25maWcgbXVzdCBiZSBhIG1hcHBpbmc6IHtwYXRofSIpDQoNCiAgICBfdmFsaWRhdGVfcGF5bG9hZChwYXlsb2FkKQ0KDQogICAgZGF0YV92YWx1ZSA9IGRhdGFfcm9vdCBvciBvcy5lbnZpcm9uLmdldCgiQ09NTUlTU0lPTklOR19EQVRBIikgb3IgcGF5bG9hZFsicGF0aHMiXVsiZGF0YV9yb290Il0NCiAgICB3b3JrX3ZhbHVlID0gd29ya19yb290IG9yIG9zLmVudmlyb24uZ2V0KCJDT01NSVNTSU9OSU5HX1dPUksiKSBvciBwYXlsb2FkWyJwYXRocyJdWyJ3b3JrX3Jvb3QiXQ0KICAgIHJldHVybiBDb25maWcoDQogICAgICAgIHBheWxvYWQ9cGF5bG9hZCwNCiAgICAgICAgY29uZmlnX3BhdGg9cGF0aC5yZXNvbHZlKCksDQogICAgICAgIGRhdGFfcm9vdD1fcmVzb2x2ZV9wYXRoKGRhdGFfdmFsdWUsIHJvb3QpLA0KICAgICAgICB3b3JrX3Jvb3Q9X3Jlc29sdmVfcGF0aCh3b3JrX3ZhbHVlLCByb290KSwNCiAgICApDQoNCg0KZGVmIF92YWxpZGF0ZV9wYXlsb2FkKHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldKSAtPiBOb25lOg0KICAgIGlmIGxpc3QocGF5bG9hZC5nZXQoIm9iamVjdHMiLCBbXSkpICE9IE9CSkVDVFMgb3IgbGlzdChwYXlsb2FkLmdldCgidGV4dHVyZXMiLCBbXSkpICE9IFRFWFRVUkVTOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIm9iamVjdHMvdGV4dHVyZXMgaW4gY29uZmlnLnlhbWwgbXVzdCBtYXRjaCB0aGUgY2Fub25pY2FsIGxpc3RzIGluIGNvbmZpZy5weS4iKQ0KICAgIGNsYXNzZXMgPSBPQkpFQ1RTICsgVEVYVFVSRVMNCiAgICBpZiBzZXQoRVhQRUNURURfQ09VTlRTKSAhPSBzZXQoY2xhc3Nlcykgb3Igc2V0KEZBTFNFX1JFSkVDVF9CVURHRVQpICE9IHNldChjbGFzc2VzKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJFWFBFQ1RFRF9DT1VOVFMgYW5kIEZBTFNFX1JFSkVDVF9CVURHRVQgbXVzdCBjb3ZlciBldmVyeSBNVlRlYyBjbGFzcy4iKQ0KICAgIGlmIGludChwYXlsb2FkWyJuX2NhbGlicmF0aW9uIl0pICE9IDIwIG9yIGZsb2F0KHBheWxvYWRbImFscGhhIl0pICE9IDAuMDU6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiVGhpcyBwcm90b2NvbCBmaXhlcyBuX2NhbGlicmF0aW9uPTIwIGFuZCBhbHBoYT0wLjA1LCBzbyB0aGUgdGhyZXNob2xkIGlzIG1heChDKS4iKQ0KICAgIGlmICJyYW5kb20iIG5vdCBpbiBwYXlsb2FkWyJzdHJhdGVnaWVzIl0gb3IgImtjZW50ZXIiIG5vdCBpbiBwYXlsb2FkWyJzdHJhdGVnaWVzIl06DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigic3RyYXRlZ2llcyBtdXN0IGluY2x1ZGUgcmFuZG9tIGFuZCBrY2VudGVyLiIpDQogICAgaWYgaW50KHBheWxvYWRbImF1bGNfbl9tYXgiXSkgIT0gMzI6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTWFpbiBBVUxDIGlzIGZpeGVkIG9uIE4gPD0gMzIuIikNCiAgICBncmlkID0gW2ludChuKSBmb3IgbiBpbiBwYXlsb2FkWyJuX2dyaWQiXV0NCiAgICBpZiBncmlkICE9IHNvcnRlZChncmlkKSBvciBsZW4oZ3JpZCkgIT0gbGVuKHNldChncmlkKSk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIm5fZ3JpZCBtdXN0IGJlIHN0cmljdGx5IGluY3JlYXNpbmc6IHtncmlkfSIpDQogICAgaWYgMSBub3QgaW4gZ3JpZCBvciAzMiBub3QgaW4gZ3JpZDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJuX2dyaWQgbXVzdCBjb250YWluIDEgYW5kIDMyLiIpDQo=", "commissioning/data.py": "IiIiTVZUZWMgQUQgZGlzY292ZXJ5LCBvZmZpY2lhbC1jb3VudCBjaGVja3MsIGFuZCBwZXJtYW5lbnQgQy9QIHNwbGl0cy4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgaGFzaGxpYg0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCBudW1weSBhcyBucA0KDQpmcm9tIGNvbW1pc3Npb25pbmcuY29uZmlnIGltcG9ydCBFWFBFQ1RFRF9DT1VOVFMsIENvbmZpZw0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCByZWFkX2pzb24sIHdyaXRlX2pzb24NCg0KSU1BR0VfU1VGRklYRVMgPSB7Ii5wbmciLCAiLmpwZyIsICIuanBlZyIsICIuYm1wIiwgIi50aWYiLCAiLnRpZmYifQ0KDQoNCmRlZiBpdGVyX2ltYWdlcyhkaXJlY3Rvcnk6IFBhdGgpIC0+IGxpc3RbUGF0aF06DQogICAgaWYgbm90IGRpcmVjdG9yeS5pc19kaXIoKToNCiAgICAgICAgcmV0dXJuIFtdDQogICAgZmlsZXMgPSBbcGF0aCBmb3IgcGF0aCBpbiBkaXJlY3RvcnkuaXRlcmRpcigpIGlmIHBhdGguaXNfZmlsZSgpIGFuZCBwYXRoLnN1ZmZpeC5sb3dlcigpIGluIElNQUdFX1NVRkZJWEVTXQ0KICAgIHJldHVybiBzb3J0ZWQoZmlsZXMsIGtleT1sYW1iZGEgcGF0aDogcGF0aC5uYW1lLmxvd2VyKCkpDQoNCg0KZGVmIGNvbGxlY3RfY2xhc3MoZGF0YV9yb290OiBQYXRoLCBjbGFzc19uYW1lOiBzdHIpIC0+IGRpY3Rbc3RyLCBsaXN0XToNCiAgICByb290ID0gZGF0YV9yb290IC8gY2xhc3NfbmFtZQ0KICAgIGlmIG5vdCByb290LmlzX2RpcigpOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIntjbGFzc19uYW1lfTogZGF0YXNldCBkaXJlY3Rvcnkgbm90IGZvdW5kOiB7cm9vdH0iKQ0KICAgIHRyYWluID0gaXRlcl9pbWFnZXMocm9vdCAvICJ0cmFpbiIgLyAiZ29vZCIpDQogICAgdGVzdF9nb29kID0gaXRlcl9pbWFnZXMocm9vdCAvICJ0ZXN0IiAvICJnb29kIikNCiAgICB0ZXN0X3Jvb3QgPSByb290IC8gInRlc3QiDQogICAgZGVmZWN0czogbGlzdFt0dXBsZVtzdHIsIFBhdGhdXSA9IFtdDQogICAgaWYgdGVzdF9yb290LmlzX2RpcigpOg0KICAgICAgICBmb3Igc3ViIGluIHNvcnRlZCgocGF0aCBmb3IgcGF0aCBpbiB0ZXN0X3Jvb3QuaXRlcmRpcigpIGlmIHBhdGguaXNfZGlyKCkpLCBrZXk9bGFtYmRhIHBhdGg6IHBhdGgubmFtZS5sb3dlcigpKToNCiAgICAgICAgICAgIGlmIHN1Yi5uYW1lLmxvd2VyKCkgPT0gImdvb2QiOg0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICBkZWZlY3RzLmV4dGVuZCgoc3ViLm5hbWUsIGltYWdlKSBmb3IgaW1hZ2UgaW4gaXRlcl9pbWFnZXMoc3ViKSkNCiAgICByZXR1cm4geyJ0cmFpbiI6IHRyYWluLCAidGVzdF9nb29kIjogdGVzdF9nb29kLCAidGVzdF9kZWZlY3QiOiBkZWZlY3RzfQ0KDQoNCmRlZiBhc3NlcnRfb2ZmaWNpYWxfY291bnRzKGNsYXNzX25hbWU6IHN0ciwgY29sbGVjdGVkOiBkaWN0W3N0ciwgbGlzdF0pIC0+IE5vbmU6DQogICAgZXhwZWN0ZWQgPSBFWFBFQ1RFRF9DT1VOVFNbY2xhc3NfbmFtZV0NCiAgICBmb3VuZCA9IHsidHJhaW4iOiBsZW4oY29sbGVjdGVkWyJ0cmFpbiJdKSwgInRlc3RfZ29vZCI6IGxlbihjb2xsZWN0ZWRbInRlc3RfZ29vZCJdKX0NCiAgICBpZiBmb3VuZCAhPSBleHBlY3RlZDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBleHBlY3RlZCBjb3VudHMge2V4cGVjdGVkfSwgZm91bmQge2ZvdW5kfSIpDQogICAgaWYgbGVuKGNvbGxlY3RlZFsidGVzdF9kZWZlY3QiXSkgPCAxOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IG5vIGRlZmVjdCBpbWFnZXMgdW5kZXIgdGVzdC8iKQ0KICAgIF9hc3NlcnRfbm9fZHVwbGljYXRlX2ZpbGVzKGNsYXNzX25hbWUsIGNvbGxlY3RlZCkNCg0KDQpkZWYgX2Fzc2VydF9ub19kdXBsaWNhdGVfZmlsZXMoY2xhc3NfbmFtZTogc3RyLCBjb2xsZWN0ZWQ6IGRpY3Rbc3RyLCBsaXN0XSkgLT4gTm9uZToNCiAgICBzZWVuOiBkaWN0W3N0ciwgc3RyXSA9IHt9DQogICAgcGF0aHM6IGxpc3RbUGF0aF0gPSBsaXN0KGNvbGxlY3RlZFsidHJhaW4iXSkgKyBsaXN0KGNvbGxlY3RlZFsidGVzdF9nb29kIl0pDQogICAgcGF0aHMuZXh0ZW5kKHBhdGggZm9yIF8sIHBhdGggaW4gY29sbGVjdGVkWyJ0ZXN0X2RlZmVjdCJdKQ0KICAgIGZvciBwYXRoIGluIHBhdGhzOg0KICAgICAgICBkaWdlc3QgPSBzaGExX2ZpbGUocGF0aCkNCiAgICAgICAgcHJldmlvdXMgPSBzZWVuLmdldChkaWdlc3QpDQogICAgICAgIGlmIHByZXZpb3VzIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBkdXBsaWNhdGUgaW1hZ2UgY29udGVudCB7cHJldmlvdXN9IGFuZCB7cGF0aH0iKQ0KICAgICAgICBzZWVuW2RpZ2VzdF0gPSBzdHIocGF0aCkNCg0KDQpkZWYgc2hhMV9maWxlKHBhdGg6IFBhdGgsIGNodW5rOiBpbnQgPSAxIDw8IDIwKSAtPiBzdHI6DQogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGExKCkNCiAgICB3aXRoIFBhdGgocGF0aCkub3BlbigicmIiKSBhcyBoYW5kbGU6DQogICAgICAgIHdoaWxlIFRydWU6DQogICAgICAgICAgICBibG9jayA9IGhhbmRsZS5yZWFkKGNodW5rKQ0KICAgICAgICAgICAgaWYgbm90IGJsb2NrOg0KICAgICAgICAgICAgICAgIGJyZWFrDQogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGJsb2NrKQ0KICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkNCg0KDQpkZWYgcmVsYXRpdmVfcGF0aChwYXRoOiBQYXRoLCBkYXRhX3Jvb3Q6IFBhdGgpIC0+IHN0cjoNCiAgICByZXR1cm4gcGF0aC5yZXNvbHZlKCkucmVsYXRpdmVfdG8oZGF0YV9yb290LnJlc29sdmUoKSkuYXNfcG9zaXgoKQ0KDQoNCmRlZiBhYnNvbHV0ZV9wYXRoKGRhdGFfcm9vdDogUGF0aCwgcmVsYXRpdmU6IHN0cikgLT4gUGF0aDoNCiAgICByZXR1cm4gZGF0YV9yb290IC8gUGF0aChyZWxhdGl2ZSkNCg0KDQpkZWYgc3BsaXRfc2VlZHMoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0ciwgcmVwOiBpbnQpIC0+IHR1cGxlW25wLnJhbmRvbS5HZW5lcmF0b3IsIG5wLnJhbmRvbS5HZW5lcmF0b3IsIG5wLnJhbmRvbS5HZW5lcmF0b3JdOg0KICAgICIiIkluZGVwZW5kZW50IHN0cmVhbXMgc28gQy9QIGRvZXMgbm90IGRlcGVuZCBvbiB3aGljaCBzdHJhdGVneSBydW5zIGZpcnN0LiIiIg0KICAgIHNlcXVlbmNlID0gbnAucmFuZG9tLlNlZWRTZXF1ZW5jZShbY2ZnLnNlZWQsIGNmZy5jbGFzc19pbmRleChjbGFzc19uYW1lKSwgaW50KHJlcCldKQ0KICAgIHNwbGl0X3NlZWQsIHJhbmRvbV9zZWVkLCBrY2VudGVyX3NlZWQgPSBzZXF1ZW5jZS5zcGF3bigzKQ0KICAgIHJldHVybiAoDQogICAgICAgIG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzcGxpdF9zZWVkKSwNCiAgICAgICAgbnAucmFuZG9tLmRlZmF1bHRfcm5nKHJhbmRvbV9zZWVkKSwNCiAgICAgICAgbnAucmFuZG9tLmRlZmF1bHRfcm5nKGtjZW50ZXJfc2VlZCksDQogICAgKQ0KDQoNCmRlZiBzcGxpdF90cmFpbl9pbmRpY2VzKG5fdHJhaW46IGludCwgbl9jYWxpYnJhdGlvbjogaW50LCBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOg0KICAgIGlmIG5fdHJhaW4gPD0gbl9jYWxpYnJhdGlvbjoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTmVlZCBtb3JlIHRoYW4ge25fY2FsaWJyYXRpb259IHRyYWluaW5nIGltYWdlcywgZm91bmQge25fdHJhaW59LiIpDQogICAgY2FsaWJyYXRpb24gPSBucC5zb3J0KHJuZy5jaG9pY2Uobl90cmFpbiwgc2l6ZT1uX2NhbGlicmF0aW9uLCByZXBsYWNlPUZhbHNlKS5hc3R5cGUoaW50KSkNCiAgICBwb29sX21hc2sgPSBucC5vbmVzKG5fdHJhaW4sIGR0eXBlPWJvb2wpDQogICAgcG9vbF9tYXNrW2NhbGlicmF0aW9uXSA9IEZhbHNlDQogICAgcG9vbCA9IG5wLmZsYXRub256ZXJvKHBvb2xfbWFzaykuYXN0eXBlKGludCkNCiAgICBfYXNzZXJ0X3NwbGl0X2luZGljZXMoY2FsaWJyYXRpb24sIHBvb2wsIG5fdHJhaW4sIG5fY2FsaWJyYXRpb24pDQogICAgcmV0dXJuIGNhbGlicmF0aW9uLCBwb29sDQoNCg0KZGVmIF9hc3NlcnRfc3BsaXRfaW5kaWNlcyhjYWxpYnJhdGlvbjogbnAubmRhcnJheSwgcG9vbDogbnAubmRhcnJheSwgbl90cmFpbjogaW50LCBuX2NhbGlicmF0aW9uOiBpbnQpIC0+IE5vbmU6DQogICAgaWYgbGVuKGNhbGlicmF0aW9uKSAhPSBuX2NhbGlicmF0aW9uOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ8Q3wgPSB7bGVuKGNhbGlicmF0aW9uKX0sIGV4cGVjdGVkIHtuX2NhbGlicmF0aW9ufSIpDQogICAgaWYgbGVuKGNhbGlicmF0aW9uKSArIGxlbihwb29sKSAhPSBuX3RyYWluOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoInxDfCArIHxQfCBtdXN0IGVxdWFsIHRoZSB0cmFpbmluZyBzZXQuIikNCiAgICBpZiBsZW4obnAuaW50ZXJzZWN0MWQoY2FsaWJyYXRpb24sIHBvb2wpKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJDIGFuZCBQIG92ZXJsYXAuIikNCiAgICBpZiBsZW4obnAudW5pcXVlKGNhbGlicmF0aW9uKSkgIT0gbGVuKGNhbGlicmF0aW9uKSBvciBsZW4obnAudW5pcXVlKHBvb2wpKSAhPSBsZW4ocG9vbCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRHVwbGljYXRlIGluZGljZXMgaW5zaWRlIEMgb3IgUC4iKQ0KDQoNCmRlZiBzcGxpdF9wYXRoKGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIHJlcDogaW50KSAtPiBQYXRoOg0KICAgIHJldHVybiBjZmcud29ya19yb290IC8gInNwbGl0cyIgLyBjbGFzc19uYW1lIC8gZiJyZXBfe3JlcDowMmR9Lmpzb24iDQoNCg0KZGVmIGJ1aWxkX3NwbGl0X3BheWxvYWQoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0ciwgcmVwOiBpbnQsIGNvbGxlY3RlZDogZGljdFtzdHIsIGxpc3RdKSAtPiBkaWN0Og0KICAgIHJuZywgXywgXyA9IHNwbGl0X3NlZWRzKGNmZywgY2xhc3NfbmFtZSwgcmVwKQ0KICAgIGNhbGlicmF0aW9uX2lkeCwgcG9vbF9pZHggPSBzcGxpdF90cmFpbl9pbmRpY2VzKGxlbihjb2xsZWN0ZWRbInRyYWluIl0pLCBjZmcubl9jYWxpYnJhdGlvbiwgcm5nKQ0KICAgIHRyYWluX3JlbCA9IFtyZWxhdGl2ZV9wYXRoKHBhdGgsIGNmZy5kYXRhX3Jvb3QpIGZvciBwYXRoIGluIGNvbGxlY3RlZFsidHJhaW4iXV0NCiAgICB0ZXN0X2dvb2RfcmVsID0gW3JlbGF0aXZlX3BhdGgocGF0aCwgY2ZnLmRhdGFfcm9vdCkgZm9yIHBhdGggaW4gY29sbGVjdGVkWyJ0ZXN0X2dvb2QiXV0NCiAgICB0ZXN0X2RlZmVjdCA9IFsNCiAgICAgICAgeyJwYXRoIjogcmVsYXRpdmVfcGF0aChwYXRoLCBjZmcuZGF0YV9yb290KSwgImRlZmVjdF90eXBlIjogZGVmZWN0X3R5cGV9DQogICAgICAgIGZvciBkZWZlY3RfdHlwZSwgcGF0aCBpbiBjb2xsZWN0ZWRbInRlc3RfZGVmZWN0Il0NCiAgICBdDQogICAgY2FsaWJyYXRpb24gPSBbdHJhaW5fcmVsW2ludChpKV0gZm9yIGkgaW4gY2FsaWJyYXRpb25faWR4XQ0KICAgIHBvb2wgPSBbdHJhaW5fcmVsW2ludChpKV0gZm9yIGkgaW4gcG9vbF9pZHhdDQogICAgdGVzdF9wYXRocyA9IHRlc3RfZ29vZF9yZWwgKyBbaXRlbVsicGF0aCJdIGZvciBpdGVtIGluIHRlc3RfZGVmZWN0XQ0KICAgIF9hc3NlcnRfc3BsaXRfcGF0aHMoY2xhc3NfbmFtZSwgY2FsaWJyYXRpb24sIHBvb2wsIHRlc3RfcGF0aHMpDQogICAgcmV0dXJuIHsNCiAgICAgICAgImNvbmZpZ19oYXNoIjogY2ZnLmhhc2gsDQogICAgICAgICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwNCiAgICAgICAgImNsYXNzX2luZGV4IjogY2ZnLmNsYXNzX2luZGV4KGNsYXNzX25hbWUpLA0KICAgICAgICAicmVwIjogaW50KHJlcCksDQogICAgICAgICJzZWVkIjogY2ZnLnNlZWQsDQogICAgICAgICJjYWxpYnJhdGlvbiI6IGNhbGlicmF0aW9uLA0KICAgICAgICAicG9vbCI6IHBvb2wsDQogICAgICAgICJ0ZXN0X2dvb2QiOiB0ZXN0X2dvb2RfcmVsLA0KICAgICAgICAidGVzdF9kZWZlY3QiOiB0ZXN0X2RlZmVjdCwNCiAgICAgICAgIm5fdHJhaW4iOiBsZW4odHJhaW5fcmVsKSwNCiAgICAgICAgIm5fdGVzdF9nb29kIjogbGVuKHRlc3RfZ29vZF9yZWwpLA0KICAgICAgICAibl90ZXN0X2RlZmVjdCI6IGxlbih0ZXN0X2RlZmVjdCksDQogICAgfQ0KDQoNCmRlZiBfYXNzZXJ0X3NwbGl0X3BhdGhzKGNsYXNzX25hbWU6IHN0ciwgY2FsaWJyYXRpb246IGxpc3Rbc3RyXSwgcG9vbDogbGlzdFtzdHJdLCB0ZXN0X3BhdGhzOiBsaXN0W3N0cl0pIC0+IE5vbmU6DQogICAgaWYgbGVuKGNhbGlicmF0aW9uKSAhPSBsZW4oc2V0KGNhbGlicmF0aW9uKSkgb3IgbGVuKHBvb2wpICE9IGxlbihzZXQocG9vbCkpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGR1cGxpY2F0ZSBwYXRocyBpbnNpZGUgQyBvciBQLiIpDQogICAgaWYgc2V0KGNhbGlicmF0aW9uKSAmIHNldChwb29sKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBDIG92ZXJsYXBzIFAuIikNCiAgICBpZiBzZXQoY2FsaWJyYXRpb24pICYgc2V0KHRlc3RfcGF0aHMpIG9yIHNldChwb29sKSAmIHNldCh0ZXN0X3BhdGhzKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiB0cmFpbiBzcGxpdCBvdmVybGFwcyB0aGUgb2ZmaWNpYWwgdGVzdCBzZXQuIikNCiAgICBpZiBsZW4odGVzdF9wYXRocykgIT0gbGVuKHNldCh0ZXN0X3BhdGhzKSk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogZHVwbGljYXRlIHRlc3QgcGF0aHMuIikNCg0KDQpkZWYgcHJlcGFyZV9jbGFzcyhjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyLCByZXBzOiByYW5nZSB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIGNvbGxlY3RlZCA9IGNvbGxlY3RfY2xhc3MoY2ZnLmRhdGFfcm9vdCwgY2xhc3NfbmFtZSkNCiAgICBhc3NlcnRfb2ZmaWNpYWxfY291bnRzKGNsYXNzX25hbWUsIGNvbGxlY3RlZCkNCiAgICByZXBfcmFuZ2UgPSByYW5nZShjZmcubl9yZXBlYXRzKSBpZiByZXBzIGlzIE5vbmUgZWxzZSByZXBzDQogICAgZm9yIHJlcCBpbiByZXBfcmFuZ2U6DQogICAgICAgIHBhdGggPSBzcGxpdF9wYXRoKGNmZywgY2xhc3NfbmFtZSwgcmVwKQ0KICAgICAgICBwYXlsb2FkID0gYnVpbGRfc3BsaXRfcGF5bG9hZChjZmcsIGNsYXNzX25hbWUsIHJlcCwgY29sbGVjdGVkKQ0KICAgICAgICBpZiBwYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgICAgIGV4aXN0aW5nID0gcmVhZF9qc29uKHBhdGgpDQogICAgICAgICAgICBpZiBleGlzdGluZyAhPSBwYXlsb2FkOg0KICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigNCiAgICAgICAgICAgICAgICAgICAgZiJSZWZ1c2luZyB0byBvdmVyd3JpdGUgYW4gZXhpc3Rpbmcgc3BsaXQgdGhhdCBkb2VzIG5vdCBtYXRjaCB0aGlzIGNvbmZpZzoge3BhdGh9Ig0KICAgICAgICAgICAgICAgICkNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHdyaXRlX2pzb24ocGF0aCwgcGF5bG9hZCkNCg0KDQpkZWYgcHJlcGFyZShjZmc6IENvbmZpZywgY2xhc3NlczogbGlzdFtzdHJdKSAtPiBOb25lOg0KICAgIGZvciBjbGFzc19uYW1lIGluIGNsYXNzZXM6DQogICAgICAgIHByZXBhcmVfY2xhc3MoY2ZnLCBjbGFzc19uYW1lKQ0KDQoNCmRlZiBsb2FkX3NwbGl0KGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIHJlcDogaW50KSAtPiBkaWN0Og0KICAgIHBhdGggPSBzcGxpdF9wYXRoKGNmZywgY2xhc3NfbmFtZSwgcmVwKQ0KICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNaXNzaW5nIHNwbGl0IHtwYXRofS4gUnVuIHByZXBhcmUgZmlyc3QuIikNCiAgICBwYXlsb2FkID0gcmVhZF9qc29uKHBhdGgpDQogICAgaWYgcGF5bG9hZC5nZXQoImNvbmZpZ19oYXNoIikgIT0gY2ZnLmhhc2g6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlNwbGl0IGhhc2ggbWlzbWF0Y2ggZm9yIHtjbGFzc19uYW1lfSByZXAge3JlcH0uIFRoZSBzcGxpdCB3YXMgYnVpbHQgZnJvbSBhbm90aGVyIGNvbmZpZy4iKQ0KICAgIF9hc3NlcnRfc3BsaXRfcGF0aHMoDQogICAgICAgIGNsYXNzX25hbWUsDQogICAgICAgIHBheWxvYWRbImNhbGlicmF0aW9uIl0sDQogICAgICAgIHBheWxvYWRbInBvb2wiXSwNCiAgICAgICAgbGlzdChwYXlsb2FkWyJ0ZXN0X2dvb2QiXSkgKyBbaXRlbVsicGF0aCJdIGZvciBpdGVtIGluIHBheWxvYWRbInRlc3RfZGVmZWN0Il1dLA0KICAgICkNCiAgICByZXR1cm4gcGF5bG9hZA0K", "commissioning/distances.py": "IiIiUGVyLWNsYXNzIGRpc3RhbmNlIHRlbnNvciBEW3F1ZXJ5LCBwYXRjaCwgdHJhaW4gaW1hZ2VdLg0KDQpEW3EsIHAsIGpdIGlzIHRoZSBFdWNsaWRlYW4gZGlzdGFuY2UgZnJvbSBwYXRjaCBwIG9mIHF1ZXJ5IHEgdG8gaXRzIG5lYXJlc3QNCnBhdGNoIGluIHRyYWluaW5nIGltYWdlIGouIFRoZSBpbWFnZSBzY29yZSBmb3IgYSBiYW5rIFMgaXMNCg0KICAgIG1heCBvdmVyIHAgb2YgbWluIG92ZXIgaiBpbiBTIG9mIERbcSwgcCwgal0NCg0Kd2hpY2ggaXMgdGhlIHNhbWUgMS1OTiBtYXhpbXVtIHVzZWQgYnkgUGF0Y2hDb3JlIHdpdGhvdXQgYSBjb3Jlc2V0Lg0KIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGxvZ2dpbmcNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KZnJvbSBjb21taXNzaW9uaW5nLmNvbmZpZyBpbXBvcnQgQ29uZmlnDQpmcm9tIGNvbW1pc3Npb25pbmcuZmVhdHVyZXMgaW1wb3J0IGxvYWRfZmVhdHVyZV9jYWNoZQ0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCBhdG9taWNfbnB5LCByZWFkX2pzb24sIHdyaXRlX2pzb24NCg0KTE9HR0VSID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pDQoNCg0KZGVmIGRpc3RhbmNlX2NhY2hlX3BhdGgoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gUGF0aDoNCiAgICByZXR1cm4gY2ZnLndvcmtfcm9vdCAvICJjYWNoZSIgLyAiZGlzdCIgLyBmIntjbGFzc19uYW1lfS5ucHkiDQoNCg0KZGVmIGRpc3RhbmNlX21ldGFfcGF0aChjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyKSAtPiBQYXRoOg0KICAgIHJldHVybiBjZmcud29ya19yb290IC8gImNhY2hlIiAvICJkaXN0IiAvIGYie2NsYXNzX25hbWV9Lmpzb24iDQoNCg0KZGVmIG1pbl9wYXRjaF9kaXN0YW5jZV9ibG9jayhxdWVyeTogbnAubmRhcnJheSwgdHJhaW46IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6DQogICAgIiIicXVlcnkgKEJxLCBQLCBDKSwgdHJhaW4gKEJ0LCBQLCBDKSAtPiAoQnEsIFAsIEJ0KSBtaW5pbXVtIG92ZXIgdHJhaW4gcGF0Y2hlcy4NCg0KICAgIE51bVB5IHJlZmVyZW5jZSB1c2VkIGJ5IHRlc3RzIGFuZCBieSB0aGUgQ1BVIGZhbGxiYWNrLiBQcm9kdWN0aW9uIHJ1bnMgdXNlIHRvcmNoLg0KICAgICIiIg0KICAgIHF1ZXJ5ID0gbnAuYXNjb250aWd1b3VzYXJyYXkocXVlcnksIGR0eXBlPW5wLmZsb2F0MzIpDQogICAgdHJhaW4gPSBucC5hc2NvbnRpZ3VvdXNhcnJheSh0cmFpbiwgZHR5cGU9bnAuZmxvYXQzMikNCiAgICBicSwgcGF0Y2hlcywgY2hhbm5lbHMgPSBxdWVyeS5zaGFwZQ0KICAgIGJ0ID0gdHJhaW4uc2hhcGVbMF0NCiAgICBpZiB0cmFpbi5zaGFwZVsxOl0gIT0gKHBhdGNoZXMsIGNoYW5uZWxzKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiUGF0Y2ggc2hhcGUgbWlzbWF0Y2g6IHF1ZXJ5IHtxdWVyeS5zaGFwZX0sIHRyYWluIHt0cmFpbi5zaGFwZX0iKQ0KICAgIGRpc3RhbmNlcyA9IG5wLmVtcHR5KChicSwgcGF0Y2hlcywgYnQpLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgIHF1ZXJ5X2ZsYXQgPSBxdWVyeS5yZXNoYXBlKGJxICogcGF0Y2hlcywgY2hhbm5lbHMpDQogICAgZm9yIHRyYWluX2luZGV4IGluIHJhbmdlKGJ0KToNCiAgICAgICAgZGVsdGEgPSBxdWVyeV9mbGF0WzosIE5vbmUsIDpdIC0gdHJhaW5bdHJhaW5faW5kZXhdW05vbmUsIDosIDpdDQogICAgICAgIG5lYXJlc3QgPSBucC5zcXJ0KG5wLnN1bShkZWx0YSAqIGRlbHRhLCBheGlzPS0xKSkubWluKGF4aXM9MSkNCiAgICAgICAgZGlzdGFuY2VzWzosIDosIHRyYWluX2luZGV4XSA9IG5lYXJlc3QucmVzaGFwZShicSwgcGF0Y2hlcykNCiAgICByZXR1cm4gZGlzdGFuY2VzDQoNCg0KZGVmIF9kaXN0YW5jZV90ZW5zb3JfdG9yY2goDQogICAgcGF0Y2hlczogbnAubmRhcnJheSwNCiAgICBuX3RyYWluOiBpbnQsDQogICAgcXVlcnlfYmF0Y2g6IGludCwNCiAgICB0cmFpbl9iYXRjaDogaW50LA0KICAgIGRldmljZTogc3RyLA0KKSAtPiBucC5uZGFycmF5Og0KICAgICIiIktlZXBzIHRoZSBmbG9hdDE2IGNhY2hlIG9uIHRoZSBkZXZpY2Ugb25jZTsgZWFjaCBibG9jayBpcyBjYXN0IHRvIGZsb2F0MzIgYmVmb3JlIGNkaXN0LiIiIg0KICAgIGltcG9ydCB0b3JjaA0KDQogICAgbl9pbWFnZXMsIG5fcGF0Y2hlcywgY2hhbm5lbHMgPSBwYXRjaGVzLnNoYXBlDQogICAgcmVzaWRlbnQgPSB0b3JjaC5mcm9tX251bXB5KG5wLmFzY29udGlndW91c2FycmF5KHBhdGNoZXMpKS50byhkZXZpY2UpDQogICAgb3V0cHV0ID0gbnAuZW1wdHkoKG5faW1hZ2VzLCBuX3BhdGNoZXMsIG5fdHJhaW4pLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICBmb3IgcTAgaW4gcmFuZ2UoMCwgbl9pbWFnZXMsIHF1ZXJ5X2JhdGNoKToNCiAgICAgICAgICAgIHExID0gbWluKHEwICsgcXVlcnlfYmF0Y2gsIG5faW1hZ2VzKQ0KICAgICAgICAgICAgaWYgcTAgPT0gMCBvciBxMSA9PSBuX2ltYWdlcyBvciAocTAgLy8gcXVlcnlfYmF0Y2gpICUgMTAgPT0gMDoNCiAgICAgICAgICAgICAgICBMT0dHRVIuaW5mbygiRGlzdGFuY2UgcXVlcmllcyAlZC8lZCIsIHExLCBuX2ltYWdlcykNCiAgICAgICAgICAgIHF1ZXJ5ID0gcmVzaWRlbnRbcTA6cTFdLmZsb2F0KCkucmVzaGFwZSgtMSwgY2hhbm5lbHMpDQogICAgICAgICAgICByb3cgPSB0b3JjaC5lbXB0eSgocTEgLSBxMCwgbl9wYXRjaGVzLCBuX3RyYWluKSwgZHR5cGU9dG9yY2guZmxvYXQzMiwgZGV2aWNlPWRldmljZSkNCiAgICAgICAgICAgIGZvciB0MCBpbiByYW5nZSgwLCBuX3RyYWluLCB0cmFpbl9iYXRjaCk6DQogICAgICAgICAgICAgICAgdDEgPSBtaW4odDAgKyB0cmFpbl9iYXRjaCwgbl90cmFpbikNCiAgICAgICAgICAgICAgICB0cmFpbiA9IHJlc2lkZW50W3QwOnQxXS5mbG9hdCgpLnJlc2hhcGUoLTEsIGNoYW5uZWxzKQ0KICAgICAgICAgICAgICAgIGJsb2NrID0gdG9yY2guY2Rpc3QocXVlcnksIHRyYWluKS5yZXNoYXBlKHExIC0gcTAsIG5fcGF0Y2hlcywgdDEgLSB0MCwgbl9wYXRjaGVzKQ0KICAgICAgICAgICAgICAgIHJvd1s6LCA6LCB0MDp0MV0gPSBibG9jay5taW4oZGltPS0xKS52YWx1ZXMNCiAgICAgICAgICAgIGlmIG5vdCBib29sKHRvcmNoLmlzZmluaXRlKHJvdykuYWxsKCkpOg0KICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIk5vbi1maW5pdGUgZGlzdGFuY2VzIGZvciBxdWVyaWVzIHtxMH06e3ExfSIpDQogICAgICAgICAgICBvdXRwdXRbcTA6cTFdID0gcm93LmNwdSgpLm51bXB5KCkNCiAgICBkZWwgcmVzaWRlbnQNCiAgICBpZiBkZXZpY2Uuc3RhcnRzd2l0aCgiY3VkYSIpOg0KICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkNCiAgICByZXR1cm4gb3V0cHV0DQoNCg0KZGVmIGRpc3RhbmNlX3RlbnNvcigNCiAgICBwYXRjaGVzOiBucC5uZGFycmF5LA0KICAgIG5fdHJhaW46IGludCwNCiAgICBxdWVyeV9iYXRjaDogaW50ID0gOCwNCiAgICB0cmFpbl9iYXRjaDogaW50ID0gMTYsDQogICAgZGV2aWNlOiBzdHIgfCBOb25lID0gTm9uZSwNCikgLT4gbnAubmRhcnJheToNCiAgICAiIiJwYXRjaGVzIHJvdyBvcmRlciBpcyB0cmFpbiBpbWFnZXMgZmlyc3QuIERpYWdvbmFsIHRyYWluLXRyYWluIGVudHJpZXMgYXJlICtpbmYuIiIiDQogICAgaWYgcGF0Y2hlcy5uZGltICE9IDM6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkV4cGVjdGVkIChOLCBQLCBDKSBwYXRjaGVzLCBnb3Qge3BhdGNoZXMuc2hhcGV9IikNCiAgICBuX2ltYWdlcywgbl9wYXRjaGVzLCBfY2hhbm5lbHMgPSBwYXRjaGVzLnNoYXBlDQogICAgaWYgbm90IDAgPCBuX3RyYWluIDw9IG5faW1hZ2VzOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJuX3RyYWluPXtuX3RyYWlufSBpcyBvdXRzaWRlIDEuLntuX2ltYWdlc30iKQ0KICAgIHRvcmNoX2RldmljZSA9IE5vbmUNCiAgICBpZiBkZXZpY2UgIT0gImNwdSI6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGltcG9ydCB0b3JjaA0KDQogICAgICAgICAgICB0b3JjaF9kZXZpY2UgPSBkZXZpY2Ugb3IgKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpDQogICAgICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoNCiAgICAgICAgICAgIHRvcmNoX2RldmljZSA9IE5vbmUNCiAgICBpZiB0b3JjaF9kZXZpY2UgaXMgbm90IE5vbmU6DQogICAgICAgIG91dHB1dCA9IF9kaXN0YW5jZV90ZW5zb3JfdG9yY2gocGF0Y2hlcywgbl90cmFpbiwgcXVlcnlfYmF0Y2gsIHRyYWluX2JhdGNoLCB0b3JjaF9kZXZpY2UpDQogICAgZWxzZToNCiAgICAgICAgb3V0cHV0ID0gbnAuZW1wdHkoKG5faW1hZ2VzLCBuX3BhdGNoZXMsIG5fdHJhaW4pLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgICAgICBmb3IgcTAgaW4gcmFuZ2UoMCwgbl9pbWFnZXMsIHF1ZXJ5X2JhdGNoKToNCiAgICAgICAgICAgIHExID0gbWluKHEwICsgcXVlcnlfYmF0Y2gsIG5faW1hZ2VzKQ0KICAgICAgICAgICAgZm9yIHQwIGluIHJhbmdlKDAsIG5fdHJhaW4sIHRyYWluX2JhdGNoKToNCiAgICAgICAgICAgICAgICB0MSA9IG1pbih0MCArIHRyYWluX2JhdGNoLCBuX3RyYWluKQ0KICAgICAgICAgICAgICAgIGJsb2NrID0gbWluX3BhdGNoX2Rpc3RhbmNlX2Jsb2NrKHBhdGNoZXNbcTA6cTFdLCBwYXRjaGVzW3QwOnQxXSkNCiAgICAgICAgICAgICAgICBpZiBub3QgbnAuaXNmaW5pdGUoYmxvY2spLmFsbCgpOg0KICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJOb24tZmluaXRlIGRpc3RhbmNlcyBmb3IgcXVlcmllcyB7cTB9OntxMX0sIHRyYWluIHt0MH06e3QxfSIpDQogICAgICAgICAgICAgICAgb3V0cHV0W3EwOnExLCA6LCB0MDp0MV0gPSBibG9jaw0KICAgIGZvciBpbmRleCBpbiByYW5nZShuX3RyYWluKToNCiAgICAgICAgb3V0cHV0W2luZGV4LCA6LCBpbmRleF0gPSBucC5pbmYNCiAgICByZXR1cm4gb3V0cHV0DQoNCg0KZGVmIGNvbXB1dGVfY2xhc3NfZGlzdGFuY2VzKGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIGRldmljZTogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IE5vbmU6DQogICAgb3V0X3BhdGggPSBkaXN0YW5jZV9jYWNoZV9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBtZXRhX3BhdGggPSBkaXN0YW5jZV9tZXRhX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIGlmIG91dF9wYXRoLmlzX2ZpbGUoKSBhbmQgbWV0YV9wYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgbWV0YSA9IHJlYWRfanNvbihtZXRhX3BhdGgpDQogICAgICAgIGlmIG1ldGEuZ2V0KCJjb25maWdfaGFzaCIpID09IGNmZy5oYXNoOg0KICAgICAgICAgICAgTE9HR0VSLmluZm8oIkRpc3RhbmNlIGNhY2hlIGV4aXN0cyBmb3IgJXMiLCBjbGFzc19uYW1lKQ0KICAgICAgICAgICAgcmV0dXJuDQogICAgY2FjaGUgPSBsb2FkX2ZlYXR1cmVfY2FjaGUoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIG5fdHJhaW4gPSBpbnQoY2FjaGVbIm1ldGEiXVsibl90cmFpbiJdKQ0KICAgIExPR0dFUi5pbmZvKCJDb21wdXRpbmcgZGlzdGFuY2UgdGVuc29yIGZvciAlcyAoJWQgdHJhaW4gaW1hZ2VzKSIsIGNsYXNzX25hbWUsIG5fdHJhaW4pDQogICAgdGVuc29yID0gZGlzdGFuY2VfdGVuc29yKA0KICAgICAgICBjYWNoZVsicGF0Y2hlcyJdLA0KICAgICAgICBuX3RyYWluPW5fdHJhaW4sDQogICAgICAgIHF1ZXJ5X2JhdGNoPWNmZy5kaXN0YW5jZV9xdWVyeV9iYXRjaCwNCiAgICAgICAgdHJhaW5fYmF0Y2g9Y2ZnLmRpc3RhbmNlX3RyYWluX2JhdGNoLA0KICAgICAgICBkZXZpY2U9ZGV2aWNlLA0KICAgICkNCiAgICBhdG9taWNfbnB5KG91dF9wYXRoLCB0ZW5zb3IpDQogICAgd3JpdGVfanNvbigNCiAgICAgICAgbWV0YV9wYXRoLA0KICAgICAgICB7DQogICAgICAgICAgICAiY29uZmlnX2hhc2giOiBjZmcuaGFzaCwNCiAgICAgICAgICAgICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwNCiAgICAgICAgICAgICJzaGFwZSI6IGxpc3QodGVuc29yLnNoYXBlKSwNCiAgICAgICAgICAgICJuX3RyYWluIjogbl90cmFpbiwNCiAgICAgICAgICAgICJuX3Rlc3RfZ29vZCI6IGludChjYWNoZVsibWV0YSJdWyJuX3Rlc3RfZ29vZCJdKSwNCiAgICAgICAgICAgICJuX3Rlc3RfZGVmZWN0IjogaW50KGNhY2hlWyJtZXRhIl1bIm5fdGVzdF9kZWZlY3QiXSksDQogICAgICAgICAgICAicGF0aHMiOiBjYWNoZVsicGF0aHMiXSwNCiAgICAgICAgICAgICJkdHlwZSI6ICJmbG9hdDMyIiwNCiAgICAgICAgICAgICJieXRlcyI6IGludCh0ZW5zb3IubmJ5dGVzKSwNCiAgICAgICAgfSwNCiAgICApDQogICAgTE9HR0VSLmluZm8oIiVzIGRpc3RhbmNlczogc2hhcGUgJXMsICUuMmYgR0IiLCBjbGFzc19uYW1lLCB0ZW5zb3Iuc2hhcGUsIHRlbnNvci5uYnl0ZXMgLyAoMTAyNCoqMykpDQoNCg0KZGVmIGxvYWRfZGlzdGFuY2VfY2FjaGUoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gZGljdDoNCiAgICBwYXRoID0gZGlzdGFuY2VfY2FjaGVfcGF0aChjZmcsIGNsYXNzX25hbWUpDQogICAgbWV0YV9wYXRoID0gZGlzdGFuY2VfbWV0YV9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBub3QgcGF0aC5pc19maWxlKCkgb3Igbm90IG1ldGFfcGF0aC5pc19maWxlKCk6DQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTWlzc2luZyBkaXN0YW5jZSBjYWNoZSBmb3Ige2NsYXNzX25hbWV9LiBSdW4gZGlzdGFuY2VzIGZpcnN0LiIpDQogICAgbWV0YSA9IHJlYWRfanNvbihtZXRhX3BhdGgpDQogICAgaWYgbWV0YS5nZXQoImNvbmZpZ19oYXNoIikgIT0gY2ZnLmhhc2g6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkRpc3RhbmNlIGNhY2hlIGhhc2ggbWlzbWF0Y2ggZm9yIHtjbGFzc19uYW1lfS4iKQ0KICAgIHRlbnNvciA9IG5wLmxvYWQocGF0aCkNCiAgICBpZiB0dXBsZSh0ZW5zb3Iuc2hhcGUpICE9IHR1cGxlKG1ldGFbInNoYXBlIl0pOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGRpc3RhbmNlIHRlbnNvciBzaGFwZSB7dGVuc29yLnNoYXBlfSAhPSBtZXRhIHttZXRhWydzaGFwZSddfSIpDQogICAgcmV0dXJuIHsiRCI6IHRlbnNvciwgIm1ldGEiOiBtZXRhfQ0KDQoNCmRlZiBjb21wdXRlX2Rpc3RhbmNlcyhjZmc6IENvbmZpZywgY2xhc3NlczogbGlzdFtzdHJdLCBkZXZpY2U6IHN0ciB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIGZvciBjbGFzc19uYW1lIGluIGNsYXNzZXM6DQogICAgICAgIGNvbXB1dGVfY2xhc3NfZGlzdGFuY2VzKGNmZywgY2xhc3NfbmFtZSwgZGV2aWNlPWRldmljZSkNCg==", "commissioning/e0.py": "IiIiRTBhIHJlZmVyZW5jZSBzYW5pdHkgY2hlY2sgYW5kIEUwYiBvd24tY2FjaGUgdmVyc3VzIEFub21hbGliLg0KDQpSUSB0cmFqZWN0b3JpZXMgcmVmdXNlIHRvIHN0YXJ0IHVubGVzcyBlMC9nYXRlLmpzb24gc2F5cyBQQVNTIGZvciB0aGlzIGNvbmZpZyBoYXNoLg0KIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGluc3BlY3QNCmltcG9ydCBsb2dnaW5nDQoNCmltcG9ydCBudW1weSBhcyBucA0KZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgc3BlYXJtYW5yDQpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgcm9jX2F1Y19zY29yZQ0KDQpmcm9tIGNvbW1pc3Npb25pbmcuY29uZmlnIGltcG9ydCBDb25maWcNCmZyb20gY29tbWlzc2lvbmluZy5kYXRhIGltcG9ydCBhYnNvbHV0ZV9wYXRoLCBsb2FkX3NwbGl0LCBzcGxpdF9zZWVkcw0KZnJvbSBjb21taXNzaW9uaW5nLmRpc3RhbmNlcyBpbXBvcnQgbG9hZF9kaXN0YW5jZV9jYWNoZQ0KZnJvbSBjb21taXNzaW9uaW5nLmZlYXR1cmVzIGltcG9ydCBidWlsZF90cmFuc2Zvcm0sIGxvYWRfZmVhdHVyZV9jYWNoZSwgcmVhZF9yZ2INCmZyb20gY29tbWlzc2lvbmluZy5pbnRlZ3JpdHkgaW1wb3J0IGFzc2VydF9tZXRob2RzX2FncmVlLCBjaGVja19jb3JydXB0ZWRfaW5wdXQsIGNoZWNrX2ZlYXR1cmVfYWxpZ25tZW50DQpmcm9tIGNvbW1pc3Npb25pbmcuaW9fdXRpbHMgaW1wb3J0IHJlYWRfanNvbiwgd3JpdGVfanNvbg0KZnJvbSBjb21taXNzaW9uaW5nLnBhdGNoY29yZSBpbXBvcnQgc2NvcmVzX2Zvcl9iYW5rDQpmcm9tIGNvbW1pc3Npb25pbmcuc2VsZWN0aW9uIGltcG9ydCByYW5kb21fb3JkZXINCg0KTE9HR0VSID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pDQoNCg0KZGVmIGdhdGVfcGF0aChjZmc6IENvbmZpZyk6DQogICAgcmV0dXJuIGNmZy53b3JrX3Jvb3QgLyAiZTAiIC8gImdhdGUuanNvbiINCg0KDQpkZWYgX2NsYXNzX3BhdGgoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cik6DQogICAgcmV0dXJuIGNmZy53b3JrX3Jvb3QgLyAiZTAiIC8gZiJ7Y2xhc3NfbmFtZX0uanNvbiINCg0KDQpkZWYgX2tjZW50ZXJfZ3JlZWR5X2NsYXNzKCk6DQogICAgdHJ5Og0KICAgICAgICBmcm9tIGFub21hbGliLm1vZGVscy5jb21wb25lbnRzIGltcG9ydCBLQ2VudGVyR3JlZWR5DQogICAgZXhjZXB0IEltcG9ydEVycm9yOg0KICAgICAgICBmcm9tIGFub21hbGliLm1vZGVscy5jb21wb25lbnRzLnNhbXBsaW5nIGltcG9ydCBLQ2VudGVyR3JlZWR5DQogICAgcmV0dXJuIEtDZW50ZXJHcmVlZHkNCg0KDQpkZWYgY2hlY2tfYW5vbWFsaWJfdXBzYW1wbGluZyhjZmc6IENvbmZpZykgLT4gc3RyOg0KICAgICIiIlRoZSBjYWNoZSBhbmQgQW5vbWFsaWIgbXVzdCByZXNpemUgbGF5ZXIzIHRoZSBzYW1lIHdheSwgb3RoZXJ3aXNlIEUwYiBjb21wYXJlcyB0d28gcGlwZWxpbmVzLiIiIg0KICAgIGZyb20gYW5vbWFsaWIubW9kZWxzLmltYWdlLnBhdGNoY29yZS50b3JjaF9tb2RlbCBpbXBvcnQgUGF0Y2hjb3JlTW9kZWwNCg0KICAgIHNvdXJjZSA9IGluc3BlY3QuZ2V0c291cmNlKFBhdGNoY29yZU1vZGVsLmdlbmVyYXRlX2VtYmVkZGluZykNCiAgICBtb2RlcyA9IFttb2RlIGZvciBtb2RlIGluICgiYmlsaW5lYXIiLCAibmVhcmVzdCIsICJiaWN1YmljIikgaWYgZidtb2RlPSJ7bW9kZX0iJyBpbiBzb3VyY2Ugb3IgZiJtb2RlPSd7bW9kZX0nIiBpbiBzb3VyY2VdDQogICAgaWYgbGVuKG1vZGVzKSAhPSAxOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJDYW5ub3QgcmVhZCB0aGUgaW50ZXJwb2xhdGlvbiBtb2RlIG9mIHRoZSBpbnN0YWxsZWQgQW5vbWFsaWIgUGF0Y2hDb3JlOiB7bW9kZXN9IikNCiAgICBpZiBtb2Rlc1swXSAhPSBjZmcudXBzYW1wbGVfbW9kZToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKA0KICAgICAgICAgICAgZiJJbnN0YWxsZWQgQW5vbWFsaWIgdXBzYW1wbGVzIGxheWVyMyB3aXRoIHttb2Rlc1swXSFyfSwgY29uZmlnLnlhbWwgdXNlcyB7Y2ZnLnVwc2FtcGxlX21vZGUhcn0uICINCiAgICAgICAgICAgICJTZXQgdXBzYW1wbGVfbW9kZSBpbiBjb25maWcueWFtbCB0byBtYXRjaCBiZWZvcmUgY29tcHV0aW5nIGZlYXR1cmVzLiINCiAgICAgICAgKQ0KICAgIHJldHVybiBtb2Rlc1swXQ0KDQoNCmRlZiByZWNvcmRfc2FtcGxlcl9zb3VyY2UoY2ZnOiBDb25maWcpIC0+IHN0cjoNCiAgICBpbXBvcnQgYW5vbWFsaWINCg0KICAgIGNodW5rcyA9IFtmImFub21hbGliIHtnZXRhdHRyKGFub21hbGliLCAnX192ZXJzaW9uX18nLCAndW5rbm93bicpfSJdDQogICAgdHJ5Og0KICAgICAgICBmcm9tIGFub21hbGliLm1vZGVscy5pbWFnZS5wYXRjaGNvcmUudG9yY2hfbW9kZWwgaW1wb3J0IFBhdGNoY29yZU1vZGVsDQoNCiAgICAgICAgY2h1bmtzLmFwcGVuZChpbnNwZWN0LmdldHNvdXJjZShQYXRjaGNvcmVNb2RlbC5zdWJzYW1wbGVfZW1iZWRkaW5nKSkNCiAgICAgICAgY2h1bmtzLmFwcGVuZChpbnNwZWN0LmdldHNvdXJjZShQYXRjaGNvcmVNb2RlbC5nZW5lcmF0ZV9lbWJlZGRpbmcpKQ0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOg0KICAgICAgICBjaHVua3MuYXBwZW5kKGYiQ291bGQgbm90IHJlYWQgUGF0Y2hjb3JlTW9kZWwgc291cmNlOiB7ZXhjfSIpDQogICAgdHJ5Og0KICAgICAgICBmcm9tIGFub21hbGliLm1vZGVscy5jb21wb25lbnRzLnNhbXBsaW5nIGltcG9ydCBLQ2VudGVyR3JlZWR5DQogICAgZXhjZXB0IEltcG9ydEVycm9yOg0KICAgICAgICB0cnk6DQogICAgICAgICAgICBmcm9tIGFub21hbGliLm1vZGVscy5jb21wb25lbnRzIGltcG9ydCBLQ2VudGVyR3JlZWR5DQogICAgICAgIGV4Y2VwdCBJbXBvcnRFcnJvciBhcyBleGM6DQogICAgICAgICAgICBLQ2VudGVyR3JlZWR5ID0gTm9uZQ0KICAgICAgICAgICAgY2h1bmtzLmFwcGVuZChmIkNvdWxkIG5vdCBpbXBvcnQgS0NlbnRlckdyZWVkeToge2V4Y30iKQ0KICAgIGlmIEtDZW50ZXJHcmVlZHkgaXMgbm90IE5vbmU6DQogICAgICAgIGNodW5rcy5hcHBlbmQoaW5zcGVjdC5nZXRzb3VyY2UoS0NlbnRlckdyZWVkeS5zYW1wbGVfY29yZXNldCkpDQogICAgcGF0aCA9IGNmZy53b3JrX3Jvb3QgLyAiZTAiIC8gInNhbXBsZXJfc291cmNlLnR4dCINCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgcGF0aC53cml0ZV90ZXh0KCJcblxuIi5qb2luKGNodW5rcyksIGVuY29kaW5nPSJ1dGYtOCIpDQogICAgcmV0dXJuIHN0cihnZXRhdHRyKGFub21hbGliLCAiX192ZXJzaW9uX18iLCAidW5rbm93biIpKQ0KDQoNCmRlZiBfYmF0Y2hlcyhjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyLCByZWxhdGl2ZV9wYXRoczogbGlzdFtzdHJdLCBiYXRjaF9zaXplOiBpbnQpOg0KICAgIGltcG9ydCB0b3JjaA0KDQogICAgdHJhbnNmb3JtID0gYnVpbGRfdHJhbnNmb3JtKGNmZykNCiAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCwgbGVuKHJlbGF0aXZlX3BhdGhzKSwgYmF0Y2hfc2l6ZSk6DQogICAgICAgIGNodW5rID0gcmVsYXRpdmVfcGF0aHNbc3RhcnQgOiBzdGFydCArIGJhdGNoX3NpemVdDQogICAgICAgIGltYWdlcyA9IFt0cmFuc2Zvcm0ocmVhZF9yZ2IoYWJzb2x1dGVfcGF0aChjZmcuZGF0YV9yb290LCBwYXRoKSwgY2xhc3NfbmFtZSkpIGZvciBwYXRoIGluIGNodW5rXQ0KICAgICAgICB5aWVsZCB0b3JjaC5zdGFjayhpbWFnZXMsIGRpbT0wKQ0KDQoNCmRlZiBfcHJlZGljdGlvbl9zY29yZXMob3V0cHV0KSAtPiAibnAubmRhcnJheSI6DQogICAgaW1wb3J0IHRvcmNoDQoNCiAgICBpZiBoYXNhdHRyKG91dHB1dCwgInByZWRfc2NvcmUiKToNCiAgICAgICAgc2NvcmVzID0gb3V0cHV0LnByZWRfc2NvcmUNCiAgICBlbGlmIGlzaW5zdGFuY2Uob3V0cHV0LCB0dXBsZSkgYW5kIGxlbihvdXRwdXQpID49IDI6DQogICAgICAgIHNjb3JlcyA9IG91dHB1dFsxXQ0KICAgIGVsc2U6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkFub21hbGliIG91dHB1dCBoYXMgbm8gaW1hZ2Ugc2NvcmU6IHt0eXBlKG91dHB1dCl9IikNCiAgICBhcnJheSA9IHNjb3Jlcy5kZXRhY2goKS5jcHUoKS5udW1weSgpLnJlc2hhcGUoLTEpDQogICAgaWYgbm90IG5wLmlzZmluaXRlKGFycmF5KS5hbGwoKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJBbm9tYWxpYiByZXR1cm5lZCBhIG5vbi1maW5pdGUgaW1hZ2Ugc2NvcmUuIikNCiAgICByZXR1cm4gYXJyYXkNCg0KDQpkZWYgX2ZpdF9hbm9tYWxpYihjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyLCB0cmFpbl9wYXRoczogbGlzdFtzdHJdLCBudW1fbmVpZ2hib3JzOiBpbnQsIGZ1bGxfYmFuazogYm9vbCk6DQogICAgIiIiVHJhaW5pbmctbW9kZSBmb3J3YXJkIHJldHVybnMgdGhlIHBhdGNoIGVtYmVkZGluZyBpbiBldmVyeSBBbm9tYWxpYiAyLnggcmVsZWFzZS4NCg0KICAgIEVtYmVkZGluZ3MgYXJlIGNvbGxlY3RlZCBoZXJlIGluc3RlYWQgb2YgdGhyb3VnaCBgYGVtYmVkZGluZ19zdG9yZWBgIG9yIGBgc3Vic2FtcGxlX2VtYmVkZGluZ2BgLA0KICAgIHdob3NlIHNpZ25hdHVyZXMgY2hhbmdlZCBiZXR3ZWVuIHJlbGVhc2VzLg0KICAgICIiIg0KICAgIGltcG9ydCB0b3JjaA0KICAgIGZyb20gYW5vbWFsaWIubW9kZWxzLmltYWdlLnBhdGNoY29yZS50b3JjaF9tb2RlbCBpbXBvcnQgUGF0Y2hjb3JlTW9kZWwNCg0KICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQ0KICAgIG1vZGVsID0gUGF0Y2hjb3JlTW9kZWwoDQogICAgICAgIGxheWVycz1saXN0KGNmZy5sYXllcnMpLA0KICAgICAgICBiYWNrYm9uZT1jZmcuYmFja2JvbmUsDQogICAgICAgIHByZV90cmFpbmVkPVRydWUsDQogICAgICAgIG51bV9uZWlnaGJvcnM9bnVtX25laWdoYm9ycywNCiAgICApLnRvKGRldmljZSkNCiAgICBtb2RlbC50cmFpbigpDQogICAgY29sbGVjdGVkID0gW10NCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgZm9yIGJhdGNoIGluIF9iYXRjaGVzKGNmZywgY2xhc3NfbmFtZSwgdHJhaW5fcGF0aHMsIGJhdGNoX3NpemU9NCk6DQogICAgICAgICAgICBlbWJlZGRpbmcgPSBtb2RlbChiYXRjaC50byhkZXZpY2UpKQ0KICAgICAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UoZW1iZWRkaW5nLCB0b3JjaC5UZW5zb3IpIG9yIGVtYmVkZGluZy5uZGltICE9IDI6DQogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiB1bmV4cGVjdGVkIEFub21hbGliIHRyYWluaW5nIG91dHB1dCB7dHlwZShlbWJlZGRpbmcpfSIpDQogICAgICAgICAgICBjb2xsZWN0ZWQuYXBwZW5kKGVtYmVkZGluZykNCiAgICAgICAgICAgIHN0b3JlID0gZ2V0YXR0cihtb2RlbCwgImVtYmVkZGluZ19zdG9yZSIsIE5vbmUpDQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHN0b3JlLCBsaXN0KToNCiAgICAgICAgICAgICAgICBzdG9yZS5jbGVhcigpDQogICAgbW9kZWwuZXZhbCgpDQogICAgc3RvcmVkID0gdG9yY2gudnN0YWNrKGNvbGxlY3RlZCkNCiAgICBkZWwgY29sbGVjdGVkDQogICAgZXhwZWN0ZWRfcm93cyA9IGxlbih0cmFpbl9wYXRocykgKiA3ODQNCiAgICBpZiBpbnQoc3RvcmVkLnNoYXBlWzBdKSAhPSBleHBlY3RlZF9yb3dzIG9yIGludChzdG9yZWQuc2hhcGVbMV0pICE9IDE1MzY6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogQW5vbWFsaWIgZW1iZWRkaW5ncyB7dHVwbGUoc3RvcmVkLnNoYXBlKX0sIGV4cGVjdGVkICh7ZXhwZWN0ZWRfcm93c30sIDE1MzYpIikNCiAgICBpZiBmdWxsX2Jhbms6DQogICAgICAgICMgc2FtcGxpbmdfcmF0aW89MSB3b3VsZCBzdGlsbCBydW4gS0NlbnRlckdyZWVkeSBvdmVyIGV2ZXJ5IHBhdGNoLiBUaGUgc2FtcGxlciBpcyBza2lwcGVkLg0KICAgICAgICBtb2RlbC5tZW1vcnlfYmFuayA9IHN0b3JlZA0KICAgIGVsc2U6DQogICAgICAgIHNhbXBsZXIgPSBfa2NlbnRlcl9ncmVlZHlfY2xhc3MoKShlbWJlZGRpbmc9c3RvcmVkLCBzYW1wbGluZ19yYXRpbz1mbG9hdChjZmcuZTBbImNvcmVzZXRfcmF0aW8iXSkpDQogICAgICAgIG1vZGVsLm1lbW9yeV9iYW5rID0gc2FtcGxlci5zYW1wbGVfY29yZXNldCgpDQogICAgICAgIGRlbCBzdG9yZWQNCiAgICBpZiBpbnQobW9kZWwubWVtb3J5X2Jhbmsuc2hhcGVbMF0pID09IDA6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogbWVtb3J5IGJhbmsgaXMgZW1wdHkuIikNCiAgICByZXR1cm4gbW9kZWwsIGRldmljZQ0KDQoNCmRlZiBfcmVsZWFzZShtb2RlbCkgLT4gTm9uZToNCiAgICBpbXBvcnQgZ2MNCg0KICAgIGltcG9ydCB0b3JjaA0KDQogICAgZGVsIG1vZGVsDQogICAgZ2MuY29sbGVjdCgpDQogICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToNCiAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpDQoNCg0KZGVmIF9zY29yZV90ZXN0KGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIG1vZGVsLCBkZXZpY2UpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOg0KICAgIGltcG9ydCB0b3JjaA0KDQogICAgc3BsaXQgPSBsb2FkX3NwbGl0KGNmZywgY2xhc3NfbmFtZSwgMCkNCiAgICB0ZXN0X3BhdGhzID0gbGlzdChzcGxpdFsidGVzdF9nb29kIl0pICsgW2l0ZW1bInBhdGgiXSBmb3IgaXRlbSBpbiBzcGxpdFsidGVzdF9kZWZlY3QiXV0NCiAgICBsYWJlbHMgPSBucC5hcnJheShbMF0gKiBsZW4oc3BsaXRbInRlc3RfZ29vZCJdKSArIFsxXSAqIGxlbihzcGxpdFsidGVzdF9kZWZlY3QiXSksIGR0eXBlPWludCkNCiAgICBzY29yZXMgPSBbXQ0KICAgIG1vZGVsLmV2YWwoKQ0KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICBmb3IgYmF0Y2ggaW4gX2JhdGNoZXMoY2ZnLCBjbGFzc19uYW1lLCB0ZXN0X3BhdGhzLCBiYXRjaF9zaXplPTIpOg0KICAgICAgICAgICAgb3V0cHV0ID0gbW9kZWwoYmF0Y2gudG8oZGV2aWNlKSkNCiAgICAgICAgICAgIHNjb3Jlcy5hcHBlbmQoX3ByZWRpY3Rpb25fc2NvcmVzKG91dHB1dCkpDQogICAgc3RhY2tlZCA9IG5wLmNvbmNhdGVuYXRlKHNjb3JlcykNCiAgICBpZiBzdGFja2VkLnNoYXBlICE9IGxhYmVscy5zaGFwZToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBzY29yZWQge3N0YWNrZWQuc2hhcGVbMF19IHRlc3QgaW1hZ2VzLCBleHBlY3RlZCB7bGFiZWxzLnNoYXBlWzBdfS4iKQ0KICAgIHJldHVybiBzdGFja2VkLCBsYWJlbHMNCg0KDQpkZWYgb3duX2Z1bGxfYmFua19zY29yZXMoc3BsaXQ6IGRpY3QsIGRpc3RhbmNlOiBucC5uZGFycmF5KSAtPiB0dXBsZVtucC5uZGFycmF5LCBucC5uZGFycmF5XToNCiAgICBuX3RyYWluID0gaW50KHNwbGl0WyJuX3RyYWluIl0pDQogICAgbl90ZXN0ID0gaW50KHNwbGl0WyJuX3Rlc3RfZ29vZCJdKSArIGludChzcGxpdFsibl90ZXN0X2RlZmVjdCJdKQ0KICAgIHF1ZXJpZXMgPSBucC5hcmFuZ2Uobl90cmFpbiwgbl90cmFpbiArIG5fdGVzdCkNCiAgICBiYW5rID0gbnAuYXJhbmdlKG5fdHJhaW4pDQogICAgc2NvcmVzID0gc2NvcmVzX2Zvcl9iYW5rKGRpc3RhbmNlLCBxdWVyaWVzLCBiYW5rKQ0KICAgIGxhYmVscyA9IG5wLmFycmF5KFswXSAqIGludChzcGxpdFsibl90ZXN0X2dvb2QiXSkgKyBbMV0gKiBpbnQoc3BsaXRbIm5fdGVzdF9kZWZlY3QiXSksIGR0eXBlPWludCkNCiAgICByZXR1cm4gc2NvcmVzLCBsYWJlbHMNCg0KDQpkZWYgX2F1cm9jKGxhYmVsczogbnAubmRhcnJheSwgc2NvcmVzOiBucC5uZGFycmF5KSAtPiBmbG9hdDoNCiAgICByZXR1cm4gZmxvYXQocm9jX2F1Y19zY29yZShsYWJlbHMsIHNjb3JlcykpDQoNCg0KZGVmIF9taW5tYXgodmFsdWVzOiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5Og0KICAgIHNwYW4gPSBmbG9hdCh2YWx1ZXMubWF4KCkgLSB2YWx1ZXMubWluKCkpDQogICAgaWYgc3BhbiA8PSAwLjA6DQogICAgICAgIHJldHVybiBucC56ZXJvc19saWtlKHZhbHVlcywgZHR5cGU9ZmxvYXQpDQogICAgcmV0dXJuICh2YWx1ZXMgLSB2YWx1ZXMubWluKCkpIC8gc3Bhbg0KDQoNCmRlZiBydW5fY2xhc3MoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0ciwgcGFydDogc3RyLCB0cmFpbl9wYXRoczogbGlzdFtzdHJdLCBkaXN0YW5jZTogbnAubmRhcnJheSkgLT4gZGljdDoNCiAgICBwYXRoID0gX2NsYXNzX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIGV4aXN0aW5nID0gcmVhZF9qc29uKHBhdGgpIGlmIHBhdGguaXNfZmlsZSgpIGVsc2Uge30NCiAgICBpZiBleGlzdGluZy5nZXQoImNvbmZpZ19oYXNoIikgIT0gY2ZnLmhhc2g6DQogICAgICAgIGV4aXN0aW5nID0ge30NCiAgICBzcGxpdCA9IGxvYWRfc3BsaXQoY2ZnLCBjbGFzc19uYW1lLCAwKQ0KICAgIGlmIHBhcnQgaW4geyJhIiwgImFsbCJ9IGFuZCAiZTBhIiBub3QgaW4gZXhpc3Rpbmc6DQogICAgICAgIExPR0dFUi5pbmZvKCJFMGEgJXMiLCBjbGFzc19uYW1lKQ0KICAgICAgICBtb2RlbCwgZGV2aWNlID0gX2ZpdF9hbm9tYWxpYihjZmcsIGNsYXNzX25hbWUsIHRyYWluX3BhdGhzLCBpbnQoY2ZnLmUwWyJudW1fbmVpZ2hib3JzX3JlZiJdKSwgZnVsbF9iYW5rPUZhbHNlKQ0KICAgICAgICBzY29yZXMsIGxhYmVscyA9IF9zY29yZV90ZXN0KGNmZywgY2xhc3NfbmFtZSwgbW9kZWwsIGRldmljZSkNCiAgICAgICAgZXhpc3RpbmdbImUwYSJdID0geyJhdXJvYyI6IF9hdXJvYyhsYWJlbHMsIHNjb3JlcyksICJuX3Rlc3QiOiBpbnQobGFiZWxzLnNpemUpfQ0KICAgICAgICBfcmVsZWFzZShtb2RlbCkNCiAgICAgICAgd3JpdGVfanNvbihwYXRoLCB7KipleGlzdGluZywgImNvbmZpZ19oYXNoIjogY2ZnLmhhc2gsICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZX0pDQogICAgaWYgcGFydCBpbiB7ImIiLCAiYWxsIn0gYW5kICJlMGIiIG5vdCBpbiBleGlzdGluZzoNCiAgICAgICAgTE9HR0VSLmluZm8oIkUwYiAlcyIsIGNsYXNzX25hbWUpDQogICAgICAgIG1vZGVsLCBkZXZpY2UgPSBfZml0X2Fub21hbGliKGNmZywgY2xhc3NfbmFtZSwgdHJhaW5fcGF0aHMsIGludChjZmcuZTBbIm51bV9uZWlnaGJvcnNfcnEiXSksIGZ1bGxfYmFuaz1UcnVlKQ0KICAgICAgICBhbm9tYWxpYl9zY29yZXMsIGxhYmVscyA9IF9zY29yZV90ZXN0KGNmZywgY2xhc3NfbmFtZSwgbW9kZWwsIGRldmljZSkNCiAgICAgICAgX3JlbGVhc2UobW9kZWwpDQogICAgICAgIG93bl9zY29yZXMsIG93bl9sYWJlbHMgPSBvd25fZnVsbF9iYW5rX3Njb3JlcyhzcGxpdCwgZGlzdGFuY2UpDQogICAgICAgIGlmIG5vdCBucC5hcnJheV9lcXVhbChsYWJlbHMsIG93bl9sYWJlbHMpOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBFMGIgbGFiZWwgb3JkZXIgZGlmZmVycyBiZXR3ZWVuIHRoZSB0d28gaW1wbGVtZW50YXRpb25zLiIpDQogICAgICAgIHJobyA9IGZsb2F0KHNwZWFybWFucihvd25fc2NvcmVzLCBhbm9tYWxpYl9zY29yZXMpLnN0YXRpc3RpYykNCiAgICAgICAgZXhpc3RpbmdbImUwYiJdID0gew0KICAgICAgICAgICAgInNwZWFybWFuIjogcmhvLA0KICAgICAgICAgICAgImF1cm9jX293biI6IF9hdXJvYyhsYWJlbHMsIG93bl9zY29yZXMpLA0KICAgICAgICAgICAgImF1cm9jX2Fub21hbGliIjogX2F1cm9jKGxhYmVscywgYW5vbWFsaWJfc2NvcmVzKSwNCiAgICAgICAgICAgICJtYXhfYWJzX3Njb3JlX2dhcCI6IGZsb2F0KG5wLm1heChucC5hYnMob3duX3Njb3JlcyAtIGFub21hbGliX3Njb3JlcykpKSwNCiAgICAgICAgICAgICJtYXhfYWJzX2dhcF9hZnRlcl9jbGFzc19taW5tYXgiOiBmbG9hdChucC5tYXgobnAuYWJzKF9taW5tYXgob3duX3Njb3JlcykgLSBfbWlubWF4KGFub21hbGliX3Njb3JlcykpKSksDQogICAgICAgIH0NCiAgICBleGlzdGluZ1siY29uZmlnX2hhc2giXSA9IGNmZy5oYXNoDQogICAgZXhpc3RpbmdbImNsYXNzX25hbWUiXSA9IGNsYXNzX25hbWUNCiAgICB3cml0ZV9qc29uKHBhdGgsIGV4aXN0aW5nKQ0KICAgIHJldHVybiBleGlzdGluZw0KDQoNCmRlZiBkaXN0YW5jZV9hZ3JlZW1lbnQoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0ciwgZmVhdHVyZXM6IGRpY3QsIGRpc3RhbmNlOiBucC5uZGFycmF5LCBwYXRoczogbGlzdFtzdHJdKSAtPiBkaWN0W3N0ciwgZmxvYXRdOg0KICAgIHNwbGl0ID0gbG9hZF9zcGxpdChjZmcsIGNsYXNzX25hbWUsIDApDQogICAgaW5kZXggPSB7cGF0aDogcG9zaXRpb24gZm9yIHBvc2l0aW9uLCBwYXRoIGluIGVudW1lcmF0ZShwYXRocyl9DQogICAgcG9vbCA9IFtpbmRleFtwYXRoXSBmb3IgcGF0aCBpbiBzcGxpdFsicG9vbCJdXQ0KICAgIF9zcGxpdF9ybmcsIHJhbmRvbV9ybmcsIF9rY2VudGVyX3JuZyA9IHNwbGl0X3NlZWRzKGNmZywgY2xhc3NfbmFtZSwgMCkNCiAgICBvcmRlciA9IHJhbmRvbV9vcmRlcihwb29sLCByYW5kb21fcm5nKQ0KICAgIHF1ZXJpZXMgPSBbaW5kZXhbcGF0aF0gZm9yIHBhdGggaW4gbGlzdChzcGxpdFsiY2FsaWJyYXRpb24iXSlbOjJdICsgbGlzdChzcGxpdFsidGVzdF9nb29kIl0pWzoyXV0NCiAgICBnYXBzID0ge30NCiAgICBmb3IgbiBpbiAoMSwgOCwgbGVuKG9yZGVyKSk6DQogICAgICAgIGlmIG4gPiBsZW4ob3JkZXIpOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgZ2Fwc1tzdHIobildID0gYXNzZXJ0X21ldGhvZHNfYWdyZWUoZmVhdHVyZXNbInBhdGNoZXMiXSwgZGlzdGFuY2UsIHF1ZXJpZXMsIG9yZGVyWzpuXSkNCiAgICByZXR1cm4gZ2Fwcw0KDQoNCmRlZiBfcGFzc2VkX2UwYihjZmc6IENvbmZpZywgcm93czogbGlzdFtkaWN0XSkgLT4gdHVwbGVbYm9vbCwgZGljdF06DQogICAgcmhvcyA9IFtmbG9hdChyb3dbImUwYiJdWyJzcGVhcm1hbiJdKSBmb3Igcm93IGluIHJvd3NdDQogICAgZGVsdGFzID0gW2FicyhmbG9hdChyb3dbImUwYiJdWyJhdXJvY19vd24iXSkgLSBmbG9hdChyb3dbImUwYiJdWyJhdXJvY19hbm9tYWxpYiJdKSkgZm9yIHJvdyBpbiByb3dzXQ0KICAgIG93bl9tZWFuID0gZmxvYXQobnAubWVhbihbZmxvYXQocm93WyJlMGIiXVsiYXVyb2Nfb3duIl0pIGZvciByb3cgaW4gcm93c10pKQ0KICAgIGFub21hbGliX21lYW4gPSBmbG9hdChucC5tZWFuKFtmbG9hdChyb3dbImUwYiJdWyJhdXJvY19hbm9tYWxpYiJdKSBmb3Igcm93IGluIHJvd3NdKSkNCiAgICBtZWFuX2RlbHRhID0gYWJzKG93bl9tZWFuIC0gYW5vbWFsaWJfbWVhbikNCiAgICBzdW1tYXJ5ID0gew0KICAgICAgICAic3BlYXJtYW5fbWluIjogZmxvYXQobnAubWluKHJob3MpKSwNCiAgICAgICAgInNwZWFybWFuX21lZGlhbiI6IGZsb2F0KG5wLm1lZGlhbihyaG9zKSksDQogICAgICAgICJtZWFuX2F1cm9jX293biI6IG93bl9tZWFuLA0KICAgICAgICAibWVhbl9hdXJvY19hbm9tYWxpYiI6IGFub21hbGliX21lYW4sDQogICAgICAgICJhYnNfZGVsdGFfb2ZfbWVhbl9hdXJvYyI6IG1lYW5fZGVsdGEsDQogICAgICAgICJtZWFuX29mX3Blcl9jbGFzc19hYnNfYXVyb2NfZGVsdGEiOiBmbG9hdChucC5tZWFuKGRlbHRhcykpLA0KICAgICAgICAiZmFpbGVkX2NsYXNzZXMiOiBbDQogICAgICAgICAgICByb3dbImNsYXNzX25hbWUiXQ0KICAgICAgICAgICAgZm9yIHJvdyBpbiByb3dzDQogICAgICAgICAgICBpZiBmbG9hdChyb3dbImUwYiJdWyJzcGVhcm1hbiJdKSA8IGZsb2F0KGNmZy5lMFsic3BlYXJtYW5fbWluIl0pDQogICAgICAgIF0sDQogICAgfQ0KICAgIG9rID0gc3VtbWFyeVsic3BlYXJtYW5fbWluIl0gPj0gZmxvYXQoY2ZnLmUwWyJzcGVhcm1hbl9taW4iXSkgYW5kIG1lYW5fZGVsdGEgPD0gZmxvYXQoY2ZnLmUwWyJhdXJvY19tZWFuX3RvbGVyYW5jZSJdKQ0KICAgIHJldHVybiBvaywgc3VtbWFyeQ0KDQoNCmRlZiBydW5fZTAoY2ZnOiBDb25maWcsIGNsYXNzZXM6IGxpc3Rbc3RyXSwgcGFydDogc3RyID0gImFsbCIpIC0+IGRpY3Q6DQogICAgdmVyc2lvbiA9IHJlY29yZF9zYW1wbGVyX3NvdXJjZShjZmcpDQogICAgdXBzYW1wbGluZyA9IGNoZWNrX2Fub21hbGliX3Vwc2FtcGxpbmcoY2ZnKQ0KICAgIGNoZWNrX2NvcnJ1cHRlZF9pbnB1dCgpDQogICAgcm93cyA9IFtdDQogICAgaW50ZWdyaXR5ID0geyJjb3JydXB0ZWRfaW5wdXQiOiAicGFzcyIsICJhbm9tYWxpYl91cHNhbXBsaW5nIjogdXBzYW1wbGluZywgImNsYXNzZXMiOiB7fX0NCiAgICBmb3IgY2xhc3NfbmFtZSBpbiBjbGFzc2VzOg0KICAgICAgICBmZWF0dXJlcyA9IGxvYWRfZmVhdHVyZV9jYWNoZShjZmcsIGNsYXNzX25hbWUpDQogICAgICAgIGNoZWNrX2ZlYXR1cmVfYWxpZ25tZW50KGNmZywgY2xhc3NfbmFtZSwgZmVhdHVyZXMpDQogICAgICAgIGRpc3RhbmNlX2NhY2hlID0gbG9hZF9kaXN0YW5jZV9jYWNoZShjZmcsIGNsYXNzX25hbWUpDQogICAgICAgIGlmIGxpc3QoZGlzdGFuY2VfY2FjaGVbIm1ldGEiXVsicGF0aHMiXSkgIT0gZmVhdHVyZXNbInBhdGhzIl06DQogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGZlYXR1cmUgYW5kIGRpc3RhbmNlIGNhY2hlcyBhcmUgbm90IGFsaWduZWQuIikNCiAgICAgICAgYWdyZWVtZW50ID0gZGlzdGFuY2VfYWdyZWVtZW50KGNmZywgY2xhc3NfbmFtZSwgZmVhdHVyZXMsIGRpc3RhbmNlX2NhY2hlWyJEIl0sIGZlYXR1cmVzWyJwYXRocyJdKQ0KICAgICAgICBpbnRlZ3JpdHlbImNsYXNzZXMiXVtjbGFzc19uYW1lXSA9IHsiZGlzdGFuY2VfdnNfa25uIjogYWdyZWVtZW50fQ0KICAgICAgICB0cmFpbl9wYXRocyA9IGZlYXR1cmVzWyJwYXRocyJdWzogaW50KGZlYXR1cmVzWyJtZXRhIl1bIm5fdHJhaW4iXSldDQogICAgICAgIGRlbCBmZWF0dXJlcw0KICAgICAgICByb3dzLmFwcGVuZChydW5fY2xhc3MoY2ZnLCBjbGFzc19uYW1lLCBwYXJ0LCB0cmFpbl9wYXRocywgZGlzdGFuY2VfY2FjaGVbIkQiXSkpDQogICAgICAgIGRlbCBkaXN0YW5jZV9jYWNoZQ0KICAgIGUwYV9vayA9IFRydWUNCiAgICBlMGFfc3VtbWFyeSA9IE5vbmUNCiAgICBpZiBwYXJ0IGluIHsiYSIsICJhbGwifToNCiAgICAgICAgcGVyX2NsYXNzID0ge3Jvd1siY2xhc3NfbmFtZSJdOiBmbG9hdChyb3dbImUwYSJdWyJhdXJvYyJdKSBmb3Igcm93IGluIHJvd3N9DQogICAgICAgIGUwYV9zdW1tYXJ5ID0gew0KICAgICAgICAgICAgIm1lYW5fYXVyb2MiOiBmbG9hdChucC5tZWFuKGxpc3QocGVyX2NsYXNzLnZhbHVlcygpKSkpLA0KICAgICAgICAgICAgIm1pbl9hdXJvYyI6IGZsb2F0KG5wLm1pbihsaXN0KHBlcl9jbGFzcy52YWx1ZXMoKSkpKSwNCiAgICAgICAgICAgICJwZXJfY2xhc3MiOiBwZXJfY2xhc3MsDQogICAgICAgIH0NCiAgICAgICAgZTBhX29rID0gZTBhX3N1bW1hcnlbIm1lYW5fYXVyb2MiXSA+PSBmbG9hdChjZmcuZTBbIm1lYW5fYXVyb2NfZmxvb3IiXSkNCiAgICBlMGJfb2sgPSBUcnVlDQogICAgZTBiX3N1bW1hcnkgPSBOb25lDQogICAgaWYgcGFydCBpbiB7ImIiLCAiYWxsIn06DQogICAgICAgIGUwYl9vaywgZTBiX3N1bW1hcnkgPSBfcGFzc2VkX2UwYihjZmcsIHJvd3MpDQogICAgY29tcGxldGUgPSBwYXJ0ID09ICJhbGwiIGFuZCBzZXQoY2xhc3NlcykgPT0gc2V0KGNmZy5jbGFzc2VzKQ0KICAgIHN0YXR1cyA9ICJQQVNTIiBpZiBjb21wbGV0ZSBhbmQgZTBhX29rIGFuZCBlMGJfb2sgZWxzZSAiRkFJTCINCiAgICBpZiBlMGFfb2sgYW5kIGUwYl9vayBhbmQgbm90IGNvbXBsZXRlOg0KICAgICAgICBzdGF0dXMgPSAiUEFSVElBTCINCiAgICBwYXlsb2FkID0gew0KICAgICAgICAic3RhdHVzIjogc3RhdHVzLA0KICAgICAgICAiY29uZmlnX2hhc2giOiBjZmcuaGFzaCwNCiAgICAgICAgImFub21hbGliX3ZlcnNpb24iOiB2ZXJzaW9uLA0KICAgICAgICAicGFydCI6IHBhcnQsDQogICAgICAgICJjbGFzc2VzIjogY2xhc3NlcywNCiAgICAgICAgImUwYSI6IGUwYV9zdW1tYXJ5LA0KICAgICAgICAiZTBiIjogZTBiX3N1bW1hcnksDQogICAgICAgICJwZXJfY2xhc3MiOiByb3dzLA0KICAgICAgICAiaW50ZWdyaXR5IjogaW50ZWdyaXR5LA0KICAgICAgICAicmVmZXJlbmNlX21lYW5fYXVyb2MiOiBmbG9hdChjZmcuZTBbInJlZmVyZW5jZV9tZWFuX2F1cm9jIl0pLA0KICAgIH0NCiAgICB3cml0ZV9qc29uKGdhdGVfcGF0aChjZmcpLCBwYXlsb2FkKQ0KICAgIExPR0dFUi5pbmZvKCJFMCBnYXRlICVzIiwgc3RhdHVzKQ0KICAgIHJldHVybiBwYXlsb2FkDQo=", "commissioning/features.py": "IiIiV2lkZS1SZXNOZXQ1MC0yIHBhdGNoIGZlYXR1cmVzOiBsYXllcjIgKyBsYXllcjMsIDN4MyBwb29saW5nLCA3ODQgeCAxNTM2LiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBsb2dnaW5nDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UNCg0KZnJvbSBjb21taXNzaW9uaW5nLmNvbmZpZyBpbXBvcnQgQ29uZmlnDQpmcm9tIGNvbW1pc3Npb25pbmcuZGF0YSBpbXBvcnQgYWJzb2x1dGVfcGF0aCwgY29sbGVjdF9jbGFzcywgbG9hZF9zcGxpdCwgcmVsYXRpdmVfcGF0aA0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCBhdG9taWNfc2F2ZXosIHJlYWRfanNvbiwgd3JpdGVfanNvbg0KDQpMT0dHRVIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykNCg0KDQpkZWYgcmVhZF9yZ2IocGF0aDogUGF0aCwgY2xhc3NfbmFtZTogc3RyKSAtPiBJbWFnZS5JbWFnZToNCiAgICB0cnk6DQogICAgICAgIHdpdGggSW1hZ2Uub3BlbihwYXRoKSBhcyBpbWFnZToNCiAgICAgICAgICAgIHJnYiA9IGltYWdlLmNvbnZlcnQoIlJHQiIpDQogICAgICAgICAgICByZ2IubG9hZCgpDQogICAgICAgIGlmIHJnYi5zaXplWzBdIDwgMSBvciByZ2Iuc2l6ZVsxXSA8IDE6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJlbXB0eSBpbWFnZSIpDQogICAgICAgIHJldHVybiByZ2INCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiB7cGF0aH0iKSBmcm9tIGV4Yw0KDQoNCmRlZiBidWlsZF90cmFuc2Zvcm0oY2ZnOiBDb25maWcpOg0KICAgICIiIk1hdGNoIEFub21hbGliJ3MgUGF0Y2hjb3JlLmNvbmZpZ3VyZV9wcmVfcHJvY2Vzc29yKDI1NiwgY2VudGVyIGNyb3AgMjI0KS4iIiINCiAgICBmcm9tIHRvcmNodmlzaW9uLnRyYW5zZm9ybXMgaW1wb3J0IHYyDQoNCiAgICByZXR1cm4gdjIuQ29tcG9zZSgNCiAgICAgICAgWw0KICAgICAgICAgICAgdjIuVG9JbWFnZSgpLA0KICAgICAgICAgICAgdjIuVG9EdHlwZSh0b3JjaF9mbG9hdDMyKCksIHNjYWxlPVRydWUpLA0KICAgICAgICAgICAgdjIuUmVzaXplKChjZmcucmVzaXplLCBjZmcucmVzaXplKSwgYW50aWFsaWFzPVRydWUpLA0KICAgICAgICAgICAgdjIuQ2VudGVyQ3JvcCgoY2ZnLmNlbnRlcl9jcm9wLCBjZmcuY2VudGVyX2Nyb3ApKSwNCiAgICAgICAgICAgIHYyLk5vcm1hbGl6ZShtZWFuPWNmZy5pbWFnZW5ldF9tZWFuLCBzdGQ9Y2ZnLmltYWdlbmV0X3N0ZCksDQogICAgICAgIF0NCiAgICApDQoNCg0KZGVmIHRvcmNoX2Zsb2F0MzIoKToNCiAgICBpbXBvcnQgdG9yY2gNCg0KICAgIHJldHVybiB0b3JjaC5mbG9hdDMyDQoNCg0KZGVmIGJ1aWxkX2ZlYXR1cmVfZXh0cmFjdG9yKGNmZzogQ29uZmlnKToNCiAgICAiIiJQcmVmZXIgQW5vbWFsaWIncyBleHRyYWN0b3Igc28gdGhlIGNhY2hlIGFuZCBFMGIgc2hhcmUgd2VpZ2h0cyBhbmQgaG9va3MuIiIiDQogICAgaW1wb3J0IHRvcmNoLm5uIGFzIG5uDQoNCiAgICB0cnk6DQogICAgICAgIGZyb20gYW5vbWFsaWIubW9kZWxzLmNvbXBvbmVudHMgaW1wb3J0IFRpbW1GZWF0dXJlRXh0cmFjdG9yDQogICAgZXhjZXB0IEltcG9ydEVycm9yIGFzIGV4YzoNCiAgICAgICAgTE9HR0VSLndhcm5pbmcoImFub21hbGliIGlzIG5vdCBpbnN0YWxsZWQgKCVzKTsgdXNpbmcgYSB0aW1tIGhvb2sgZXh0cmFjdG9yLiIsIGV4YykNCiAgICAgICAgcmV0dXJuIF9Ib29rRXh0cmFjdG9yKGNmZy5iYWNrYm9uZSwgY2ZnLmxheWVycyksIGYidGltbS5ob29rcyAoe2V4Y30pIg0KDQogICAgbW9kZWwgPSBUaW1tRmVhdHVyZUV4dHJhY3RvcihiYWNrYm9uZT1jZmcuYmFja2JvbmUsIGxheWVycz1saXN0KGNmZy5sYXllcnMpLCBwcmVfdHJhaW5lZD1UcnVlKQ0KICAgIGlmIG5vdCBpc2luc3RhbmNlKG1vZGVsLCBubi5Nb2R1bGUpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRpbW1GZWF0dXJlRXh0cmFjdG9yIGRpZCBub3QgcmV0dXJuIGEgdG9yY2ggbW9kdWxlLiIpDQogICAgcmV0dXJuIG1vZGVsLCBmImFub21hbGliLlRpbW1GZWF0dXJlRXh0cmFjdG9yIHdlaWdodHM9e193ZWlnaHRfdGFnKG1vZGVsKX0iDQoNCg0KZGVmIF93ZWlnaHRfdGFnKG1vZGVsKSAtPiBzdHI6DQogICAgIiIidGltbSBuYW1lcyBpdHMgd2VpZ2h0cywgZS5nLiB3aWRlX3Jlc25ldDUwXzIucmFjbV9pbjFrIHZzIC50dl9pbjFrLiBUaGUgcGFwZXIgbXVzdCBzdGF0ZSB3aGljaC4iIiINCiAgICBmb3IgbW9kdWxlIGluIG1vZGVsLm1vZHVsZXMoKToNCiAgICAgICAgY29uZmlnID0gZ2V0YXR0cihtb2R1bGUsICJwcmV0cmFpbmVkX2NmZyIsIE5vbmUpDQogICAgICAgIGlmIGlzaW5zdGFuY2UoY29uZmlnLCBkaWN0KSBhbmQgKGNvbmZpZy5nZXQoInRhZyIpIG9yIGNvbmZpZy5nZXQoImhmX2h1Yl9pZCIpKToNCiAgICAgICAgICAgIGFyY2hpdGVjdHVyZSA9IGNvbmZpZy5nZXQoImFyY2hpdGVjdHVyZSIsICIiKQ0KICAgICAgICAgICAgdGFnID0gY29uZmlnLmdldCgidGFnIiwgIiIpDQogICAgICAgICAgICByZXR1cm4gZiJ7YXJjaGl0ZWN0dXJlfS57dGFnfSIgaWYgdGFnIGVsc2Ugc3RyKGNvbmZpZy5nZXQoImhmX2h1Yl9pZCIpKQ0KICAgIHJldHVybiAidW5rbm93biINCg0KDQpjbGFzcyBfSG9va0V4dHJhY3RvcjoNCiAgICBkZWYgX19pbml0X18oc2VsZiwgYmFja2JvbmU6IHN0ciwgbGF5ZXJzOiBsaXN0W3N0cl0pIC0+IE5vbmU6DQogICAgICAgIGltcG9ydCB0aW1tDQogICAgICAgIGltcG9ydCB0b3JjaC5ubiBhcyBubg0KDQogICAgICAgIHNlbGYubGF5ZXJzID0gbGlzdChsYXllcnMpDQogICAgICAgIHNlbGYuYmFja2JvbmUgPSB0aW1tLmNyZWF0ZV9tb2RlbChiYWNrYm9uZSwgcHJldHJhaW5lZD1UcnVlKQ0KICAgICAgICBzZWxmLmJhY2tib25lLmV2YWwoKQ0KICAgICAgICBuYW1lZCA9IGRpY3Qoc2VsZi5iYWNrYm9uZS5uYW1lZF9tb2R1bGVzKCkpDQogICAgICAgIG1pc3NpbmcgPSBbbmFtZSBmb3IgbmFtZSBpbiBzZWxmLmxheWVycyBpZiBuYW1lIG5vdCBpbiBuYW1lZF0NCiAgICAgICAgaWYgbWlzc2luZzoNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkJhY2tib25lIHtiYWNrYm9uZX0gaGFzIG5vIGxheWVycyB7bWlzc2luZ30iKQ0KICAgICAgICBzZWxmLl9mZWF0dXJlczogZGljdCA9IHt9DQogICAgICAgIGZvciBuYW1lIGluIHNlbGYubGF5ZXJzOg0KICAgICAgICAgICAgbmFtZWRbbmFtZV0ucmVnaXN0ZXJfZm9yd2FyZF9ob29rKHNlbGYuX21ha2VfaG9vayhuYW1lKSkNCiAgICAgICAgc2VsZi5fbm4gPSBubg0KDQogICAgZGVmIF9tYWtlX2hvb2soc2VsZiwgbmFtZTogc3RyKToNCiAgICAgICAgZGVmIGhvb2soX21vZHVsZSwgX2lucHV0cywgb3V0cHV0KSAtPiBOb25lOg0KICAgICAgICAgICAgc2VsZi5fZmVhdHVyZXNbbmFtZV0gPSBvdXRwdXQNCg0KICAgICAgICByZXR1cm4gaG9vaw0KDQogICAgZGVmIGV2YWwoc2VsZik6DQogICAgICAgIHNlbGYuYmFja2JvbmUuZXZhbCgpDQogICAgICAgIHJldHVybiBzZWxmDQoNCiAgICBkZWYgdG8oc2VsZiwgZGV2aWNlKToNCiAgICAgICAgc2VsZi5iYWNrYm9uZS50byhkZXZpY2UpDQogICAgICAgIHJldHVybiBzZWxmDQoNCiAgICBkZWYgX19jYWxsX18oc2VsZiwgYmF0Y2gpOg0KICAgICAgICBzZWxmLl9mZWF0dXJlcyA9IHt9DQogICAgICAgIF8gPSBzZWxmLmJhY2tib25lKGJhdGNoKQ0KICAgICAgICBtaXNzaW5nID0gW25hbWUgZm9yIG5hbWUgaW4gc2VsZi5sYXllcnMgaWYgbmFtZSBub3QgaW4gc2VsZi5fZmVhdHVyZXNdDQogICAgICAgIGlmIG1pc3Npbmc6DQogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJGZWF0dXJlIGhvb2tzIGRpZCBub3QgZmlyZSBmb3Ige21pc3Npbmd9IikNCiAgICAgICAgcmV0dXJuIHtuYW1lOiBzZWxmLl9mZWF0dXJlc1tuYW1lXSBmb3IgbmFtZSBpbiBzZWxmLmxheWVyc30NCg0KDQpkZWYgcG9vbGVkX2VtYmVkZGluZyhmZWF0dXJlczogZGljdCwgbGF5ZXJzOiBsaXN0W3N0cl0sIHBvb2wsIHVwc2FtcGxlX21vZGU6IHN0cik6DQogICAgIiIiQXZlcmFnZS1wb29sIGVhY2ggbWFwLCB1cHNhbXBsZSBsYXRlciBtYXBzIHRvIHRoZSBmaXJzdCBtYXAsIHRoZW4gY29uY2F0ZW5hdGUuIiIiDQogICAgaW1wb3J0IHRvcmNoDQogICAgaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRg0KDQogICAgcG9vbGVkID0gW3Bvb2woZmVhdHVyZXNbbmFtZV0pIGZvciBuYW1lIGluIGxheWVyc10NCiAgICBlbWJlZGRpbmcgPSBwb29sZWRbMF0NCiAgICBmb3IgbGF5ZXIgaW4gcG9vbGVkWzE6XToNCiAgICAgICAga3dhcmdzID0geyJhbGlnbl9jb3JuZXJzIjogRmFsc2V9IGlmIHVwc2FtcGxlX21vZGUgaW4geyJiaWxpbmVhciIsICJiaWN1YmljIn0gZWxzZSB7fQ0KICAgICAgICBsYXllciA9IEYuaW50ZXJwb2xhdGUobGF5ZXIsIHNpemU9ZW1iZWRkaW5nLnNoYXBlWy0yOl0sIG1vZGU9dXBzYW1wbGVfbW9kZSwgKiprd2FyZ3MpDQogICAgICAgIGVtYmVkZGluZyA9IHRvcmNoLmNhdCgoZW1iZWRkaW5nLCBsYXllciksIGRpbT0xKQ0KICAgIHJldHVybiBlbWJlZGRpbmcNCg0KDQpkZWYgcGF0Y2hlc19hbmRfZ2FwKGVtYmVkZGluZyk6DQogICAgIiIiZW1iZWRkaW5nOiAoQiwgQywgSCwgVykgLT4gcGF0Y2hlcyAoQiwgSCpXLCBDKSwgZ2FwIChCLCBDKS4iIiINCiAgICBiYXRjaCwgY2hhbm5lbHMsIF9oZWlnaHQsIF93aWR0aCA9IGVtYmVkZGluZy5zaGFwZQ0KICAgIHBhdGNoZXMgPSBlbWJlZGRpbmcucGVybXV0ZSgwLCAyLCAzLCAxKS5yZXNoYXBlKGJhdGNoLCAtMSwgY2hhbm5lbHMpDQogICAgZ2FwID0gcGF0Y2hlcy5tZWFuKGRpbT0xKQ0KICAgIHJldHVybiBwYXRjaGVzLCBnYXANCg0KDQpkZWYgZmVhdHVyZV9jYWNoZV9wYXRoKGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIpIC0+IFBhdGg6DQogICAgcmV0dXJuIGNmZy53b3JrX3Jvb3QgLyAiY2FjaGUiIC8gImZlYXR1cmVzIiAvIGYie2NsYXNzX25hbWV9Lm5weiINCg0KDQpkZWYgZmVhdHVyZV9tZXRhX3BhdGgoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gUGF0aDoNCiAgICByZXR1cm4gY2ZnLndvcmtfcm9vdCAvICJjYWNoZSIgLyAiZmVhdHVyZXMiIC8gZiJ7Y2xhc3NfbmFtZX0uanNvbiINCg0KDQpkZWYgaW1hZ2Vfb3JkZXIoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gbGlzdFtzdHJdOg0KICAgICIiIlRyYWluLCB0aGVuIG9mZmljaWFsIGdvb2QgdGVzdCwgdGhlbiBkZWZlY3RzLiBUaGlzIG9yZGVyIGlzIHRoZSByb3cgb3JkZXIgb2YgdGhlIGNhY2hlLiIiIg0KICAgIGNvbGxlY3RlZCA9IGNvbGxlY3RfY2xhc3MoY2ZnLmRhdGFfcm9vdCwgY2xhc3NfbmFtZSkNCiAgICBwYXRocyA9IFtyZWxhdGl2ZV9wYXRoKHBhdGgsIGNmZy5kYXRhX3Jvb3QpIGZvciBwYXRoIGluIGNvbGxlY3RlZFsidHJhaW4iXV0NCiAgICBwYXRocy5leHRlbmQocmVsYXRpdmVfcGF0aChwYXRoLCBjZmcuZGF0YV9yb290KSBmb3IgcGF0aCBpbiBjb2xsZWN0ZWRbInRlc3RfZ29vZCJdKQ0KICAgIHBhdGhzLmV4dGVuZChyZWxhdGl2ZV9wYXRoKHBhdGgsIGNmZy5kYXRhX3Jvb3QpIGZvciBfLCBwYXRoIGluIGNvbGxlY3RlZFsidGVzdF9kZWZlY3QiXSkNCiAgICByZXR1cm4gcGF0aHMNCg0KDQpkZWYgZXh0cmFjdF9jbGFzcyhjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyLCBkZXZpY2U6IHN0ciB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIGltcG9ydCB0b3JjaA0KICAgIGltcG9ydCB0b3JjaC5ubiBhcyBubg0KDQogICAgb3V0X3BhdGggPSBmZWF0dXJlX2NhY2hlX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIG1ldGFfcGF0aCA9IGZlYXR1cmVfbWV0YV9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBvdXRfcGF0aC5pc19maWxlKCkgYW5kIG1ldGFfcGF0aC5pc19maWxlKCk6DQogICAgICAgIG1ldGEgPSByZWFkX2pzb24obWV0YV9wYXRoKQ0KICAgICAgICBpZiBtZXRhLmdldCgiY29uZmlnX2hhc2giKSA9PSBjZmcuaGFzaCBhbmQgaW50KG1ldGEuZ2V0KCJuX2ltYWdlcyIsIDApKSA+IDA6DQogICAgICAgICAgICBMT0dHRVIuaW5mbygiRmVhdHVyZSBjYWNoZSBleGlzdHMgZm9yICVzIiwgY2xhc3NfbmFtZSkNCiAgICAgICAgICAgIHJldHVybg0KDQogICAgcGF0aHMgPSBpbWFnZV9vcmRlcihjZmcsIGNsYXNzX25hbWUpDQogICAgaWYgbm90IHBhdGhzOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IG5vIGltYWdlcyB0byBlbmNvZGUiKQ0KICAgIGRldmljZV90ID0gdG9yY2guZGV2aWNlKGRldmljZSBvciAoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikpDQogICAgZXh0cmFjdG9yLCBzb3VyY2UgPSBidWlsZF9mZWF0dXJlX2V4dHJhY3RvcihjZmcpDQogICAgZXh0cmFjdG9yID0gZXh0cmFjdG9yLmV2YWwoKS50byhkZXZpY2VfdCkNCiAgICB0cmFuc2Zvcm0gPSBidWlsZF90cmFuc2Zvcm0oY2ZnKQ0KICAgIHBvb2wgPSBubi5BdmdQb29sMmQoa2VybmVsX3NpemU9Mywgc3RyaWRlPTEsIHBhZGRpbmc9MSkudG8oZGV2aWNlX3QpDQoNCiAgICBwYXRjaF9yb3dzOiBsaXN0W25wLm5kYXJyYXldID0gW10NCiAgICBnYXBfcm93czogbGlzdFtucC5uZGFycmF5XSA9IFtdDQogICAgYmF0Y2hfc2l6ZSA9IGNmZy5mZWF0dXJlX2JhdGNoX3NpemUNCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGxlbihwYXRocyksIGJhdGNoX3NpemUpOg0KICAgICAgICAgICAgYmF0Y2hfcGF0aHMgPSBwYXRoc1tzdGFydCA6IHN0YXJ0ICsgYmF0Y2hfc2l6ZV0NCiAgICAgICAgICAgIGltYWdlcyA9IFtdDQogICAgICAgICAgICBmb3IgcmVsYXRpdmUgaW4gYmF0Y2hfcGF0aHM6DQogICAgICAgICAgICAgICAgcmdiID0gcmVhZF9yZ2IoYWJzb2x1dGVfcGF0aChjZmcuZGF0YV9yb290LCByZWxhdGl2ZSksIGNsYXNzX25hbWUpDQogICAgICAgICAgICAgICAgaW1hZ2VzLmFwcGVuZCh0cmFuc2Zvcm0ocmdiKSkNCiAgICAgICAgICAgIGJhdGNoID0gdG9yY2guc3RhY2soaW1hZ2VzLCBkaW09MCkudG8oZGV2aWNlX3QpDQogICAgICAgICAgICBmZWF0dXJlcyA9IGV4dHJhY3RvcihiYXRjaCkNCiAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKGZlYXR1cmVzLCBkaWN0KToNCiAgICAgICAgICAgICAgICBmZWF0dXJlcyA9IHtuYW1lOiB0ZW5zb3IgZm9yIG5hbWUsIHRlbnNvciBpbiB6aXAoY2ZnLmxheWVycywgZmVhdHVyZXMpfQ0KICAgICAgICAgICAgZW1iZWRkaW5nID0gcG9vbGVkX2VtYmVkZGluZyhmZWF0dXJlcywgY2ZnLmxheWVycywgcG9vbCwgY2ZnLnVwc2FtcGxlX21vZGUpDQogICAgICAgICAgICBpZiBlbWJlZGRpbmcuc2hhcGVbMV0gIT0gMTUzNiBvciBlbWJlZGRpbmcuc2hhcGVbLTI6XSAhPSAoMjgsIDI4KToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IHVuZXhwZWN0ZWQgZW1iZWRkaW5nIHNoYXBlIHt0dXBsZShlbWJlZGRpbmcuc2hhcGUpfSIpDQogICAgICAgICAgICBwYXRjaGVzLCBnYXAgPSBwYXRjaGVzX2FuZF9nYXAoZW1iZWRkaW5nKQ0KICAgICAgICAgICAgcGF0Y2hlc19ucCA9IHBhdGNoZXMuZGV0YWNoKCkuY3B1KCkubnVtcHkoKQ0KICAgICAgICAgICAgZ2FwX25wID0gZ2FwLmRldGFjaCgpLmNwdSgpLm51bXB5KCkNCiAgICAgICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShwYXRjaGVzX25wKS5hbGwoKSBvciBub3QgbnAuaXNmaW5pdGUoZ2FwX25wKS5hbGwoKToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IG5vbi1maW5pdGUgZmVhdHVyZXMgaW4gYmF0Y2ggc3RhcnRpbmcgYXQge2JhdGNoX3BhdGhzWzBdfSIpDQogICAgICAgICAgICBwYXRjaF9yb3dzLmFwcGVuZChwYXRjaGVzX25wLmFzdHlwZShucC5mbG9hdDE2KSkNCiAgICAgICAgICAgIGdhcF9yb3dzLmFwcGVuZChnYXBfbnAuYXN0eXBlKG5wLmZsb2F0MzIpKQ0KDQogICAgcGF0Y2hlc19hbGwgPSBucC5jb25jYXRlbmF0ZShwYXRjaF9yb3dzLCBheGlzPTApDQogICAgZ2FwX2FsbCA9IG5wLmNvbmNhdGVuYXRlKGdhcF9yb3dzLCBheGlzPTApDQogICAgaWYgcGF0Y2hlc19hbGwuc2hhcGVbMTpdICE9ICg3ODQsIDE1MzYpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGV4cGVjdGVkIHBhdGNoZXMgKE4sIDc4NCwgMTUzNiksIGdvdCB7cGF0Y2hlc19hbGwuc2hhcGV9IikNCiAgICBzcGxpdCA9IGxvYWRfc3BsaXQoY2ZnLCBjbGFzc19uYW1lLCAwKQ0KICAgIG5fdHJhaW4gPSBpbnQoc3BsaXRbIm5fdHJhaW4iXSkNCiAgICBuX3Rlc3RfZ29vZCA9IGludChzcGxpdFsibl90ZXN0X2dvb2QiXSkNCiAgICBuX3Rlc3RfZGVmZWN0ID0gaW50KHNwbGl0WyJuX3Rlc3RfZGVmZWN0Il0pDQogICAgaWYgcGF0Y2hlc19hbGwuc2hhcGVbMF0gIT0gbl90cmFpbiArIG5fdGVzdF9nb29kICsgbl90ZXN0X2RlZmVjdDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBmZWF0dXJlIHJvd3MgZG8gbm90IG1hdGNoIHNwbGl0IGNvdW50cyIpDQoNCiAgICBhdG9taWNfc2F2ZXooDQogICAgICAgIG91dF9wYXRoLA0KICAgICAgICBwYXRjaGVzPXBhdGNoZXNfYWxsLA0KICAgICAgICBnYXA9Z2FwX2FsbCwNCiAgICAgICAgcGF0aHM9bnAuYXJyYXkocGF0aHMpLA0KICAgICkNCiAgICB3cml0ZV9qc29uKA0KICAgICAgICBtZXRhX3BhdGgsDQogICAgICAgIHsNCiAgICAgICAgICAgICJjb25maWdfaGFzaCI6IGNmZy5oYXNoLA0KICAgICAgICAgICAgImNsYXNzX25hbWUiOiBjbGFzc19uYW1lLA0KICAgICAgICAgICAgIm5faW1hZ2VzIjogaW50KHBhdGNoZXNfYWxsLnNoYXBlWzBdKSwNCiAgICAgICAgICAgICJuX3RyYWluIjogbl90cmFpbiwNCiAgICAgICAgICAgICJuX3Rlc3RfZ29vZCI6IG5fdGVzdF9nb29kLA0KICAgICAgICAgICAgIm5fdGVzdF9kZWZlY3QiOiBuX3Rlc3RfZGVmZWN0LA0KICAgICAgICAgICAgInBhdGNoX3NoYXBlIjogWzc4NCwgMTUzNl0sDQogICAgICAgICAgICAiZXh0cmFjdG9yIjogc291cmNlLA0KICAgICAgICAgICAgInVwc2FtcGxlX21vZGUiOiBjZmcudXBzYW1wbGVfbW9kZSwNCiAgICAgICAgICAgICJyZXNpemUiOiBjZmcucmVzaXplLA0KICAgICAgICAgICAgImNlbnRlcl9jcm9wIjogY2ZnLmNlbnRlcl9jcm9wLA0KICAgICAgICAgICAgImJ5dGVzX3BhdGNoZXMiOiBpbnQocGF0Y2hlc19hbGwubmJ5dGVzKSwNCiAgICAgICAgfSwNCiAgICApDQogICAgTE9HR0VSLmluZm8oDQogICAgICAgICIlcyBmZWF0dXJlczogJXMgaW1hZ2VzLCAlLjJmIEdCIGZsb2F0MTYiLA0KICAgICAgICBjbGFzc19uYW1lLA0KICAgICAgICBwYXRjaGVzX2FsbC5zaGFwZVswXSwNCiAgICAgICAgcGF0Y2hlc19hbGwubmJ5dGVzIC8gKDEwMjQqKjMpLA0KICAgICkNCg0KDQpkZWYgbG9hZF9nYXBfdGFibGUoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gZGljdDoNCiAgICAiIiJHQVAgdmVjdG9ycyBvbmx5LiBUcmFqZWN0b3J5IHNlbGVjdGlvbiBtdXN0IG5vdCBsb2FkIHRoZSBwYXRjaCB0ZW5zb3IuIiIiDQogICAgcGF0aCA9IGZlYXR1cmVfY2FjaGVfcGF0aChjZmcsIGNsYXNzX25hbWUpDQogICAgbWV0YV9wYXRoID0gZmVhdHVyZV9tZXRhX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKSBvciBub3QgbWV0YV9wYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNaXNzaW5nIGZlYXR1cmUgY2FjaGUgZm9yIHtjbGFzc19uYW1lfS4gUnVuIGZlYXR1cmVzIGZpcnN0LiIpDQogICAgbWV0YSA9IHJlYWRfanNvbihtZXRhX3BhdGgpDQogICAgaWYgbWV0YS5nZXQoImNvbmZpZ19oYXNoIikgIT0gY2ZnLmhhc2g6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkZlYXR1cmUgY2FjaGUgaGFzaCBtaXNtYXRjaCBmb3Ige2NsYXNzX25hbWV9LiIpDQogICAgd2l0aCBucC5sb2FkKHBhdGgsIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgZGF0YToNCiAgICAgICAgZ2FwID0gbnAuYXJyYXkoZGF0YVsiZ2FwIl0pDQogICAgICAgIHBhdGhzID0gW3N0cihpdGVtKSBmb3IgaXRlbSBpbiBkYXRhWyJwYXRocyJdLnRvbGlzdCgpXQ0KICAgIGlmIG5vdCBucC5pc2Zpbml0ZShnYXApLmFsbCgpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IG5vbi1maW5pdGUgR0FQIGZlYXR1cmVzIikNCiAgICByZXR1cm4geyJnYXAiOiBnYXAsICJwYXRocyI6IHBhdGhzLCAibWV0YSI6IG1ldGF9DQoNCg0KZGVmIGxvYWRfZmVhdHVyZV9jYWNoZShjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyKSAtPiBkaWN0Og0KICAgIHBhdGggPSBmZWF0dXJlX2NhY2hlX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIG1ldGFfcGF0aCA9IGZlYXR1cmVfbWV0YV9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBub3QgcGF0aC5pc19maWxlKCkgb3Igbm90IG1ldGFfcGF0aC5pc19maWxlKCk6DQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTWlzc2luZyBmZWF0dXJlIGNhY2hlIGZvciB7Y2xhc3NfbmFtZX0uIFJ1biBmZWF0dXJlcyBmaXJzdC4iKQ0KICAgIG1ldGEgPSByZWFkX2pzb24obWV0YV9wYXRoKQ0KICAgIGlmIG1ldGEuZ2V0KCJjb25maWdfaGFzaCIpICE9IGNmZy5oYXNoOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJGZWF0dXJlIGNhY2hlIGhhc2ggbWlzbWF0Y2ggZm9yIHtjbGFzc19uYW1lfS4iKQ0KICAgIHdpdGggbnAubG9hZChwYXRoLCBhbGxvd19waWNrbGU9RmFsc2UpIGFzIGRhdGE6DQogICAgICAgIHBhdGNoZXMgPSBucC5hcnJheShkYXRhWyJwYXRjaGVzIl0pDQogICAgICAgIGdhcCA9IG5wLmFycmF5KGRhdGFbImdhcCJdKQ0KICAgICAgICBwYXRocyA9IFtzdHIoaXRlbSkgZm9yIGl0ZW0gaW4gZGF0YVsicGF0aHMiXS50b2xpc3QoKV0NCiAgICBpZiBwYXRjaGVzLmR0eXBlICE9IG5wLmZsb2F0MTYgb3IgcGF0Y2hlcy5zaGFwZVsxOl0gIT0gKDc4NCwgMTUzNik6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogYmFkIHBhdGNoIGNhY2hlIHNoYXBlIHtwYXRjaGVzLnNoYXBlfSB7cGF0Y2hlcy5kdHlwZX0iKQ0KICAgIGlmIG5vdCBucC5pc2Zpbml0ZShwYXRjaGVzLmFzdHlwZShucC5mbG9hdDMyKSkuYWxsKCkgb3Igbm90IG5wLmlzZmluaXRlKGdhcCkuYWxsKCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogbm9uLWZpbml0ZSB2YWx1ZXMgaW4gdGhlIGZlYXR1cmUgY2FjaGUiKQ0KICAgIHJldHVybiB7InBhdGNoZXMiOiBwYXRjaGVzLCAiZ2FwIjogZ2FwLCAicGF0aHMiOiBwYXRocywgIm1ldGEiOiBtZXRhfQ0KDQoNCmRlZiBleHRyYWN0KGNmZzogQ29uZmlnLCBjbGFzc2VzOiBsaXN0W3N0cl0sIGRldmljZTogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IE5vbmU6DQogICAgZm9yIGNsYXNzX25hbWUgaW4gY2xhc3NlczoNCiAgICAgICAgZXh0cmFjdF9jbGFzcyhjZmcsIGNsYXNzX25hbWUsIGRldmljZT1kZXZpY2UpDQo=", "commissioning/integrity.py": "IiIiQ2hlY2tzIHRoYXQgbXVzdCBwYXNzIGJlZm9yZSBhbnkgUlEgdHJhamVjdG9yeSBpcyB0cnVzdGVkLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBvcw0KaW1wb3J0IHRlbXBmaWxlDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQoNCmZyb20gY29tbWlzc2lvbmluZy5jb25maWcgaW1wb3J0IENvbmZpZw0KZnJvbSBjb21taXNzaW9uaW5nLmRhdGEgaW1wb3J0IGxvYWRfc3BsaXQNCmZyb20gY29tbWlzc2lvbmluZy5mZWF0dXJlcyBpbXBvcnQgaW1hZ2Vfb3JkZXIsIGxvYWRfZmVhdHVyZV9jYWNoZSwgcmVhZF9yZ2INCmZyb20gY29tbWlzc2lvbmluZy5wYXRjaGNvcmUgaW1wb3J0IGtubl9zY29yZXMsIHNjb3Jlc19mb3JfYmFuaw0KDQoNCmRlZiBjaGVja19jb3JydXB0ZWRfaW5wdXQoY2xhc3NfbmFtZTogc3RyID0gImludGVncml0eSIpIC0+IE5vbmU6DQogICAgaGFuZGxlLCBuYW1lID0gdGVtcGZpbGUubWtzdGVtcChzdWZmaXg9Ii5wbmciKQ0KICAgIG9zLndyaXRlKGhhbmRsZSwgYiJ0aGlzIGlzIG5vdCBhIHBuZyIpDQogICAgb3MuY2xvc2UoaGFuZGxlKQ0KICAgIHBhdGggPSBQYXRoKG5hbWUpDQogICAgdHJ5Og0KICAgICAgICB0cnk6DQogICAgICAgICAgICByZWFkX3JnYihwYXRoLCBjbGFzc19uYW1lKQ0KICAgICAgICBleGNlcHQgUnVudGltZUVycm9yIGFzIGV4YzoNCiAgICAgICAgICAgIHRleHQgPSBzdHIoZXhjKQ0KICAgICAgICAgICAgaWYgY2xhc3NfbmFtZSBub3QgaW4gdGV4dCBvciBzdHIocGF0aCkgbm90IGluIHRleHQ6DQogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiQ29ycnVwdGVkLWltYWdlIGVycm9yIG11c3QgbmFtZSB0aGUgY2xhc3MgYW5kIHBhdGgsIGdvdDoge3RleHR9IikgZnJvbSBleGMNCiAgICAgICAgICAgIHJldHVybg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkEgY29ycnVwdGVkIGltYWdlIHdhcyByZWFkIHdpdGhvdXQgUnVudGltZUVycm9yLiIpDQogICAgZmluYWxseToNCiAgICAgICAgcGF0aC51bmxpbmsobWlzc2luZ19vaz1UcnVlKQ0KDQoNCmRlZiBjaGVja19mZWF0dXJlX2FsaWdubWVudChjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyLCBjYWNoZTogZGljdCB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIGlmIGNhY2hlIGlzIE5vbmU6DQogICAgICAgIGNhY2hlID0gbG9hZF9mZWF0dXJlX2NhY2hlKGNmZywgY2xhc3NfbmFtZSkNCiAgICBleHBlY3RlZCA9IGltYWdlX29yZGVyKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBjYWNoZVsicGF0aHMiXSAhPSBleHBlY3RlZDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBmZWF0dXJlLWNhY2hlIG9yZGVyIGRvZXMgbm90IG1hdGNoIHRoZSBzb3J0ZWQgZGF0YXNldCBvcmRlci4iKQ0KICAgIGlmIGNhY2hlWyJwYXRjaGVzIl0uc2hhcGVbMTpdICE9ICg3ODQsIDE1MzYpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IHBhdGNoIHRlbnNvciBzaGFwZSBpcyB7Y2FjaGVbJ3BhdGNoZXMnXS5zaGFwZX0iKQ0KICAgIGlmIGxlbihjYWNoZVsicGF0aHMiXSkgIT0gbGVuKHNldChjYWNoZVsicGF0aHMiXSkpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGR1cGxpY2F0ZSBwYXRocyBpbiB0aGUgZmVhdHVyZSBjYWNoZS4iKQ0KICAgIHNwbGl0ID0gbG9hZF9zcGxpdChjZmcsIGNsYXNzX25hbWUsIDApDQogICAgbl90cmFpbiA9IGludChzcGxpdFsibl90cmFpbiJdKQ0KICAgIHRyYWluX3BhdGhzID0gY2FjaGVbInBhdGhzIl1bOm5fdHJhaW5dDQogICAgdGVzdF9wYXRocyA9IGNhY2hlWyJwYXRocyJdW25fdHJhaW46XQ0KICAgIGlmIHNldChzcGxpdFsiY2FsaWJyYXRpb24iXSkgfCBzZXQoc3BsaXRbInBvb2wiXSkgIT0gc2V0KHRyYWluX3BhdGhzKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBDIGFuZCBQIGFyZSBub3QgYSBwYXJ0aXRpb24gb2YgdGhlIGNhY2hlZCB0cmFpbmluZyBpbWFnZXMuIikNCiAgICBvZmZpY2lhbF90ZXN0ID0gbGlzdChzcGxpdFsidGVzdF9nb29kIl0pICsgW2l0ZW1bInBhdGgiXSBmb3IgaXRlbSBpbiBzcGxpdFsidGVzdF9kZWZlY3QiXV0NCiAgICBpZiB0ZXN0X3BhdGhzICE9IG9mZmljaWFsX3Rlc3Q6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogY2FjaGVkIHRlc3Qgb3JkZXIgZG9lcyBub3QgbWF0Y2ggdGhlIHNwbGl0LiIpDQoNCg0KZGVmIGFzc2VydF9tZXRob2RzX2FncmVlKA0KICAgIHBhdGNoZXM6IG5wLm5kYXJyYXksDQogICAgZGlzdGFuY2U6IG5wLm5kYXJyYXksDQogICAgcXVlcnlfaW5kaWNlczogbnAubmRhcnJheSB8IGxpc3RbaW50XSwNCiAgICBiYW5rX2luZGljZXM6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0sDQogICAgdG9sZXJhbmNlOiBmbG9hdCA9IDFlLTQsDQopIC0+IGZsb2F0Og0KICAgICIiIlRoZSBkaXN0YW5jZS10ZW5zb3Igc2NvcmUgYW5kIGEgZGlyZWN0IDEtTk4gb3ZlciB0aGUgc2FtZSBwYXRjaGVzIG11c3QgYWdyZWUuDQoNCiAgICBCb3RoIGdvIHRocm91Z2ggZmxvYXQzMiBHRU1NLWJhc2VkIGRpc3RhbmNlcyB3aXRoIGRpZmZlcmVudCBibG9jayBzaGFwZXMsIHNvIHRoZSBjaGVjayBpcyByZWxhdGl2ZS4NCiAgICAiIiINCiAgICB2aWFfZGlzdGFuY2UgPSBzY29yZXNfZm9yX2JhbmsoZGlzdGFuY2UsIHF1ZXJ5X2luZGljZXMsIGJhbmtfaW5kaWNlcykNCiAgICB2aWFfa25uID0ga25uX3Njb3JlcyhwYXRjaGVzW25wLmFzYXJyYXkocXVlcnlfaW5kaWNlcyldLCBwYXRjaGVzW25wLmFzYXJyYXkoYmFua19pbmRpY2VzKV0pDQogICAgZ2FwID0gZmxvYXQobnAubWF4KG5wLmFicyh2aWFfZGlzdGFuY2UgLSB2aWFfa25uKSAvIG5wLm1heGltdW0oMS4wLCBucC5hYnModmlhX2tubikpKSkNCiAgICBpZiBnYXAgPiB0b2xlcmFuY2U6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkRpc3RhbmNlIHRlbnNvciBhbmQgZGlyZWN0IGtOTiBkaWZmZXIgYnkgYSByZWxhdGl2ZSB7Z2FwfSAodG9sZXJhbmNlIHt0b2xlcmFuY2V9KS4iKQ0KICAgIHJldHVybiBnYXANCg==", "commissioning/io_utils.py": "IiIiQXRvbWljIHdyaXRlcyBhbmQgc21hbGwgZmlsZSBoZWxwZXJzLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBqc29uDQppbXBvcnQgb3MNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KZnJvbSB0eXBpbmcgaW1wb3J0IEFueQ0KDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KDQpkZWYgZW5zdXJlX2RpcihwYXRoOiBQYXRoKSAtPiBQYXRoOg0KICAgIHBhdGgubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIHJldHVybiBwYXRoDQoNCg0KZGVmIHdyaXRlX2pzb24ocGF0aDogUGF0aCwgcGF5bG9hZDogZGljdFtzdHIsIEFueV0pIC0+IE5vbmU6DQogICAgcGF0aCA9IFBhdGgocGF0aCkNCiAgICBlbnN1cmVfZGlyKHBhdGgucGFyZW50KQ0KICAgIHRtcCA9IHBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXggKyAiLnRtcCIpDQogICAgdG1wLndyaXRlX3RleHQoanNvbi5kdW1wcyhwYXlsb2FkLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpLCBlbmNvZGluZz0idXRmLTgiKQ0KICAgIG9zLnJlcGxhY2UodG1wLCBwYXRoKQ0KDQoNCmRlZiByZWFkX2pzb24ocGF0aDogUGF0aCkgLT4gZGljdFtzdHIsIEFueV06DQogICAgcmV0dXJuIGpzb24ubG9hZHMoUGF0aChwYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpDQoNCg0KZGVmIGF0b21pY19zYXZleihwYXRoOiBQYXRoLCAqKmFycmF5czogQW55KSAtPiBOb25lOg0KICAgIHBhdGggPSBQYXRoKHBhdGgpDQogICAgZW5zdXJlX2RpcihwYXRoLnBhcmVudCkNCiAgICBwYXJ0aWFsID0gcGF0aC5wYXJlbnQgLyBmIntwYXRoLm5hbWV9LnBhcnRpYWwubnB6Ig0KICAgIG5wLnNhdmV6KHBhcnRpYWwsICoqYXJyYXlzKQ0KICAgIG9zLnJlcGxhY2UocGFydGlhbCwgcGF0aCkNCg0KDQpkZWYgYXRvbWljX25weShwYXRoOiBQYXRoLCBhcnJheTogbnAubmRhcnJheSkgLT4gTm9uZToNCiAgICBwYXRoID0gUGF0aChwYXRoKQ0KICAgIGVuc3VyZV9kaXIocGF0aC5wYXJlbnQpDQogICAgcGFydGlhbCA9IHBhdGgucGFyZW50IC8gZiJ7cGF0aC5uYW1lfS5wYXJ0aWFsLm5weSINCiAgICBucC5zYXZlKHBhcnRpYWwsIGFycmF5KQ0KICAgIG9zLnJlcGxhY2UocGFydGlhbCwgcGF0aCkNCg0KDQpkZWYgYXRvbWljX3BhcnF1ZXQocGF0aDogUGF0aCwgZnJhbWU6IEFueSkgLT4gTm9uZToNCiAgICBwYXRoID0gUGF0aChwYXRoKQ0KICAgIGVuc3VyZV9kaXIocGF0aC5wYXJlbnQpDQogICAgcGFydGlhbCA9IHBhdGgucGFyZW50IC8gZiJ7cGF0aC5uYW1lfS5wYXJ0aWFsLnBhcnF1ZXQiDQogICAgZnJhbWUudG9fcGFycXVldChwYXJ0aWFsLCBpbmRleD1GYWxzZSkNCiAgICBvcy5yZXBsYWNlKHBhcnRpYWwsIHBhdGgpDQo=", "commissioning/patchcore.py": "IiIiSW1hZ2Ugc2NvcmVzIGZyb20gYSBtZW1vcnkgYmFuay4NCg0KVGhlIG9wZXJhdGlvbmFsIHNjb3JlIGlzIHRoZSBtYXhpbXVtLCBvdmVyIHBhdGNoZXMgb2YgdGhlIHF1ZXJ5LCBvZiB0aGUgZGlzdGFuY2UNCnRvIHRoZSBuZWFyZXN0IHBhdGNoIGluIHRoZSBiYW5rLiBCYW5rcyBhcmUgcHJlZml4ZXMgb2Ygb25lIHRyYWplY3RvcnksIHNvIHRoZQ0Kc2NvcmUgbmV2ZXIgaW5jcmVhc2VzIGFzIE4gZ3Jvd3MuDQoiIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KDQpkZWYgX2FzX2luZGV4KGluZGljZXM6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0pIC0+IG5wLm5kYXJyYXk6DQogICAgYXJyYXkgPSBucC5hc2FycmF5KGluZGljZXMsIGR0eXBlPWludCkNCiAgICBpZiBhcnJheS5uZGltICE9IDE6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkV4cGVjdGVkIGEgMS1EIGluZGV4LCBnb3Qgc2hhcGUge2FycmF5LnNoYXBlfSIpDQogICAgcmV0dXJuIGFycmF5DQoNCg0KZGVmIHNjb3Jlc19mb3JfYmFuayhkaXN0YW5jZTogbnAubmRhcnJheSwgcXVlcnlfaW5kaWNlczogbnAubmRhcnJheSB8IGxpc3RbaW50XSwgYmFua19pbmRpY2VzOiBucC5uZGFycmF5IHwgbGlzdFtpbnRdKSAtPiBucC5uZGFycmF5Og0KICAgIHF1ZXJpZXMgPSBfYXNfaW5kZXgocXVlcnlfaW5kaWNlcykNCiAgICBiYW5rID0gX2FzX2luZGV4KGJhbmtfaW5kaWNlcykNCiAgICBpZiBiYW5rLnNpemUgPT0gMDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJFbXB0eSBtZW1vcnkgYmFuayIpDQogICAgaWYgbGVuKG5wLnVuaXF1ZShiYW5rKSkgIT0gbGVuKGJhbmspOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk1lbW9yeSBiYW5rIGNvbnRhaW5zIGR1cGxpY2F0ZSBpbWFnZXMiKQ0KICAgIGJsb2NrID0gZGlzdGFuY2VbcXVlcmllc11bOiwgOiwgYmFua10NCiAgICBzY29yZXMgPSBibG9jay5taW4oYXhpcz0tMSkubWF4KGF4aXM9LTEpDQogICAgaWYgbm90IG5wLmlzZmluaXRlKHNjb3JlcykuYWxsKCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTm9uLWZpbml0ZSBpbWFnZSBzY29yZSIpDQogICAgcmV0dXJuIHNjb3Jlcy5hc3R5cGUobnAuZmxvYXQ2NCkNCg0KDQpkZWYgc2NvcmVzX2Fsb25nX29yZGVyKA0KICAgIGRpc3RhbmNlOiBucC5uZGFycmF5LA0KICAgIHF1ZXJ5X2luZGljZXM6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0sDQogICAgb3JkZXI6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0sDQogICAgZ3JpZDogbGlzdFtpbnRdLA0KKSAtPiBucC5uZGFycmF5Og0KICAgICIiIlNjb3JlcyBhdCBlYWNoIG5lc3RlZCBwcmVmaXggb2YgYGBvcmRlcmBgLiBSZXR1cm5zIChsZW4oZ3JpZCksIG5fcXVlcmllcykuIiIiDQogICAgcXVlcmllcyA9IF9hc19pbmRleChxdWVyeV9pbmRpY2VzKQ0KICAgIG9yZGVyID0gX2FzX2luZGV4KG9yZGVyKQ0KICAgIGlmIGxlbihucC51bmlxdWUob3JkZXIpKSAhPSBsZW4ob3JkZXIpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRyYWplY3Rvcnkgb3JkZXIgY29udGFpbnMgZHVwbGljYXRlIGltYWdlcyIpDQogICAgZ3JpZCA9IFtpbnQobikgZm9yIG4gaW4gZ3JpZF0NCiAgICBpZiBncmlkICE9IHNvcnRlZChncmlkKSBvciBhbnkobiA8IDEgb3IgbiA+IGxlbihvcmRlcikgZm9yIG4gaW4gZ3JpZCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkdyaWQge2dyaWR9IGlzIG5vdCBhIHZhbGlkIGluY3JlYXNpbmcgc3Vic2V0IG9mIDEuLntsZW4ob3JkZXIpfSIpDQogICAgc2VsZWN0ZWQgPSBkaXN0YW5jZVtxdWVyaWVzXQ0KICAgIHJ1bm5pbmcgPSBOb25lDQogICAgcHJldmlvdXMgPSAwDQogICAgcm93cyA9IFtdDQogICAgZm9yIG4gaW4gZ3JpZDoNCiAgICAgICAgYWRkZWQgPSBvcmRlcltwcmV2aW91czpuXQ0KICAgICAgICBibG9ja19taW4gPSBzZWxlY3RlZFs6LCA6LCBhZGRlZF0ubWluKGF4aXM9LTEpDQogICAgICAgIHJ1bm5pbmcgPSBibG9ja19taW4gaWYgcnVubmluZyBpcyBOb25lIGVsc2UgbnAubWluaW11bShydW5uaW5nLCBibG9ja19taW4pDQogICAgICAgIHNjb3JlcyA9IHJ1bm5pbmcubWF4KGF4aXM9LTEpDQogICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShzY29yZXMpLmFsbCgpOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJOb24tZmluaXRlIGltYWdlIHNjb3JlIikNCiAgICAgICAgcm93cy5hcHBlbmQoc2NvcmVzKQ0KICAgICAgICBwcmV2aW91cyA9IG4NCiAgICBzdGFja2VkID0gbnAuc3RhY2socm93cywgYXhpcz0wKS5hc3R5cGUobnAuZmxvYXQ2NCkNCiAgICBpZiBzdGFja2VkLnNoYXBlWzBdID4gMSBhbmQgbnAuYW55KG5wLmRpZmYoc3RhY2tlZCwgYXhpcz0wKSA+IDFlLTUpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkltYWdlIHNjb3JlIGluY3JlYXNlZCBhcyB0aGUgbmVzdGVkIGJhbmsgZ3JldyIpDQogICAgcmV0dXJuIHN0YWNrZWQNCg0KDQpkZWYga25uX3Njb3JlcyhxdWVyeV9wYXRjaGVzOiBucC5uZGFycmF5LCBiYW5rX3BhdGNoZXM6IG5wLm5kYXJyYXksIHF1ZXJ5X2NodW5rOiBpbnQgPSA4KSAtPiBucC5uZGFycmF5Og0KICAgICIiIlJlZmVyZW5jZSAxLU5OIG1heGltdW0uIGBgYmFua19wYXRjaGVzYGAgaXMgKE5fYmFuaywgUCwgQykgb3IgKE5fcGF0Y2hlcywgQykuIiIiDQogICAgaWYgYmFua19wYXRjaGVzLm5kaW0gPT0gMzoNCiAgICAgICAgYmFuayA9IG5wLmFzY29udGlndW91c2FycmF5KGJhbmtfcGF0Y2hlcy5yZXNoYXBlKC0xLCBiYW5rX3BhdGNoZXMuc2hhcGVbLTFdKSwgZHR5cGU9bnAuZmxvYXQzMikNCiAgICBlbGlmIGJhbmtfcGF0Y2hlcy5uZGltID09IDI6DQogICAgICAgIGJhbmsgPSBucC5hc2NvbnRpZ3VvdXNhcnJheShiYW5rX3BhdGNoZXMsIGR0eXBlPW5wLmZsb2F0MzIpDQogICAgZWxzZToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiQmFkIGJhbmsgc2hhcGUge2JhbmtfcGF0Y2hlcy5zaGFwZX0iKQ0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IHRvcmNoDQogICAgZXhjZXB0IEltcG9ydEVycm9yOg0KICAgICAgICB0b3JjaCA9IE5vbmUNCiAgICBpZiB0b3JjaCBpcyBOb25lIGFuZCBiYW5rLnNoYXBlWzBdID4gODE5MjoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJEaXJlY3Qga05OIG9uIGEgYmFuayBvZiB0aGlzIHNpemUgbmVlZHMgdG9yY2guIikNCiAgICBuX3F1ZXJ5LCBuX3BhdGNoZXMsIGNoYW5uZWxzID0gcXVlcnlfcGF0Y2hlcy5zaGFwZQ0KICAgIHNjb3JlcyA9IG5wLmVtcHR5KG5fcXVlcnksIGR0eXBlPW5wLmZsb2F0NjQpDQogICAgZGV2aWNlID0gImNwdSINCiAgICBiYW5rX3QgPSBOb25lDQogICAgaWYgdG9yY2ggaXMgbm90IE5vbmU6DQogICAgICAgIGRldmljZSA9ICJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSINCiAgICAgICAgYmFua190ID0gdG9yY2guZnJvbV9udW1weShiYW5rKS50byhkZXZpY2UpDQogICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIG5fcXVlcnksIHF1ZXJ5X2NodW5rKToNCiAgICAgICAgc3RvcCA9IG1pbihzdGFydCArIHF1ZXJ5X2NodW5rLCBuX3F1ZXJ5KQ0KICAgICAgICBmbGF0ID0gbnAuYXNjb250aWd1b3VzYXJyYXkocXVlcnlfcGF0Y2hlc1tzdGFydDpzdG9wXS5yZXNoYXBlKC0xLCBjaGFubmVscyksIGR0eXBlPW5wLmZsb2F0MzIpDQogICAgICAgIGlmIGJhbmtfdCBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIHF1ZXJ5X3QgPSB0b3JjaC5mcm9tX251bXB5KGZsYXQpLnRvKGRldmljZSkNCiAgICAgICAgICAgIG5lYXJlc3QgPSBbXQ0KICAgICAgICAgICAgc3RlcCA9IDI1Ng0KICAgICAgICAgICAgZm9yIHAwIGluIHJhbmdlKDAsIGZsYXQuc2hhcGVbMF0sIHN0ZXApOg0KICAgICAgICAgICAgICAgIGRpc3RhbmNlcyA9IHRvcmNoLmNkaXN0KHF1ZXJ5X3RbcDAgOiBwMCArIHN0ZXBdLCBiYW5rX3QpDQogICAgICAgICAgICAgICAgbmVhcmVzdC5hcHBlbmQoZGlzdGFuY2VzLm1pbihkaW09MSkudmFsdWVzLmRldGFjaCgpLmNwdSgpLm51bXB5KCkpDQogICAgICAgICAgICBwYXRjaF9zY29yZXMgPSBucC5jb25jYXRlbmF0ZShuZWFyZXN0KS5yZXNoYXBlKHN0b3AgLSBzdGFydCwgbl9wYXRjaGVzKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgbmVhcmVzdCA9IG5wLmVtcHR5KGZsYXQuc2hhcGVbMF0sIGR0eXBlPW5wLmZsb2F0MzIpDQogICAgICAgICAgICBmb3IgcDAgaW4gcmFuZ2UoMCwgZmxhdC5zaGFwZVswXSwgMzIpOg0KICAgICAgICAgICAgICAgIHAxID0gbWluKHAwICsgMzIsIGZsYXQuc2hhcGVbMF0pDQogICAgICAgICAgICAgICAgZGVsdGEgPSBmbGF0W3AwOnAxLCBOb25lLCA6XSAtIGJhbmtbTm9uZSwgOiwgOl0NCiAgICAgICAgICAgICAgICBuZWFyZXN0W3AwOnAxXSA9IG5wLnNxcnQobnAuc3VtKGRlbHRhICogZGVsdGEsIGF4aXM9LTEpKS5taW4oYXhpcz0xKQ0KICAgICAgICAgICAgcGF0Y2hfc2NvcmVzID0gbmVhcmVzdC5yZXNoYXBlKHN0b3AgLSBzdGFydCwgbl9wYXRjaGVzKQ0KICAgICAgICBzY29yZXNbc3RhcnQ6c3RvcF0gPSBwYXRjaF9zY29yZXMubWF4KGF4aXM9MSkNCiAgICBpZiBub3QgbnAuaXNmaW5pdGUoc2NvcmVzKS5hbGwoKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJOb24tZmluaXRlIHJlZmVyZW5jZSBrTk4gc2NvcmUiKQ0KICAgIHJldHVybiBzY29yZXMNCg0KDQpkZWYgYW5vbWFseV9tYXAocGF0Y2hfc2NvcmVzOiBucC5uZGFycmF5LCBpbWFnZV9zaXplOiBpbnQgPSAyMjQsIHNpZ21hOiBmbG9hdCA9IDQuMCkgLT4gbnAubmRhcnJheToNCiAgICAiIiJVcHNhbXBsZSBhIDI4eDI4IHBhdGNoLXNjb3JlIGdyaWQgYW5kIGJsdXIgaXQgZm9yIGEgZmlndXJlIG92ZXJsYXkuIiIiDQogICAgZnJvbSBzY2lweS5uZGltYWdlIGltcG9ydCBnYXVzc2lhbl9maWx0ZXINCg0KICAgIGdyaWQgPSBucC5hc2FycmF5KHBhdGNoX3Njb3JlcywgZHR5cGU9bnAuZmxvYXQ2NCkucmVzaGFwZSgyOCwgMjgpDQogICAgem9vbSA9IGltYWdlX3NpemUgLy8gMjgNCiAgICBpZiBpbWFnZV9zaXplICUgMjggIT0gMDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJpbWFnZV9zaXplIG11c3QgYmUgYSBtdWx0aXBsZSBvZiAyOCIpDQogICAgdXBzYW1wbGVkID0gbnAucmVwZWF0KG5wLnJlcGVhdChncmlkLCB6b29tLCBheGlzPTApLCB6b29tLCBheGlzPTEpDQogICAgcmV0dXJuIGdhdXNzaWFuX2ZpbHRlcih1cHNhbXBsZWQsIHNpZ21hPXNpZ21hKQ0KDQoNCmRlZiBwYXRjaF9zY29yZXNfZm9yX2JhbmsoZGlzdGFuY2Vfcm93OiBucC5uZGFycmF5LCBiYW5rX2luZGljZXM6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0pIC0+IG5wLm5kYXJyYXk6DQogICAgIiIiZGlzdGFuY2Vfcm93OiAoNzg0LCBuX3RyYWluKSBmb3Igb25lIHF1ZXJ5LiIiIg0KICAgIGJhbmsgPSBfYXNfaW5kZXgoYmFua19pbmRpY2VzKQ0KICAgIHJldHVybiBkaXN0YW5jZV9yb3dbOiwgYmFua10ubWluKGF4aXM9MSkNCg==", "commissioning/report.py": "...", "commissioning/run.py": "IiIiUnVuIHRoZSAxNSB4IDIwIHggMiBuZXN0ZWQgdHJhamVjdG9yaWVzIGFuZCByZXN1bWUgZnJvbSBmaW5pc2hlZCBwYXJ0cy4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbG9nZ2luZw0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBwYW5kYXMgYXMgcGQNCg0KZnJvbSBjb21taXNzaW9uaW5nLmNhbGlicmF0aW9uIGltcG9ydCBldmFsdWF0ZV9zY29yZXMNCmZyb20gY29tbWlzc2lvbmluZy5jb25maWcgaW1wb3J0IENvbmZpZw0KZnJvbSBjb21taXNzaW9uaW5nLmRhdGEgaW1wb3J0IGxvYWRfc3BsaXQsIHNwbGl0X3NlZWRzDQpmcm9tIGNvbW1pc3Npb25pbmcuZGlzdGFuY2VzIGltcG9ydCBsb2FkX2Rpc3RhbmNlX2NhY2hlDQpmcm9tIGNvbW1pc3Npb25pbmcuZTAgaW1wb3J0IGdhdGVfcGF0aA0KZnJvbSBjb21taXNzaW9uaW5nLmZlYXR1cmVzIGltcG9ydCBsb2FkX2dhcF90YWJsZQ0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCBhdG9taWNfcGFycXVldCwgYXRvbWljX3NhdmV6LCByZWFkX2pzb24NCmZyb20gY29tbWlzc2lvbmluZy5wYXRjaGNvcmUgaW1wb3J0IHNjb3Jlc19hbG9uZ19vcmRlcg0KZnJvbSBjb21taXNzaW9uaW5nLnNlbGVjdGlvbiBpbXBvcnQgZ3JpZF9mb3JfcG9vbCwga2NlbnRlcl9vcmRlciwgcmFuZG9tX29yZGVyDQoNCkxPR0dFUiA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQ0KDQoNCmRlZiB0cmFqZWN0b3J5X3BhdGhzKGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIHJlcDogaW50LCBzdHJhdGVneTogc3RyKSAtPiB0dXBsZToNCiAgICBzdGVtID0gZiJ7Y2xhc3NfbmFtZX1fcmVwe3JlcDowMmR9X3tzdHJhdGVneX0iDQogICAgdGFibGUgPSBjZmcud29ya19yb290IC8gInJ1bnMiIC8gInBhcnRzIiAvIGYie3N0ZW19LnBhcnF1ZXQiDQogICAgc2NvcmVzID0gY2ZnLndvcmtfcm9vdCAvICJydW5zIiAvICJzY29yZXMiIC8gZiJ7c3RlbX0ubnB6Ig0KICAgIHJldHVybiB0YWJsZSwgc2NvcmVzDQoNCg0KZGVmIGdhdGVfaXNfZXhwbG9yYXRvcnkoY2ZnOiBDb25maWcsIGNsYXNzZXM6IGxpc3Rbc3RyXSwgZXhwbG9yYXRvcnk6IGJvb2wpIC0+IGJvb2w6DQogICAgcGF0aCA9IGdhdGVfcGF0aChjZmcpDQogICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOg0KICAgICAgICBpZiBleHBsb3JhdG9yeToNCiAgICAgICAgICAgIExPR0dFUi53YXJuaW5nKCJFMCBnYXRlIGlzIG1pc3NpbmcuIFRoaXMgcnVuIGlzIGV4cGxvcmF0b3J5IGFuZCBjYW5ub3QgYmUgcmVwb3J0ZWQgYXMgY29uZmlybWF0b3J5LiIpDQogICAgICAgICAgICByZXR1cm4gVHJ1ZQ0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkUwIGdhdGUgaXMgbWlzc2luZy4gUlEgdHJhamVjdG9yaWVzIHN0YXkgYmxvY2tlZCB1bnRpbCBlMC9nYXRlLmpzb24gaGFzIHN0YXR1cyBQQVNTLiIpDQogICAgZ2F0ZSA9IHJlYWRfanNvbihwYXRoKQ0KICAgIGlmIGdhdGUuZ2V0KCJjb25maWdfaGFzaCIpICE9IGNmZy5oYXNoOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkUwIGdhdGUgd2FzIHdyaXR0ZW4gZm9yIGEgZGlmZmVyZW50IGNvbmZpZyBoYXNoLiIpDQogICAgY292ZXJlZCA9IHNldChnYXRlLmdldCgiY2xhc3NlcyIsIFtdKSkNCiAgICBtaXNzaW5nID0gW25hbWUgZm9yIG5hbWUgaW4gY2xhc3NlcyBpZiBuYW1lIG5vdCBpbiBjb3ZlcmVkXQ0KICAgIHBhc3NlZCA9IGdhdGUuZ2V0KCJzdGF0dXMiKSA9PSAiUEFTUyIgYW5kIGdhdGUuZ2V0KCJwYXJ0IikgPT0gImFsbCIgYW5kIG5vdCBtaXNzaW5nDQogICAgaWYgcGFzc2VkOg0KICAgICAgICByZXR1cm4gRmFsc2UNCiAgICBpZiBleHBsb3JhdG9yeToNCiAgICAgICAgTE9HR0VSLndhcm5pbmcoIkUwIGdhdGUgZG9lcyBub3QgY292ZXIgdGhpcyBydW4gKCVzKS4gTWFya2luZyBpdCBleHBsb3JhdG9yeS4iLCBnYXRlLmdldCgic3RhdHVzIikpDQogICAgICAgIHJldHVybiBUcnVlDQogICAgaWYgbWlzc2luZzoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiRTAgZ2F0ZSBkb2VzIG5vdCBjb3Zlcjoge21pc3Npbmd9LiBSdW4gZTAgb24gdGhlc2UgY2xhc3NlcyBiZWZvcmUgUlEuIikNCiAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJFMCBnYXRlIHN0YXR1cyBpcyB7Z2F0ZS5nZXQoJ3N0YXR1cycpfS4gUlEgdHJhamVjdG9yaWVzIHN0YXkgYmxvY2tlZC4iKQ0KDQoNCmRlZiBfZmluaXNoZWQoY2ZnOiBDb25maWcsIHRhYmxlX3BhdGgsIHNjb3Jlc19wYXRoKSAtPiBib29sOg0KICAgIGlmIG5vdCB0YWJsZV9wYXRoLmlzX2ZpbGUoKSBvciBub3Qgc2NvcmVzX3BhdGguaXNfZmlsZSgpOg0KICAgICAgICByZXR1cm4gRmFsc2UNCiAgICBmcmFtZSA9IHBkLnJlYWRfcGFycXVldCh0YWJsZV9wYXRoKQ0KICAgIGhhc2hlcyA9IHNldChmcmFtZVsiY29uZmlnX2hhc2giXS5hc3R5cGUoc3RyKSkNCiAgICBpZiBoYXNoZXMgIT0ge2NmZy5oYXNofToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiUmVmdXNpbmcgdG8gcmV1c2UgYSB0cmFqZWN0b3J5IGZyb20gYW5vdGhlciBjb25maWc6IHt0YWJsZV9wYXRofSIpDQogICAgcmV0dXJuIFRydWUNCg0KDQpkZWYgX2luZGljZXMocGF0aHM6IGxpc3Rbc3RyXSwgc2VsZWN0ZWQ6IGxpc3Rbc3RyXSwgY2xhc3NfbmFtZTogc3RyKSAtPiBsaXN0W2ludF06DQogICAgaW5kZXggPSB7cGF0aDogcG9zaXRpb24gZm9yIHBvc2l0aW9uLCBwYXRoIGluIGVudW1lcmF0ZShwYXRocyl9DQogICAgdHJ5Og0KICAgICAgICByZXR1cm4gW2luZGV4W3BhdGhdIGZvciBwYXRoIGluIHNlbGVjdGVkXQ0KICAgIGV4Y2VwdCBLZXlFcnJvciBhcyBleGM6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogc3BsaXQgcGF0aCBpcyBub3QgaW4gdGhlIGZlYXR1cmUgY2FjaGU6IHtleGN9IikgZnJvbSBleGMNCg0KDQpkZWYgcnVuX29uZSgNCiAgICBjZmc6IENvbmZpZywNCiAgICBjbGFzc19uYW1lOiBzdHIsDQogICAgcmVwOiBpbnQsDQogICAgc3RyYXRlZ3k6IHN0ciwNCiAgICBleHBsb3JhdG9yeTogYm9vbCwNCiAgICBwYXRoczogbGlzdFtzdHJdLA0KICAgIGRpc3RhbmNlOiBucC5uZGFycmF5LA0KICAgIGdhcDogbnAubmRhcnJheSwNCikgLT4gTm9uZToNCiAgICB0YWJsZV9wYXRoLCBzY29yZXNfcGF0aCA9IHRyYWplY3RvcnlfcGF0aHMoY2ZnLCBjbGFzc19uYW1lLCByZXAsIHN0cmF0ZWd5KQ0KICAgIGlmIF9maW5pc2hlZChjZmcsIHRhYmxlX3BhdGgsIHNjb3Jlc19wYXRoKToNCiAgICAgICAgTE9HR0VSLmluZm8oIlNraXBwaW5nIGZpbmlzaGVkICVzIHJlcCAlMDJkICVzIiwgY2xhc3NfbmFtZSwgcmVwLCBzdHJhdGVneSkNCiAgICAgICAgcmV0dXJuDQogICAgc3BsaXQgPSBsb2FkX3NwbGl0KGNmZywgY2xhc3NfbmFtZSwgcmVwKQ0KICAgIG5fdHJhaW4gPSBpbnQoc3BsaXRbIm5fdHJhaW4iXSkNCiAgICBjYWxpYnJhdGlvbiA9IF9pbmRpY2VzKHBhdGhzLCBzcGxpdFsiY2FsaWJyYXRpb24iXSwgY2xhc3NfbmFtZSkNCiAgICBwb29sID0gX2luZGljZXMocGF0aHMsIHNwbGl0WyJwb29sIl0sIGNsYXNzX25hbWUpDQogICAgdGVzdF9wYXRocyA9IGxpc3Qoc3BsaXRbInRlc3RfZ29vZCJdKSArIFtpdGVtWyJwYXRoIl0gZm9yIGl0ZW0gaW4gc3BsaXRbInRlc3RfZGVmZWN0Il1dDQogICAgdGVzdCA9IF9pbmRpY2VzKHBhdGhzLCB0ZXN0X3BhdGhzLCBjbGFzc19uYW1lKQ0KICAgIGlmIHNldChjYWxpYnJhdGlvbikgJiBzZXQocG9vbCkgb3IgbWF4KGNhbGlicmF0aW9uICsgcG9vbCkgPj0gbl90cmFpbiBvciBtaW4odGVzdCkgPCBuX3RyYWluOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IEMvUC90ZXN0IGluZGljZXMgYXJlIG5vdCBhIHZhbGlkIHBhcnRpdGlvbi4iKQ0KICAgIF9zcGxpdF9ybmcsIHJhbmRvbV9ybmcsIGtjZW50ZXJfcm5nID0gc3BsaXRfc2VlZHMoY2ZnLCBjbGFzc19uYW1lLCByZXApDQogICAgaWYgc3RyYXRlZ3kgPT0gInJhbmRvbSI6DQogICAgICAgIG9yZGVyID0gcmFuZG9tX29yZGVyKHBvb2wsIHJhbmRvbV9ybmcpDQogICAgZWxpZiBzdHJhdGVneSA9PSAia2NlbnRlciI6DQogICAgICAgIG9yZGVyID0ga2NlbnRlcl9vcmRlcihnYXAsIHBvb2wsIGtjZW50ZXJfcm5nKQ0KICAgIGVsc2U6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlVua25vd24gc3RyYXRlZ3k6IHtzdHJhdGVneX0iKQ0KICAgIGdyaWQgPSBncmlkX2Zvcl9wb29sKGxlbihwb29sKSwgY2ZnLm5fZ3JpZCkNCiAgICBxdWVyaWVzID0gbnAuYXNhcnJheShjYWxpYnJhdGlvbiArIHRlc3QsIGR0eXBlPWludCkNCiAgICBzdGFja2VkID0gc2NvcmVzX2Fsb25nX29yZGVyKGRpc3RhbmNlLCBxdWVyaWVzLCBvcmRlciwgZ3JpZCkNCiAgICBjYWxpYnJhdGlvbl9zY29yZXMgPSBzdGFja2VkWzosIDogbGVuKGNhbGlicmF0aW9uKV0NCiAgICB0ZXN0X3Njb3JlcyA9IHN0YWNrZWRbOiwgbGVuKGNhbGlicmF0aW9uKSA6XQ0KICAgIGxhYmVscyA9IG5wLmFycmF5KFswXSAqIGxlbihzcGxpdFsidGVzdF9nb29kIl0pICsgWzFdICogbGVuKHNwbGl0WyJ0ZXN0X2RlZmVjdCJdKSwgZHR5cGU9aW50KQ0KICAgIHJvd3MgPSBbXQ0KICAgIGZvciByb3dfaW5kZXgsIG4gaW4gZW51bWVyYXRlKGdyaWQpOg0KICAgICAgICBtZXRyaWNzID0gZXZhbHVhdGVfc2NvcmVzKA0KICAgICAgICAgICAgdGVzdF9zY29yZXNbcm93X2luZGV4XSwNCiAgICAgICAgICAgIGxhYmVscywNCiAgICAgICAgICAgIGNhbGlicmF0aW9uX3Njb3Jlc1tyb3dfaW5kZXhdLA0KICAgICAgICAgICAgY2ZnLmFscGhhLA0KICAgICAgICAgICAgY2ZnLnByYWN0aWNhbF90cHIsDQogICAgICAgICAgICBjZmcucHJhY3RpY2FsX2ZwciwNCiAgICAgICAgKQ0KICAgICAgICByb3dzLmFwcGVuZCgNCiAgICAgICAgICAgIHsNCiAgICAgICAgICAgICAgICAiY2xhc3NfbmFtZSI6IGNsYXNzX25hbWUsDQogICAgICAgICAgICAgICAgImdyb3VwIjogY2ZnLmdyb3VwX29mKGNsYXNzX25hbWUpLA0KICAgICAgICAgICAgICAgICJyZXAiOiBpbnQocmVwKSwNCiAgICAgICAgICAgICAgICAic3RyYXRlZ3kiOiBzdHJhdGVneSwNCiAgICAgICAgICAgICAgICAiTiI6IGludChuKSwNCiAgICAgICAgICAgICAgICAicG9vbF9zaXplIjogaW50KGxlbihwb29sKSksDQogICAgICAgICAgICAgICAgImNvbmZpZ19oYXNoIjogY2ZnLmhhc2gsDQogICAgICAgICAgICAgICAgImV4cGxvcmF0b3J5IjogYm9vbChleHBsb3JhdG9yeSksDQogICAgICAgICAgICAgICAgKiptZXRyaWNzLA0KICAgICAgICAgICAgfQ0KICAgICAgICApDQogICAgYXRvbWljX3BhcnF1ZXQodGFibGVfcGF0aCwgcGQuRGF0YUZyYW1lKHJvd3MpKQ0KICAgIGF0b21pY19zYXZleigNCiAgICAgICAgc2NvcmVzX3BhdGgsDQogICAgICAgIGdyaWQ9bnAuYXNhcnJheShncmlkLCBkdHlwZT1ucC5pbnQzMiksDQogICAgICAgIHNjb3Jlc190ZXN0PXRlc3Rfc2NvcmVzLmFzdHlwZShucC5mbG9hdDMyKSwNCiAgICAgICAgc2NvcmVzX2NhbGlicmF0aW9uPWNhbGlicmF0aW9uX3Njb3Jlcy5hc3R5cGUobnAuZmxvYXQzMiksDQogICAgICAgIHRlc3RfbGFiZWxzPWxhYmVscy5hc3R5cGUobnAuaW50OCksDQogICAgICAgIG9yZGVyPW9yZGVyLmFzdHlwZShucC5pbnQzMiksDQogICAgICAgIHRlc3RfcGF0aHM9bnAuYXNhcnJheSh0ZXN0X3BhdGhzKSwNCiAgICAgICAgY2FsaWJyYXRpb25fcGF0aHM9bnAuYXNhcnJheShzcGxpdFsiY2FsaWJyYXRpb24iXSksDQogICAgKQ0KICAgIExPR0dFUi5pbmZvKCJGaW5pc2hlZCAlcyByZXAgJTAyZCAlcyAoJWQgZ3JpZCBwb2ludHMpIiwgY2xhc3NfbmFtZSwgcmVwLCBzdHJhdGVneSwgbGVuKGdyaWQpKQ0KDQoNCmRlZiBydW4oY2ZnOiBDb25maWcsIGNsYXNzZXM6IGxpc3Rbc3RyXSwgZXhwbG9yYXRvcnk6IGJvb2wgPSBGYWxzZSkgLT4gTm9uZToNCiAgICBtYXJrID0gZ2F0ZV9pc19leHBsb3JhdG9yeShjZmcsIGNsYXNzZXMsIGV4cGxvcmF0b3J5KQ0KICAgIHRvdGFsID0gbGVuKGNsYXNzZXMpICogY2ZnLm5fcmVwZWF0cyAqIGxlbihjZmcuc3RyYXRlZ2llcykNCiAgICBkb25lID0gMA0KICAgIGZvciBjbGFzc19uYW1lIGluIGNsYXNzZXM6DQogICAgICAgIGRpc3RhbmNlX2NhY2hlID0gbG9hZF9kaXN0YW5jZV9jYWNoZShjZmcsIGNsYXNzX25hbWUpDQogICAgICAgIGdhcF90YWJsZSA9IGxvYWRfZ2FwX3RhYmxlKGNmZywgY2xhc3NfbmFtZSkNCiAgICAgICAgaWYgZ2FwX3RhYmxlWyJwYXRocyJdICE9IGxpc3QoZGlzdGFuY2VfY2FjaGVbIm1ldGEiXVsicGF0aHMiXSk6DQogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGZlYXR1cmUgYW5kIGRpc3RhbmNlIGNhY2hlcyBhcmUgbm90IGFsaWduZWQuIikNCiAgICAgICAgZm9yIHJlcCBpbiByYW5nZShjZmcubl9yZXBlYXRzKToNCiAgICAgICAgICAgIGZvciBzdHJhdGVneSBpbiBjZmcuc3RyYXRlZ2llczoNCiAgICAgICAgICAgICAgICBydW5fb25lKA0KICAgICAgICAgICAgICAgICAgICBjZmcsDQogICAgICAgICAgICAgICAgICAgIGNsYXNzX25hbWUsDQogICAgICAgICAgICAgICAgICAgIHJlcCwNCiAgICAgICAgICAgICAgICAgICAgc3RyYXRlZ3ksDQogICAgICAgICAgICAgICAgICAgIG1hcmssDQogICAgICAgICAgICAgICAgICAgIGdhcF90YWJsZVsicGF0aHMiXSwNCiAgICAgICAgICAgICAgICAgICAgZGlzdGFuY2VfY2FjaGVbIkQiXSwNCiAgICAgICAgICAgICAgICAgICAgZ2FwX3RhYmxlWyJnYXAiXSwNCiAgICAgICAgICAgICAgICApDQogICAgICAgICAgICAgICAgZG9uZSArPSAxDQogICAgICAgICAgICAgICAgTE9HR0VSLmluZm8oIlByb2dyZXNzICVkLyVkIiwgZG9uZSwgdG90YWwpDQo=", "commissioning/selection.py": "IiIiTmVzdGVkIG1lbW9yeS1iYW5rIHRyYWplY3Rvcmllcy4gQm90aCBzdHJhdGVnaWVzIHJldHVybiBhIHBlcm11dGF0aW9uIG9mIFAuIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQoNCg0KZGVmIGdyaWRfZm9yX3Bvb2wocG9vbF9zaXplOiBpbnQsIGJhc2VfZ3JpZDogbGlzdFtpbnRdKSAtPiBsaXN0W2ludF06DQogICAgaWYgcG9vbF9zaXplIDwgMToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJQb29sIGlzIGVtcHR5LiIpDQogICAgZ3JpZCA9IFtpbnQobikgZm9yIG4gaW4gYmFzZV9ncmlkIGlmIDEgPD0gaW50KG4pIDw9IHBvb2xfc2l6ZV0NCiAgICBpZiBwb29sX3NpemUgbm90IGluIGdyaWQ6DQogICAgICAgIGdyaWQuYXBwZW5kKGludChwb29sX3NpemUpKQ0KICAgIGlmIGdyaWQgIT0gc29ydGVkKHNldChncmlkKSk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkdyaWQgaXMgbm90IHN0cmljdGx5IGluY3JlYXNpbmc6IHtncmlkfSIpDQogICAgcmV0dXJuIGdyaWQNCg0KDQpkZWYgcmFuZG9tX29yZGVyKHBvb2w6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0sIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvcikgLT4gbnAubmRhcnJheToNCiAgICBvcmRlciA9IG5wLmFycmF5KHBvb2wsIGR0eXBlPWludCwgY29weT1UcnVlKQ0KICAgIGlmIG9yZGVyLnNpemUgPT0gMCBvciBsZW4obnAudW5pcXVlKG9yZGVyKSkgIT0gbGVuKG9yZGVyKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJSYW5kb20gdHJhamVjdG9yeSBleHBlY3RzIGEgbm9uLWVtcHR5IHBvb2wgd2l0aG91dCBkdXBsaWNhdGVzLiIpDQogICAgcm5nLnNodWZmbGUob3JkZXIpDQogICAgcmV0dXJuIG9yZGVyDQoNCg0KZGVmIGtjZW50ZXJfb3JkZXIoZmVhdHVyZXM6IG5wLm5kYXJyYXksIHBvb2w6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0sIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvcikgLT4gbnAubmRhcnJheToNCiAgICAiIiJGYXJ0aGVzdC1maXJzdCB0cmF2ZXJzYWwgb24gR0FQIGZlYXR1cmVzLiBUaGUgZmlyc3QgaW1hZ2UgaXMgcmFuZG9tLg0KDQogICAgVGhpcyBpcyBpbWFnZS1sZXZlbCBkaXZlcnNpdHkgc2VsZWN0aW9uLiBJdCBpcyBub3QgY29yZXNldCBzZWxlY3Rpb24gb24gUGF0Y2hDb3JlIHBhdGNoZXMuDQogICAgIiIiDQogICAgcG9vbCA9IG5wLmFycmF5KHBvb2wsIGR0eXBlPWludCwgY29weT1UcnVlKQ0KICAgIGlmIHBvb2wuc2l6ZSA9PSAwIG9yIGxlbihucC51bmlxdWUocG9vbCkpICE9IGxlbihwb29sKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJrLWNlbnRlciB0cmFqZWN0b3J5IGV4cGVjdHMgYSBub24tZW1wdHkgcG9vbCB3aXRob3V0IGR1cGxpY2F0ZXMuIikNCiAgICBpZiBucC5hbnkocG9vbCA8IDApIG9yIG5wLmFueShwb29sID49IGxlbihmZWF0dXJlcykpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlBvb2wgaW5kZXggaXMgb3V0c2lkZSB0aGUgZmVhdHVyZSB0YWJsZS4iKQ0KICAgIGZpcnN0ID0gaW50KHJuZy5pbnRlZ2VycygwLCBsZW4ocG9vbCkpKQ0KICAgIHJlbWFpbmluZyA9IG5wLm9uZXMobGVuKHBvb2wpLCBkdHlwZT1ib29sKQ0KICAgIHJlbWFpbmluZ1tmaXJzdF0gPSBGYWxzZQ0KICAgIHNlbGVjdGVkID0gW2ZpcnN0XQ0KICAgIGNob3NlbiA9IGZlYXR1cmVzW3Bvb2xbZmlyc3RdXQ0KICAgIG1pbl9kaXN0ID0gbnAubGluYWxnLm5vcm0oZmVhdHVyZXNbcG9vbF0gLSBjaG9zZW4sIGF4aXM9MSkNCiAgICB3aGlsZSBib29sKHJlbWFpbmluZy5hbnkoKSk6DQogICAgICAgIG1hc2tlZCA9IG5wLndoZXJlKHJlbWFpbmluZywgbWluX2Rpc3QsIC1ucC5pbmYpDQogICAgICAgIG54dCA9IGludChucC5hcmdtYXgobWFza2VkKSkNCiAgICAgICAgcmVtYWluaW5nW254dF0gPSBGYWxzZQ0KICAgICAgICBzZWxlY3RlZC5hcHBlbmQobnh0KQ0KICAgICAgICB1cGRhdGVkID0gbnAubGluYWxnLm5vcm0oZmVhdHVyZXNbcG9vbF0gLSBmZWF0dXJlc1twb29sW254dF1dLCBheGlzPTEpDQogICAgICAgIG1pbl9kaXN0ID0gbnAubWluaW11bShtaW5fZGlzdCwgdXBkYXRlZCkNCiAgICBvcmRlciA9IHBvb2xbbnAuYXJyYXkoc2VsZWN0ZWQsIGR0eXBlPWludCldDQogICAgaWYgbGVuKG5wLnVuaXF1ZShvcmRlcikpICE9IGxlbihwb29sKSBvciBzZXQob3JkZXIudG9saXN0KCkpICE9IHNldChwb29sLnRvbGlzdCgpKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJrLWNlbnRlciBkaWQgbm90IHJldHVybiBhIHBlcm11dGF0aW9uIG9mIHRoZSBwb29sLiIpDQogICAgcmV0dXJuIG9yZGVyDQo=", "commissioning/stats.py": "IiIiUHJpbWFyeSBwYWlyZWQgY29tcGFyaXNvbnM6IGV4YWN0IFdpbGNveG9uLCBIb2xtLCBzaWduLWZsaXAsIGFuZCBhIGNsYXNzLWxldmVsIGJvb3RzdHJhcC4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBwYW5kYXMgYXMgcGQNCmZyb20gc2NpcHkuc3RhdHMgaW1wb3J0IHdpbGNveG9uDQoNCg0KZGVmIGhvbG0ocHZhbHVlczogbGlzdFtmbG9hdF0pIC0+IGxpc3RbZmxvYXRdOg0KICAgIHAgPSBucC5hc2FycmF5KHB2YWx1ZXMsIGR0eXBlPWZsb2F0KQ0KICAgIG9yZGVyID0gbnAuYXJnc29ydChwLCBraW5kPSJtZXJnZXNvcnQiKQ0KICAgIGFkanVzdGVkID0gbnAuZW1wdHkobGVuKHApLCBkdHlwZT1mbG9hdCkNCiAgICBydW5uaW5nID0gMC4wDQogICAgZm9yIHJhbmssIGluZGV4IGluIGVudW1lcmF0ZShvcmRlcik6DQogICAgICAgIHJ1bm5pbmcgPSBtYXgocnVubmluZywgbWluKDEuMCwgKGxlbihwKSAtIHJhbmspICogZmxvYXQocFtpbmRleF0pKSkNCiAgICAgICAgYWRqdXN0ZWRbaW5kZXhdID0gcnVubmluZw0KICAgIHJldHVybiBhZGp1c3RlZC50b2xpc3QoKQ0KDQoNCmRlZiBwYWlyZWRfZGlmZmVyZW5jZShzdW1tYXJ5OiBwZC5EYXRhRnJhbWUsIG1ldHJpYzogc3RyKSAtPiBwZC5TZXJpZXM6DQogICAgd2lkZSA9IHN1bW1hcnkucGl2b3QoaW5kZXg9ImNsYXNzX25hbWUiLCBjb2x1bW5zPSJzdHJhdGVneSIsIHZhbHVlcz1tZXRyaWMpDQogICAgaWYgImtjZW50ZXIiIG5vdCBpbiB3aWRlLmNvbHVtbnMgb3IgInJhbmRvbSIgbm90IGluIHdpZGUuY29sdW1uczoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiQm90aCBzdHJhdGVnaWVzIGFyZSByZXF1aXJlZCB0byBjb21wYXJlIHttZXRyaWN9LiIpDQogICAgZGlmZmVyZW5jZSA9ICh3aWRlWyJrY2VudGVyIl0gLSB3aWRlWyJyYW5kb20iXSkuYXN0eXBlKGZsb2F0KQ0KICAgIHJldHVybiBkaWZmZXJlbmNlLnNvcnRfaW5kZXgoKQ0KDQoNCmRlZiB3aWxjb3hvbl9wdmFsdWUoZGlmZmVyZW5jZTogbnAubmRhcnJheSkgLT4gZmxvYXQ6DQogICAgdmFsdWVzID0gbnAuYXNhcnJheShkaWZmZXJlbmNlLCBkdHlwZT1mbG9hdCkNCiAgICB2YWx1ZXMgPSB2YWx1ZXNbbnAuaXNmaW5pdGUodmFsdWVzKV0NCiAgICBpZiB2YWx1ZXMuc2l6ZSA9PSAwIG9yIG5wLmFsbGNsb3NlKHZhbHVlcywgMC4wKToNCiAgICAgICAgcmV0dXJuIDEuMA0KICAgIHRyeToNCiAgICAgICAgcmVzdWx0ID0gd2lsY294b24odmFsdWVzLCB6ZXJvX21ldGhvZD0id2lsY294IiwgYWx0ZXJuYXRpdmU9InR3by1zaWRlZCIsIG1ldGhvZD0iZXhhY3QiKQ0KICAgIGV4Y2VwdCBWYWx1ZUVycm9yOg0KICAgICAgICByZXN1bHQgPSB3aWxjb3hvbih2YWx1ZXMsIHplcm9fbWV0aG9kPSJ3aWxjb3giLCBhbHRlcm5hdGl2ZT0idHdvLXNpZGVkIiwgbWV0aG9kPSJhdXRvIikNCiAgICByZXR1cm4gZmxvYXQocmVzdWx0LnB2YWx1ZSkNCg0KDQpkZWYgc2lnbmZsaXBfcHZhbHVlKGRpZmZlcmVuY2U6IG5wLm5kYXJyYXkpIC0+IGZsb2F0Og0KICAgICIiIkV4YWN0IHR3by1zaWRlZCBwYWlyZWQgcGVybXV0YXRpb24gdGVzdCBvZiB0aGUgbWVhbiBkaWZmZXJlbmNlLiAyXm4gc2lnbiBmbGlwcy4iIiINCiAgICB2YWx1ZXMgPSBucC5hc2FycmF5KGRpZmZlcmVuY2UsIGR0eXBlPWZsb2F0KQ0KICAgIGlmIHZhbHVlcy5uZGltICE9IDEgb3Igbm90IG5wLmlzZmluaXRlKHZhbHVlcykuYWxsKCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiU2lnbi1mbGlwIGV4cGVjdHMgYSBmaW5pdGUgMS1EIHZlY3RvciBvZiBwYWlyZWQgZGlmZmVyZW5jZXMuIikNCiAgICBpZiB2YWx1ZXMuc2l6ZSA+IDIwOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkV4YWN0IHNpZ24tZmxpcCBpcyBsaW1pdGVkIHRvIDIwIG9yIGZld2VyIGNsYXNzZXMuIikNCiAgICBvYnNlcnZlZCA9IGFicyhmbG9hdChucC5tZWFuKHZhbHVlcykpKQ0KICAgIG4gPSBpbnQodmFsdWVzLnNpemUpDQogICAgdG90YWwgPSAxIDw8IG4NCiAgICBjb3VudCA9IDANCiAgICBjaHVuayA9IDQwOTYNCiAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCwgdG90YWwsIGNodW5rKToNCiAgICAgICAgbWFza3MgPSBucC5hcmFuZ2Uoc3RhcnQsIG1pbihzdGFydCArIGNodW5rLCB0b3RhbCksIGR0eXBlPW5wLnVpbnQzMikNCiAgICAgICAgc2lnbnMgPSBucC5vbmVzKChtYXNrcy5zaXplLCBuKSwgZHR5cGU9bnAuZmxvYXQ2NCkNCiAgICAgICAgZm9yIGJpdCBpbiByYW5nZShuKToNCiAgICAgICAgICAgIGZsaXAgPSAoKG1hc2tzID4+IG5wLnVpbnQzMihiaXQpKSAmIG5wLnVpbnQzMigxKSkuYXN0eXBlKGJvb2wpDQogICAgICAgICAgICBzaWduc1tmbGlwLCBiaXRdID0gLTEuMA0KICAgICAgICBtZWFucyA9IG5wLmFicyhzaWducyBAIHZhbHVlcyAvIG4pDQogICAgICAgIGNvdW50ICs9IGludChucC5zdW0obWVhbnMgKyAxZS0xMiA+PSBvYnNlcnZlZCkpDQogICAgcmV0dXJuIGNvdW50IC8gdG90YWwNCg0KDQpkZWYgYm9vdHN0cmFwX21lZGlhbl9pbnRlcnZhbChkaWZmZXJlbmNlOiBucC5uZGFycmF5LCBuX2Jvb3RzdHJhcDogaW50LCBzZWVkOiBpbnQpIC0+IGRpY3Q6DQogICAgdmFsdWVzID0gbnAuYXNhcnJheShkaWZmZXJlbmNlLCBkdHlwZT1mbG9hdCkNCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkNCiAgICBkcmF3cyA9IHJuZy5pbnRlZ2VycygwLCB2YWx1ZXMuc2l6ZSwgc2l6ZT0oaW50KG5fYm9vdHN0cmFwKSwgdmFsdWVzLnNpemUpKQ0KICAgIG1lZGlhbnMgPSBucC5tZWRpYW4odmFsdWVzW2RyYXdzXSwgYXhpcz0xKQ0KICAgIGxvdywgaGlnaCA9IG5wLnF1YW50aWxlKG1lZGlhbnMsIFswLjAyNSwgMC45NzVdKQ0KICAgIHJldHVybiB7DQogICAgICAgICJtZWRpYW4iOiBmbG9hdChucC5tZWRpYW4odmFsdWVzKSksDQogICAgICAgICJjaTk1IjogW2Zsb2F0KGxvdyksIGZsb2F0KGhpZ2gpXSwNCiAgICAgICAgIm5fYm9vdHN0cmFwIjogaW50KG5fYm9vdHN0cmFwKSwNCiAgICAgICAgInNlZWQiOiBpbnQoc2VlZCksDQogICAgfQ0KDQoNCmRlZiBfZGlyZWN0aW9uX2NvdW50cyhkaWZmZXJlbmNlOiBwZC5TZXJpZXMsIGxvd2VyX2lzX2JldHRlcjogYm9vbCkgLT4gZGljdDoNCiAgICB2YWx1ZXMgPSBkaWZmZXJlbmNlLnRvX251bXB5KGR0eXBlPWZsb2F0KQ0KICAgIGJldHRlciA9IHZhbHVlcyA8IDAgaWYgbG93ZXJfaXNfYmV0dGVyIGVsc2UgdmFsdWVzID4gMA0KICAgIHdvcnNlID0gdmFsdWVzID4gMCBpZiBsb3dlcl9pc19iZXR0ZXIgZWxzZSB2YWx1ZXMgPCAwDQogICAgcmV0dXJuIHsNCiAgICAgICAgIm5fa2NlbnRlcl9iZXR0ZXIiOiBpbnQobnAuc3VtKGJldHRlcikpLA0KICAgICAgICAibl9yYW5kb21fYmV0dGVyIjogaW50KG5wLnN1bSh3b3JzZSkpLA0KICAgICAgICAibl90aWVzIjogaW50KG5wLnN1bShucC5pc2Nsb3NlKHZhbHVlcywgMC4wKSkpLA0KICAgICAgICAicGVyX2NsYXNzIjoge3N0cihuYW1lKTogZmxvYXQodmFsdWUpIGZvciBuYW1lLCB2YWx1ZSBpbiBkaWZmZXJlbmNlLml0ZW1zKCl9LA0KICAgIH0NCg0KDQpkZWYgY29tcGFyZV9zdHJhdGVnaWVzKHN1bW1hcnk6IHBkLkRhdGFGcmFtZSwgbWFyZ2luOiBmbG9hdCwgbl9ib290c3RyYXA6IGludCwgc2VlZDogaW50KSAtPiBkaWN0Og0KICAgIHRwciA9IHBhaXJlZF9kaWZmZXJlbmNlKHN1bW1hcnksICJhdWxjX3RwciIpDQogICAgZnByID0gcGFpcmVkX2RpZmZlcmVuY2Uoc3VtbWFyeSwgImF1bGNfZnByIikNCiAgICBwX3RwciA9IHdpbGNveG9uX3B2YWx1ZSh0cHIudG9fbnVtcHkoKSkNCiAgICBwX2ZwciA9IHdpbGNveG9uX3B2YWx1ZShmcHIudG9fbnVtcHkoKSkNCiAgICBhZGp1c3RlZCA9IGhvbG0oW3BfdHByLCBwX2Zwcl0pDQogICAgdHByX2Jvb3QgPSBib290c3RyYXBfbWVkaWFuX2ludGVydmFsKHRwci50b19udW1weSgpLCBuX2Jvb3RzdHJhcCwgc2VlZCkNCiAgICBmcHJfYm9vdCA9IGJvb3RzdHJhcF9tZWRpYW5faW50ZXJ2YWwoZnByLnRvX251bXB5KCksIG5fYm9vdHN0cmFwLCBzZWVkICsgMSkNCiAgICBwcmFjdGljYWwgPSBib29sKHRwcl9ib290WyJtZWRpYW4iXSA+IDAuMCBhbmQgZnByX2Jvb3RbIm1lZGlhbiJdIDw9IGZsb2F0KG1hcmdpbikpDQogICAgcmV0dXJuIHsNCiAgICAgICAgImRpZmZlcmVuY2UiOiAia2NlbnRlciAtIHJhbmRvbSIsDQogICAgICAgICJwcmFjdGljYWxfY3JpdGVyaW9uX3Bhc3NlZCI6IHByYWN0aWNhbCwNCiAgICAgICAgImZwcl9tYXJnaW4iOiBmbG9hdChtYXJnaW4pLA0KICAgICAgICAiYXVsY190cHIiOiB7DQogICAgICAgICAgICAid2lsY294b25fcCI6IHBfdHByLA0KICAgICAgICAgICAgImhvbG1fcCI6IGFkanVzdGVkWzBdLA0KICAgICAgICAgICAgInBlcm11dGF0aW9uX3AiOiBzaWduZmxpcF9wdmFsdWUodHByLnRvX251bXB5KCkpLA0KICAgICAgICAgICAgImJvb3RzdHJhcCI6IHRwcl9ib290LA0KICAgICAgICAgICAgKipfZGlyZWN0aW9uX2NvdW50cyh0cHIsIGxvd2VyX2lzX2JldHRlcj1GYWxzZSksDQogICAgICAgIH0sDQogICAgICAgICJhdWxjX2ZwciI6IHsNCiAgICAgICAgICAgICJ3aWxjb3hvbl9wIjogcF9mcHIsDQogICAgICAgICAgICAiaG9sbV9wIjogYWRqdXN0ZWRbMV0sDQogICAgICAgICAgICAicGVybXV0YXRpb25fcCI6IHNpZ25mbGlwX3B2YWx1ZShmcHIudG9fbnVtcHkoKSksDQogICAgICAgICAgICAiYm9vdHN0cmFwIjogZnByX2Jvb3QsDQogICAgICAgICAgICAqKl9kaXJlY3Rpb25fY291bnRzKGZwciwgbG93ZXJfaXNfYmV0dGVyPVRydWUpLA0KICAgICAgICB9LA0KICAgIH0NCg==", "commissioning/stopping.py": "IiIiTm9ybWFsLW9ubHkgc3RvcHBpbmcgcnVsZS4gRGVsdGEgaXMgbWVhc3VyZWQgYWdhaW5zdCB0aGUgcHJldmlvdXMgZ3JpZCBwb2ludCwgbm90IE4vMi4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbWF0aA0KDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KDQpkZWYgcmVsYXRpdmVfZGVsdGFzKG1lZGlhbnM6IGxpc3RbZmxvYXRdIHwgbnAubmRhcnJheSkgLT4gbGlzdFtmbG9hdCB8IE5vbmVdOg0KICAgIHZhbHVlcyA9IFtmbG9hdCh2KSBmb3IgdiBpbiBtZWRpYW5zXQ0KICAgIGlmIGFueShub3QgbWF0aC5pc2Zpbml0ZSh2KSBmb3IgdiBpbiB2YWx1ZXMpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlN0b3BwaW5nIG1lZGlhbnMgbXVzdCBiZSBmaW5pdGUuIikNCiAgICBkZWx0YXM6IGxpc3RbZmxvYXQgfCBOb25lXSA9IFtOb25lXQ0KICAgIGZvciBwcmV2aW91cywgY3VycmVudCBpbiB6aXAodmFsdWVzLCB2YWx1ZXNbMTpdKToNCiAgICAgICAgaWYgcHJldmlvdXMgPT0gMC4wOg0KICAgICAgICAgICAgZGVsdGFzLmFwcGVuZCgwLjAgaWYgY3VycmVudCA9PSAwLjAgZWxzZSBtYXRoLmluZikNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIGRlbHRhcy5hcHBlbmQoYWJzKGN1cnJlbnQgLSBwcmV2aW91cykgLyBhYnMocHJldmlvdXMpKQ0KICAgIHJldHVybiBkZWx0YXMNCg0KDQpkZWYgbl9zdG9wKGdyaWQ6IGxpc3RbaW50XSwgbWVkaWFuczogbGlzdFtmbG9hdF0gfCBucC5uZGFycmF5LCBkZWx0YV90aHJlc2hvbGQ6IGZsb2F0KSAtPiBpbnQgfCBOb25lOg0KICAgICIiIkVuZHBvaW50IG9mIHRoZSBzZWNvbmQgY29uc2VjdXRpdmUgZ3JpZCB0cmFuc2l0aW9uIHdob3NlIHJlbGF0aXZlIGNoYW5nZSBpcyBiZWxvdyB0aGUgdGhyZXNob2xkLg0KDQogICAgUmV0dXJucyBOb25lIHdoZW4gdGhlIHJ1bGUgbmV2ZXIgZmlyZXMuIFRoZXJlIGlzIG5vIGludGVycG9sYXRpb24gYmV0d2VlbiBncmlkIHBvaW50cy4NCiAgICAiIiINCiAgICBpZiBsZW4oZ3JpZCkgIT0gbGVuKG1lZGlhbnMpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkdyaWQgYW5kIGNhbGlicmF0aW9uIG1lZGlhbnMgbXVzdCBoYXZlIHRoZSBzYW1lIGxlbmd0aC4iKQ0KICAgIGlmIGxlbihncmlkKSA8IDI6DQogICAgICAgIHJldHVybiBOb25lDQogICAgZGVsdGFzID0gcmVsYXRpdmVfZGVsdGFzKG1lZGlhbnMpDQogICAgc3RyZWFrID0gMA0KICAgIGZvciBpbmRleCBpbiByYW5nZSgxLCBsZW4oZ3JpZCkpOg0KICAgICAgICBpZiBkZWx0YXNbaW5kZXhdIGlzIG5vdCBOb25lIGFuZCBkZWx0YXNbaW5kZXhdIDwgZGVsdGFfdGhyZXNob2xkOg0KICAgICAgICAgICAgc3RyZWFrICs9IDENCiAgICAgICAgICAgIGlmIHN0cmVhayA+PSAyOg0KICAgICAgICAgICAgICAgIHJldHVybiBpbnQoZ3JpZFtpbmRleF0pDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBzdHJlYWsgPSAwDQogICAgcmV0dXJuIE5vbmUNCg=="}

for relative, encoded in FILES.items():
    destination = PROJECT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))
print(f"Zapisano {len(FILES)} plików do {PROJECT}")


In [ ]:
# KOMÓRKA 3/10 — biblioteki i rozpakowanie danych
# Kolejne komórki uruchamiają eksperyment w osobnym procesie, więc widzą wersje zainstalowane tutaj.
%pip install -q "scipy>=1.11" "pandas>=2.1" "pyarrow>=14" "scikit-learn>=1.3" "matplotlib>=3.8" "PyYAML>=6" "Pillow>=10" "tqdm>=4.66" "timm>=0.9" "anomalib>=2.0"
import subprocess, sys
print(subprocess.check_output(
    [sys.executable, "-c", "import torch, anomalib; print('torch', torch.__version__, '| anomalib', anomalib.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'BRAK')"],
    text=True,
))
# Jeśli wypisało GPU BRAK: Runtime -> Change runtime type -> GPU, potem od komórki 1.

import tarfile
import os
from pathlib import Path

tar_path = Path(os.environ["MVTEC_TAR"])
local_data_dir = Path("/content/mvtec_anomaly_detection")
local_data_dir.mkdir(parents=True, exist_ok=True)

print("Przygotowywanie danych...")
if tar_path.exists():
    if not (local_data_dir / "transistor").exists():
        print(f"Rozpakowywanie pliku {tar_path.name} do środowiska Colab...")
        with tarfile.open(tar_path, 'r:*') as tar_ref:
            tar_ref.extractall(path=local_data_dir)
        print("Rozpakowywanie zakończone!")

        # Jeśli plik tar ma wewnątrz folder bazowy 'mvtec_anomaly_detection', przenosimy jego zawartość wyżej
        nested_dir = local_data_dir / "mvtec_anomaly_detection"
        if nested_dir.exists() and nested_dir.is_dir():
            print("Przenoszenie plików z zagnieżdżonego folderu...")
            os.system(f"mv {nested_dir}/* {local_data_dir}/")
            os.system(f"rm -rf {nested_dir}")
    else:
        print("Pliki są już rozpakowane i gotowe.")
else:
    print(f"BŁĄD: Brak archiwum MVTEC_TAR: {tar_path}")
print("Gotowe!")


In [ ]:
# KOMÓRKA 4/10 — JEDYNA EDYCJA: ścieżki i tryb
import os

# False = 15 klas, bramka E0, przebieg do artykułu.
# True  = sam transistor, bez E0, wynik exploratory.
PILOT = False

# Sciezka zmieniona na dysk lokalny, gdzie rozpakowano pliki ZIP
DATA = "/content/mvtec_anomaly_detection"
WORK = os.environ.get("COMMISSIONING_WORK", "commissioning_runs")

os.environ["COMMISSIONING_DATA"] = DATA
os.environ["COMMISSIONING_WORK"] = WORK

if not os.path.exists("/content/commissioning_project"):
    raise FileNotFoundError("Katalog /content/commissioning_project nie istnieje! Upewnij się, że uruchomiłeś pomyślnie Komórkę 2/10.")

os.chdir("/content/commissioning_project")
import sys
if "/content/commissioning_project" not in sys.path:
    sys.path.insert(0, "/content/commissioning_project")
if not os.path.isdir(os.path.join(DATA, "transistor", "train", "good")):
    raise FileNotFoundError(f"Nie widzę {DATA}/transistor/train/good. Popraw DATA lub upewnij się, że rozpakowano pliki z komórki 3/10.")

from commissioning.config import load_config
cfg = load_config()
print("hash", cfg.hash)
print("dane", cfg.data_root)
print("wyniki", cfg.work_root)
print("pilot", PILOT)

In [ ]:
# KOMÓRKA 5/10 — podziały C/P i kontrola liczności
import subprocess, sys
command = [sys.executable, "-m", "commissioning", "prepare"]
if PILOT:
    command += ["--classes", "transistor"]
subprocess.check_call(command)

In [ ]:
# KOMÓRKA 6/10 — cechy Wide-ResNet-50-2, raz na klasę
import subprocess, sys
command = [sys.executable, "-m", "commissioning", "features"]
if PILOT:
    command += ["--classes", "transistor"]
subprocess.check_call(command)

In [ ]:
# KOMÓRKA 7/10 — tensor odległości, raz na klasę
import subprocess, sys
command = [sys.executable, "-m", "commissioning", "distances"]
if PILOT:
    command += ["--classes", "transistor"]
subprocess.check_call(command)

In [ ]:
# KOMÓRKA 8/10 — bramka E0
# Przy PILOT = True ta komórka nic nie liczy. Pełny przebieg bez PASS stąd nie przejdzie do komórki 9.
import subprocess, sys
if PILOT:
    print("Pilot: pomijam E0. Komórka 9 oznaczy trajektorie jako exploratory.")
else:
    subprocess.check_call([sys.executable, "-m", "commissioning", "e0"])

In [ ]:
# KOMÓRKA 9/10 — trajektorie RQ1–RQ4
import subprocess, sys
command = [sys.executable, "-m", "commissioning", "run"]
if PILOT:
    command += ["--classes", "transistor", "--exploratory"]
subprocess.check_call(command)

In [ ]:
# KOMÓRKA 10/10 — statystyka i raport HTML
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "commissioning", "analyze"])
subprocess.check_call([sys.executable, "-m", "commissioning", "report"])
print("Raport:", WORK + "/report/index.html")